In [1]:
#%pip install "https://github.com/RenatoVassallo/MacroPy/releases/download/0.1.5/macropy-0.1.5-py3-none-any.whl"
#%pip install -r "C:/Users/mlxfl/Documents/umich/699/project/requirements.txt"

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import json
from io import StringIO
import statsmodels.api as sm
from statsmodels.stats.stattools import omni_normtest
from MacroPy import BayesianVAR

In [3]:
# Preset alpha value for the Bernferroni correction
alpha = 0.05

In [4]:
parent_dir = Path.cwd().parent.parent

In [5]:
# Import datasets (google trends will not be used for the regression analysis)

# Monthly
fin_gig_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'gig_data_monthly.csv')) 
fin_sp500_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'sp500_data_monthly.csv'))
fred_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'fred', 'fred_monthly_data.csv'))
#google_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'google_trends', 'google_trends_monthly.csv'))

# Quarter
fin_gig_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'gig_data_quarterly.csv'))
fin_sp500_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'sp500_data_quarterly.csv'))
fred_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'fred', 'fred_quarterly_data.csv'))
#google_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'google_trends', 'google_trends_quarterly.csv'))

# Annual
fin_gig_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'gig_data_annually.csv'))
fin_sp500_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'sp500_data_annually.csv'))
fred_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'fred', 'fred_annual_data.csv'))
#google_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'google_trends', 'google_trends_annual.csv'))

In [6]:
# # Get lag and non-lag datasets

# # Monthly
# # Non-lag
# fin_gig_m_nolag_df = fin_gig_m_df[[c for c in fin_gig_m_df.columns if 'lag' not in c]]
# fin_sp500_m_nolag_df = fin_sp500_m_df[[c for c in fin_sp500_m_df.columns if 'lag' not in c]]
# fred_m_nolag_df = fred_m_df[[c for c in fred_m_df.columns if 'lag' not in c and 'quarterly' not in c and 'annual' not in c]]
# #google_m_nolag_df = google_m_df[[c for c in google_m_df.columns if 'lag' not in c]]
# # Lag
# fin_gig_m_lag_df = fin_gig_m_df[[c for c in fin_gig_m_df.columns if 'lag' in c or c in ('Date','Month','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fin_sp500_m_lag_df = fin_sp500_m_df[[c for c in fin_sp500_m_df.columns if 'lag' in c or c in ('Date','Month','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fred_m_lag_df = fred_m_df[[c for c in fred_m_df.columns if ('lag' in c and 'quarterly' not in c and 'annual' not in c) or c in ('date','month','quarter','year')]]
# #google_m_lag_df = google_m_df[[c for c in google_m_df.columns if 'lag' in c or c in ('year','month','date')]]

# # Quarter
# # Non-lag
# fin_gig_q_nolag_df = fin_gig_q_df[[c for c in fin_gig_q_df.columns if 'lag' not in c]]
# fin_sp500_q_nolag_df = fin_sp500_q_df[[c for c in fin_sp500_q_df.columns if 'lag' not in c]]
# fred_q_nolag_df = fred_q_df[[c for c in fred_q_df.columns if 'lag' not in c and 'annual' not in c]]
# #google_q_nolag_df = google_q_df[[c for c in google_q_df.columns if 'lag' not in c]]
# # Lag
# fin_gig_q_lag_df = fin_gig_q_df[[c for c in fin_gig_q_df.columns if 'lag' in c or c in ('Date','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fin_sp500_q_lag_df = fin_sp500_q_df[[c for c in fin_sp500_q_df.columns if 'lag' in c or c in ('Date','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fred_q_lag_df = fred_q_df[[c for c in fred_q_df.columns if ('lag' in c and 'annual' not in c) or c in ('date','quarter','year')]]
# #google_q_lag_df = google_q_df[[c for c in google_q_df.columns if 'lag' in c or c in ('year','quarter','date')]]

# # Annual
# # Non-lag
# fin_gig_a_nolag_df = fin_gig_a_df[[c for c in fin_gig_a_df.columns if 'lag' not in c]]
# fin_sp500_a_nolag_df = fin_sp500_a_df[[c for c in fin_sp500_a_df.columns if 'lag' not in c]]
# fred_a_nolag_df = fred_a_df[[c for c in fred_a_df.columns if 'lag' not in c and 'quarterly' not in c]]
# #google_a_nolag_df = google_a_df[[c for c in google_a_df.columns if 'lag' not in c]]
# # Lag
# fin_gig_a_lag_df = fin_gig_a_df[[c for c in fin_gig_a_df.columns if 'lag' in c or c in ('Date','Year','Company','Ticker','industryKey','sectorKey')]]
# fin_sp500_a_lag_df = fin_sp500_a_df[[c for c in fin_sp500_a_df.columns if 'lag' in c or c in ('Date','Year','Company','Ticker','industryKey','sectorKey')]]
# fred_a_lag_df = fred_a_df[[c for c in fred_a_df.columns if 'lag' in c or c in ('date','year')]]
# #google_a_lag_df = google_a_df[[c for c in google_a_df.columns if 'lag' in c or c in ('year','date')]]

In [7]:
# # Merge datasets
# # Merging with the fred datasets being the driver because they go back the farthest
# # Note: statsmodels does not process nulls. So, for simplicity, they are going to be dropped

# # Monthly
# # Non-lag
# monthly_nolag_gig_df = fred_m_nolag_df.merge(fin_gig_m_nolag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_nolag_df, on=['month','year'], how='left')
# monthly_nolag_sp500_df = fred_m_nolag_df.merge(fin_sp500_m_nolag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_nolag_df, on=['month','year'], how='left')
# # Lag
# monthly_lag_gig_df = fred_m_lag_df.merge(fin_gig_m_lag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_lag_df, on=['month','year'], how='left')
# monthly_lag_sp500_df = fred_m_lag_df.merge(fin_sp500_m_lag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_lag_df, on=['month','year'], how='left')

# # Quarterly
# # Non-lag
# quarterly_nolag_gig_df = fred_q_nolag_df.merge(fin_gig_q_nolag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_nolag_df, on=['quarter','year'], how='left')
# quarterly_nolag_sp500_df = fred_q_nolag_df.merge(fin_sp500_q_nolag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_nolag_df, on=['quarter','year'], how='left')
# # Lag
# quarterly_lag_gig_df = fred_q_lag_df.merge(fin_gig_q_lag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_lag_df, on=['quarter','year'], how='left')
# quarterly_lag_sp500_df = fred_q_lag_df.merge(fin_sp500_q_lag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_lag_df, on=['quarter','year'], how='left')

# # Annual
# # Non-lag
# annual_nolag_gig_df = fred_a_nolag_df.merge(fin_gig_a_nolag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_nolag_df, on=['year'], how='left')
# annual_nolag_sp500_df = fred_a_nolag_df.merge(fin_sp500_a_nolag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_nolag_df, on=['year'], how='left')
# # Lag
# annual_lag_gig_df = fred_a_lag_df.merge(fin_gig_a_lag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_lag_df, on=['year'], how='left')
# annual_lag_sp500_df = fred_a_lag_df.merge(fin_sp500_a_lag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_lag_df, on=['year'], how='left')


In [8]:
# Merge datasets
# Merging with the fred datasets being the driver because they go back the farthest
# Note: statsmodels does not process nulls. So, for simplicity, they are going to be dropped

# Monthly
monthly_gig_df = fred_m_df.merge(fin_gig_m_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_df, on=['month','year'], how='left')
monthly_sp500_df = fred_m_df.merge(fin_sp500_m_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_df, on=['month','year'], how='left')

# Quarterly
quarterly_gig_df = fred_q_df.merge(fin_gig_q_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_df, on=['quarter','year'], how='left')
quarterly_sp500_df = fred_q_df.merge(fin_sp500_q_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_df, on=['quarter','year'], how='left')

# Annual
annual_gig_df = fred_a_df.merge(fin_gig_a_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_df, on=['year'], how='left')
annual_sp500_df = fred_a_df.merge(fin_sp500_a_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_df, on=['year'], how='left')


In [9]:
# # Regression analysis section

# # Get ys
# y_nolag_gig_monthly = monthly_nolag_gig_df['Close']
# y_nolag_sp500_monthly = monthly_nolag_sp500_df['Close']
# y_lag_gig_monthly = monthly_lag_gig_df['Close_lag1']
# y_lag_sp500_monthly = monthly_lag_sp500_df['Close_lag1']
# y_nolag_gig_quarterly = quarterly_nolag_gig_df['Close']
# y_nolag_sp500_quarterly = quarterly_nolag_sp500_df['Close']
# y_lag_gig_quarterly = quarterly_lag_gig_df['Close_lag1']
# y_lag_sp500_quarterly = quarterly_lag_sp500_df['Close_lag1']
# y_nolag_gig_annual = annual_nolag_gig_df['Close']
# y_nolag_sp500_annual = annual_nolag_sp500_df['Close']
# y_lag_gig_annual = annual_lag_gig_df['Close_lag1']
# y_lag_sp500_annual = annual_lag_sp500_df['Close_lag1']

In [10]:
# Regression analysis section

# Get ys
y_gig_monthly = monthly_gig_df['Close']
y_sp500_monthly = monthly_sp500_df['Close']
y_gig_quarterly = quarterly_gig_df['Close']
y_sp500_quarterly = quarterly_sp500_df['Close']
y_gig_annual = annual_gig_df['Close']
y_sp500_annual = annual_sp500_df['Close']

In [11]:
# # Regression analysis on quarterly data

# # Since Megan is doing correlation analysis on one variable, I will only do regression on at least 2 variables
# # Get all variable combinations for each dataset
# # I am just going to use the gig datasets arbitrarily for everything because the columns line up in both the gig and sp500 datasets
# # I will convert the dataset to a frozenset in order to get the unique column combinations, but then convert back to a list for processing

# # Monthly
# # No-lag columns 
# monthly_nolag_cols = []
# for c in monthly_nolag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close']:
#         temp_cols.append(c)
#         for c2 in monthly_nolag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 monthly_nolag_cols.append(temp_col_set)
# monthly_nolag_cols = frozenset(monthly_nolag_cols)
# monthly_nolag_cols = [sorted(i) for i in monthly_nolag_cols]
# # Lag columns
# monthly_lag_cols = []
# for c in monthly_lag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter','Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1']:
#         temp_cols.append(c)
#         for c2 in monthly_lag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 monthly_lag_cols.append(temp_col_set)
# monthly_lag_cols = frozenset(monthly_lag_cols)
# monthly_lag_cols = [sorted(i) for i in monthly_lag_cols]

# # Quarterly
# # No-lag columns
# quarterly_nolag_cols = []
# for c in quarterly_nolag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close']:
#         temp_cols.append(c)
#         for c2 in quarterly_nolag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 quarterly_nolag_cols.append(temp_col_set)
# quarterly_nolag_cols = frozenset(quarterly_nolag_cols)
# quarterly_nolag_cols = [sorted(i) for i in quarterly_nolag_cols]
# # Lag columns
# quarterly_lag_cols = []
# for c in quarterly_lag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter','Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1']:
#         temp_cols.append(c)
#         for c2 in quarterly_lag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 quarterly_lag_cols.append(temp_col_set)
# quarterly_lag_cols = frozenset(quarterly_lag_cols)
# quarterly_lag_cols = [sorted(i) for i in quarterly_lag_cols]

# # Annual 
# # No-lag columns
# annual_nolag_cols = []
# for c in annual_nolag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close']:
#         temp_cols.append(c)
#         for c2 in annual_nolag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 annual_nolag_cols.append(temp_col_set)
# annual_nolag_cols = frozenset(annual_nolag_cols)
# annual_nolag_cols = [sorted(i) for i in annual_nolag_cols]
# # Lag columns
# annual_lag_cols = []
# for c in annual_lag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter','Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1']:
#         temp_cols.append(c)
#         for c2 in annual_lag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 annual_lag_cols.append(temp_col_set)
# annual_lag_cols = frozenset(annual_lag_cols)
# annual_lag_cols = [sorted(i) for i in annual_lag_cols]

In [12]:
# Regression analysis on quarterly data

# Since Megan is doing correlation analysis on one variable, I will only do regression on at least 2 variables
# Get all variable combinations for each dataset
# I am just going to use the gig datasets arbitrarily for everything because the columns line up in both the gig and sp500 datasets
# I will convert the dataset to a frozenset in order to get the unique column combinations, but then convert back to a list for processing

# Monthly
monthly_cols = []
for c in monthly_gig_df.columns:
    temp_cols = []
    if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close']:
        temp_cols.append(c)
        for c2 in monthly_gig_df.columns:
            if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close'] and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                monthly_cols.append(temp_col_set)
monthly_cols = frozenset(monthly_cols)
monthly_cols = [sorted(i) for i in monthly_cols]

# Quarterly
quarterly_cols = []
for c in quarterly_gig_df.columns:
    temp_cols = []
    if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close']:
        temp_cols.append(c)
        for c2 in quarterly_gig_df.columns:
            if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close'] and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                quarterly_cols.append(temp_col_set)
quarterly_cols = frozenset(quarterly_cols)
quarterly_cols = [sorted(i) for i in quarterly_cols]

# Annual 
annual_cols = []
for c in annual_gig_df.columns:
    temp_cols = []
    if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close']:
        temp_cols.append(c)
        for c2 in annual_gig_df.columns:
            if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close'] and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                annual_cols.append(temp_col_set)
annual_cols = frozenset(annual_cols)
annual_cols = [sorted(i) for i in annual_cols]

In [13]:
# # Bernferroni correction variable for each data frequency
# bonferroni_monthly_nolag = alpha / len(monthly_nolag_cols)
# bonferroni_monthly_lag = alpha / len(monthly_lag_cols)
# bonferroni_quarterly_nolag = alpha / len(quarterly_nolag_cols)
# bonferroni_quarterly_lag = alpha / len(quarterly_lag_cols)
# bonferroni_annual_nolag = alpha / len(annual_nolag_cols)
# bonferroni_annual_lag = alpha / len(annual_lag_cols)

In [14]:
# Bernferroni correction variable for each data frequency
bonferroni_monthly = alpha / len(monthly_cols)
bonferroni_quarterly = alpha / len(quarterly_cols)
bonferroni_annual = alpha / len(annual_cols)

In [15]:
# # Run regression on all column combinations and extract 
# #for x_cols in monthly_nolag_cols:
# model = sm.OLS(y_nolag_gig_monthly, monthly_nolag_gig_df[monthly_nolag_cols[0]]).fit()

# model_results ={
#    "columns": str(monthly_nolag_cols[0]),
#    "model_summary": str({"coef": model.params.to_dict(), "ci": model.conf_int().to_dict(), "p_values": model.pvalues.to_dict(), "t_values": model.tvalues.to_dict()}),
#    "r2": [float(model.rsquared)],
#    "r2_adj": [float(model.rsquared_adj)],
#    "f_score": [float(model.fvalue)],
#    "f_prob": [float(model.f_pvalue)],
#    "log_liklihood": [float(model.llf)],
#    "aic": [float(model.aic)],
#    "bic": [float(model.bic)],
#    "rse": [float(model.resid.std(ddof=len(monthly_nolag_cols[0])+1))] # 2nd answer from https://stackoverflow.com/questions/63333999/residual-standard-error-of-a-regression-in-python
# }

# for row in model.summary().tables[2].data:
#     for idx, val in enumerate(row):
#         if "Omnibus" in val:
#             model_results["omnibus"] = [float(row[idx+1])]
#         elif "Durbin-Watson" in val:
#             model_results["durbin_watson"] = [float(row[idx+1])]
#         elif "Prob(Omnibus)" in val:
#             model_results["omnibus_prob"] = [float(row[idx+1])]
#         elif "Jarque-Bera" in val:
#             model_results["jarque_bera"] = [float(row[idx+1])]
#         elif "Prob(Jarque-Bera)" in val:
#             model_results["jarque_bera_prob"] = [float(row[idx+1])]
#         elif "Skew" in val:
#             model_results["skew"] = [float(row[idx+1])]
#         elif "Kurtosis" in val:
#             model_results["kurtosis"] = [float(row[idx+1])]
#         elif "Cond." in val:
#             model_results["cond_no"] = [float(row[idx+1])]

# #model_summary = str(model_results.pop("model_summary"))
# #model_columns = model_results.pop("columns")
# model_df = pd.DataFrame.from_dict(model_results, orient='columns')
# #model_df["model_summary"] = model_summary

# model_df


In [42]:
# Bayesian VAR Analysis

# Monthly

monthly_gig_df_dateindex = monthly_gig_df.copy()#[monthly_cols]
monthly_cols = [c for c in monthly_gig_df.columns if '_lag' not in c and c not in ('month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
        'sectorKey')]
monthly_gig_df_dateindex = monthly_gig_df_dateindex[monthly_cols]
monthly_gig_df_dateindex['date'] = pd.to_datetime(monthly_gig_df_dateindex['date'])
monthly_gig_df_dateindex.set_index('date', inplace=True)

# y = pd.DataFrame(monthly_gig_df_dateindex['Close'])

# monthly_cols = [c for c in monthly_gig_df.columns if '_lag' in c or c in ('month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close')]
# x = monthly_gig_df_dateindex.drop(columns=monthly_cols)

bvar_monthly_gig = BayesianVAR(monthly_gig_df_dateindex, lags=6)
bvar_monthly_gig.model_summary()


**MacroPy: A Toolbox for Bayesian Macroeconometric Analysis in Python**  
Developed by [Renato Vassallo](https://www.linkedin.com/in/renatovassallo) - Institute for Economic Analysis (IAE-CSIC)  
Version 0.1.5 - July 2025  

---

**Model Specifications**  
- **Model Type**: Bayesian VAR  
- **Endogenous Variables**: interest_rate_mtg_data_weekly, cpi_data_monthly, cpi_fesl_data_monthly, rec_sahm_data_monthly, rec_smooth_prob_data_monthly, rec_nber_data_monthly, unemployment_data_monthly, unemployment_level_data_monthly, all_employed_data_monthly, employment_population_ratio_data_monthly, labor_force_participation_data_monthly, m1_data_monthly, m2_data_monthly, m2_real_data_monthly, interest_rate_fedfunds_data_monthly, consumer_sentiment_data_monthly, consumer_sentiment_inflation_data_monthly, consumer_sentiment_composite_data_monthly, consumer_sentiment_composite_amplitude_data_monthly, job_openings_data_monthly, job_hires_data_monthly, job_separations_data_monthly, average_weekly_earnings_data_monthly, gdp_data_quarterly, gdp_real_data_quarterly, cpi_inflation_data_annual, median_income_real_data_annual, median_income_data_annual, Close, Volume  
- **Exogenous Variables**: Constant   
- **Number of Lags**: 6  
- **Sample Period**: 2006-09-01 to 2024-01-01 (883 observations)
- **Total Parameters Estimated**: 5430

---

**Bayesian Estimation Settings**
- **Posterior Simulation**: Gibbs Sampling
- **Prior Type**: Minnesota  
- **Total Draws**: 5000
- **Burn-in**: 2500 (50%)  

---

**Forecast & IRF Details**  
- **Impulse Response Horizon**: 20  
- **Forecast Horizon**: 12  
- **IRF Computation**: 1 Standard Deviation  

---

**VAR Model Equations**

$$
\begin{align*}
interest_rate_mtg_data_weekly_{t} &= b_{1,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{1,2}^{1} cpi_data_monthly_{t-1} + b_{1,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{1,4}^{1} rec_sahm_data_monthly_{t-1} + b_{1,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{1,6}^{1} rec_nber_data_monthly_{t-1} + b_{1,7}^{1} unemployment_data_monthly_{t-1} + b_{1,8}^{1} unemployment_level_data_monthly_{t-1} + b_{1,9}^{1} all_employed_data_monthly_{t-1} + b_{1,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{1,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{1,12}^{1} m1_data_monthly_{t-1} + b_{1,13}^{1} m2_data_monthly_{t-1} + b_{1,14}^{1} m2_real_data_monthly_{t-1} + b_{1,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{1,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{1,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{1,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{1,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{1,20}^{1} job_openings_data_monthly_{t-1} + b_{1,21}^{1} job_hires_data_monthly_{t-1} + b_{1,22}^{1} job_separations_data_monthly_{t-1} + b_{1,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{1,24}^{1} gdp_data_quarterly_{t-1} + b_{1,25}^{1} gdp_real_data_quarterly_{t-1} + b_{1,26}^{1} cpi_inflation_data_annual_{t-1} + b_{1,27}^{1} median_income_real_data_annual_{t-1} + b_{1,28}^{1} median_income_data_annual_{t-1} + b_{1,29}^{1} Close_{t-1} + b_{1,30}^{1} Volume_{t-1} + b_{1,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{1,2}^{2} cpi_data_monthly_{t-2} + b_{1,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{1,4}^{2} rec_sahm_data_monthly_{t-2} + b_{1,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{1,6}^{2} rec_nber_data_monthly_{t-2} + b_{1,7}^{2} unemployment_data_monthly_{t-2} + b_{1,8}^{2} unemployment_level_data_monthly_{t-2} + b_{1,9}^{2} all_employed_data_monthly_{t-2} + b_{1,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{1,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{1,12}^{2} m1_data_monthly_{t-2} + b_{1,13}^{2} m2_data_monthly_{t-2} + b_{1,14}^{2} m2_real_data_monthly_{t-2} + b_{1,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{1,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{1,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{1,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{1,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{1,20}^{2} job_openings_data_monthly_{t-2} + b_{1,21}^{2} job_hires_data_monthly_{t-2} + b_{1,22}^{2} job_separations_data_monthly_{t-2} + b_{1,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{1,24}^{2} gdp_data_quarterly_{t-2} + b_{1,25}^{2} gdp_real_data_quarterly_{t-2} + b_{1,26}^{2} cpi_inflation_data_annual_{t-2} + b_{1,27}^{2} median_income_real_data_annual_{t-2} + b_{1,28}^{2} median_income_data_annual_{t-2} + b_{1,29}^{2} Close_{t-2} + b_{1,30}^{2} Volume_{t-2} + b_{1,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{1,2}^{3} cpi_data_monthly_{t-3} + b_{1,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{1,4}^{3} rec_sahm_data_monthly_{t-3} + b_{1,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{1,6}^{3} rec_nber_data_monthly_{t-3} + b_{1,7}^{3} unemployment_data_monthly_{t-3} + b_{1,8}^{3} unemployment_level_data_monthly_{t-3} + b_{1,9}^{3} all_employed_data_monthly_{t-3} + b_{1,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{1,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{1,12}^{3} m1_data_monthly_{t-3} + b_{1,13}^{3} m2_data_monthly_{t-3} + b_{1,14}^{3} m2_real_data_monthly_{t-3} + b_{1,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{1,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{1,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{1,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{1,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{1,20}^{3} job_openings_data_monthly_{t-3} + b_{1,21}^{3} job_hires_data_monthly_{t-3} + b_{1,22}^{3} job_separations_data_monthly_{t-3} + b_{1,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{1,24}^{3} gdp_data_quarterly_{t-3} + b_{1,25}^{3} gdp_real_data_quarterly_{t-3} + b_{1,26}^{3} cpi_inflation_data_annual_{t-3} + b_{1,27}^{3} median_income_real_data_annual_{t-3} + b_{1,28}^{3} median_income_data_annual_{t-3} + b_{1,29}^{3} Close_{t-3} + b_{1,30}^{3} Volume_{t-3} + b_{1,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{1,2}^{4} cpi_data_monthly_{t-4} + b_{1,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{1,4}^{4} rec_sahm_data_monthly_{t-4} + b_{1,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{1,6}^{4} rec_nber_data_monthly_{t-4} + b_{1,7}^{4} unemployment_data_monthly_{t-4} + b_{1,8}^{4} unemployment_level_data_monthly_{t-4} + b_{1,9}^{4} all_employed_data_monthly_{t-4} + b_{1,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{1,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{1,12}^{4} m1_data_monthly_{t-4} + b_{1,13}^{4} m2_data_monthly_{t-4} + b_{1,14}^{4} m2_real_data_monthly_{t-4} + b_{1,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{1,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{1,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{1,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{1,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{1,20}^{4} job_openings_data_monthly_{t-4} + b_{1,21}^{4} job_hires_data_monthly_{t-4} + b_{1,22}^{4} job_separations_data_monthly_{t-4} + b_{1,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{1,24}^{4} gdp_data_quarterly_{t-4} + b_{1,25}^{4} gdp_real_data_quarterly_{t-4} + b_{1,26}^{4} cpi_inflation_data_annual_{t-4} + b_{1,27}^{4} median_income_real_data_annual_{t-4} + b_{1,28}^{4} median_income_data_annual_{t-4} + b_{1,29}^{4} Close_{t-4} + b_{1,30}^{4} Volume_{t-4} + b_{1,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{1,2}^{5} cpi_data_monthly_{t-5} + b_{1,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{1,4}^{5} rec_sahm_data_monthly_{t-5} + b_{1,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{1,6}^{5} rec_nber_data_monthly_{t-5} + b_{1,7}^{5} unemployment_data_monthly_{t-5} + b_{1,8}^{5} unemployment_level_data_monthly_{t-5} + b_{1,9}^{5} all_employed_data_monthly_{t-5} + b_{1,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{1,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{1,12}^{5} m1_data_monthly_{t-5} + b_{1,13}^{5} m2_data_monthly_{t-5} + b_{1,14}^{5} m2_real_data_monthly_{t-5} + b_{1,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{1,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{1,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{1,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{1,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{1,20}^{5} job_openings_data_monthly_{t-5} + b_{1,21}^{5} job_hires_data_monthly_{t-5} + b_{1,22}^{5} job_separations_data_monthly_{t-5} + b_{1,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{1,24}^{5} gdp_data_quarterly_{t-5} + b_{1,25}^{5} gdp_real_data_quarterly_{t-5} + b_{1,26}^{5} cpi_inflation_data_annual_{t-5} + b_{1,27}^{5} median_income_real_data_annual_{t-5} + b_{1,28}^{5} median_income_data_annual_{t-5} + b_{1,29}^{5} Close_{t-5} + b_{1,30}^{5} Volume_{t-5} + b_{1,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{1,2}^{6} cpi_data_monthly_{t-6} + b_{1,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{1,4}^{6} rec_sahm_data_monthly_{t-6} + b_{1,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{1,6}^{6} rec_nber_data_monthly_{t-6} + b_{1,7}^{6} unemployment_data_monthly_{t-6} + b_{1,8}^{6} unemployment_level_data_monthly_{t-6} + b_{1,9}^{6} all_employed_data_monthly_{t-6} + b_{1,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{1,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{1,12}^{6} m1_data_monthly_{t-6} + b_{1,13}^{6} m2_data_monthly_{t-6} + b_{1,14}^{6} m2_real_data_monthly_{t-6} + b_{1,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{1,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{1,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{1,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{1,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{1,20}^{6} job_openings_data_monthly_{t-6} + b_{1,21}^{6} job_hires_data_monthly_{t-6} + b_{1,22}^{6} job_separations_data_monthly_{t-6} + b_{1,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{1,24}^{6} gdp_data_quarterly_{t-6} + b_{1,25}^{6} gdp_real_data_quarterly_{t-6} + b_{1,26}^{6} cpi_inflation_data_annual_{t-6} + b_{1,27}^{6} median_income_real_data_annual_{t-6} + b_{1,28}^{6} median_income_data_annual_{t-6} + b_{1,29}^{6} Close_{t-6} + b_{1,30}^{6} Volume_{t-6} + c_{1} + e_{t}^{interest_rate_mtg_data_weekly} \\
cpi_data_monthly_{t} &= b_{2,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{2,2}^{1} cpi_data_monthly_{t-1} + b_{2,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{2,4}^{1} rec_sahm_data_monthly_{t-1} + b_{2,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{2,6}^{1} rec_nber_data_monthly_{t-1} + b_{2,7}^{1} unemployment_data_monthly_{t-1} + b_{2,8}^{1} unemployment_level_data_monthly_{t-1} + b_{2,9}^{1} all_employed_data_monthly_{t-1} + b_{2,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{2,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{2,12}^{1} m1_data_monthly_{t-1} + b_{2,13}^{1} m2_data_monthly_{t-1} + b_{2,14}^{1} m2_real_data_monthly_{t-1} + b_{2,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{2,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{2,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{2,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{2,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{2,20}^{1} job_openings_data_monthly_{t-1} + b_{2,21}^{1} job_hires_data_monthly_{t-1} + b_{2,22}^{1} job_separations_data_monthly_{t-1} + b_{2,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{2,24}^{1} gdp_data_quarterly_{t-1} + b_{2,25}^{1} gdp_real_data_quarterly_{t-1} + b_{2,26}^{1} cpi_inflation_data_annual_{t-1} + b_{2,27}^{1} median_income_real_data_annual_{t-1} + b_{2,28}^{1} median_income_data_annual_{t-1} + b_{2,29}^{1} Close_{t-1} + b_{2,30}^{1} Volume_{t-1} + b_{2,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{2,2}^{2} cpi_data_monthly_{t-2} + b_{2,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{2,4}^{2} rec_sahm_data_monthly_{t-2} + b_{2,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{2,6}^{2} rec_nber_data_monthly_{t-2} + b_{2,7}^{2} unemployment_data_monthly_{t-2} + b_{2,8}^{2} unemployment_level_data_monthly_{t-2} + b_{2,9}^{2} all_employed_data_monthly_{t-2} + b_{2,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{2,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{2,12}^{2} m1_data_monthly_{t-2} + b_{2,13}^{2} m2_data_monthly_{t-2} + b_{2,14}^{2} m2_real_data_monthly_{t-2} + b_{2,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{2,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{2,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{2,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{2,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{2,20}^{2} job_openings_data_monthly_{t-2} + b_{2,21}^{2} job_hires_data_monthly_{t-2} + b_{2,22}^{2} job_separations_data_monthly_{t-2} + b_{2,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{2,24}^{2} gdp_data_quarterly_{t-2} + b_{2,25}^{2} gdp_real_data_quarterly_{t-2} + b_{2,26}^{2} cpi_inflation_data_annual_{t-2} + b_{2,27}^{2} median_income_real_data_annual_{t-2} + b_{2,28}^{2} median_income_data_annual_{t-2} + b_{2,29}^{2} Close_{t-2} + b_{2,30}^{2} Volume_{t-2} + b_{2,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{2,2}^{3} cpi_data_monthly_{t-3} + b_{2,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{2,4}^{3} rec_sahm_data_monthly_{t-3} + b_{2,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{2,6}^{3} rec_nber_data_monthly_{t-3} + b_{2,7}^{3} unemployment_data_monthly_{t-3} + b_{2,8}^{3} unemployment_level_data_monthly_{t-3} + b_{2,9}^{3} all_employed_data_monthly_{t-3} + b_{2,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{2,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{2,12}^{3} m1_data_monthly_{t-3} + b_{2,13}^{3} m2_data_monthly_{t-3} + b_{2,14}^{3} m2_real_data_monthly_{t-3} + b_{2,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{2,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{2,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{2,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{2,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{2,20}^{3} job_openings_data_monthly_{t-3} + b_{2,21}^{3} job_hires_data_monthly_{t-3} + b_{2,22}^{3} job_separations_data_monthly_{t-3} + b_{2,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{2,24}^{3} gdp_data_quarterly_{t-3} + b_{2,25}^{3} gdp_real_data_quarterly_{t-3} + b_{2,26}^{3} cpi_inflation_data_annual_{t-3} + b_{2,27}^{3} median_income_real_data_annual_{t-3} + b_{2,28}^{3} median_income_data_annual_{t-3} + b_{2,29}^{3} Close_{t-3} + b_{2,30}^{3} Volume_{t-3} + b_{2,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{2,2}^{4} cpi_data_monthly_{t-4} + b_{2,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{2,4}^{4} rec_sahm_data_monthly_{t-4} + b_{2,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{2,6}^{4} rec_nber_data_monthly_{t-4} + b_{2,7}^{4} unemployment_data_monthly_{t-4} + b_{2,8}^{4} unemployment_level_data_monthly_{t-4} + b_{2,9}^{4} all_employed_data_monthly_{t-4} + b_{2,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{2,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{2,12}^{4} m1_data_monthly_{t-4} + b_{2,13}^{4} m2_data_monthly_{t-4} + b_{2,14}^{4} m2_real_data_monthly_{t-4} + b_{2,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{2,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{2,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{2,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{2,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{2,20}^{4} job_openings_data_monthly_{t-4} + b_{2,21}^{4} job_hires_data_monthly_{t-4} + b_{2,22}^{4} job_separations_data_monthly_{t-4} + b_{2,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{2,24}^{4} gdp_data_quarterly_{t-4} + b_{2,25}^{4} gdp_real_data_quarterly_{t-4} + b_{2,26}^{4} cpi_inflation_data_annual_{t-4} + b_{2,27}^{4} median_income_real_data_annual_{t-4} + b_{2,28}^{4} median_income_data_annual_{t-4} + b_{2,29}^{4} Close_{t-4} + b_{2,30}^{4} Volume_{t-4} + b_{2,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{2,2}^{5} cpi_data_monthly_{t-5} + b_{2,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{2,4}^{5} rec_sahm_data_monthly_{t-5} + b_{2,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{2,6}^{5} rec_nber_data_monthly_{t-5} + b_{2,7}^{5} unemployment_data_monthly_{t-5} + b_{2,8}^{5} unemployment_level_data_monthly_{t-5} + b_{2,9}^{5} all_employed_data_monthly_{t-5} + b_{2,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{2,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{2,12}^{5} m1_data_monthly_{t-5} + b_{2,13}^{5} m2_data_monthly_{t-5} + b_{2,14}^{5} m2_real_data_monthly_{t-5} + b_{2,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{2,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{2,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{2,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{2,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{2,20}^{5} job_openings_data_monthly_{t-5} + b_{2,21}^{5} job_hires_data_monthly_{t-5} + b_{2,22}^{5} job_separations_data_monthly_{t-5} + b_{2,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{2,24}^{5} gdp_data_quarterly_{t-5} + b_{2,25}^{5} gdp_real_data_quarterly_{t-5} + b_{2,26}^{5} cpi_inflation_data_annual_{t-5} + b_{2,27}^{5} median_income_real_data_annual_{t-5} + b_{2,28}^{5} median_income_data_annual_{t-5} + b_{2,29}^{5} Close_{t-5} + b_{2,30}^{5} Volume_{t-5} + b_{2,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{2,2}^{6} cpi_data_monthly_{t-6} + b_{2,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{2,4}^{6} rec_sahm_data_monthly_{t-6} + b_{2,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{2,6}^{6} rec_nber_data_monthly_{t-6} + b_{2,7}^{6} unemployment_data_monthly_{t-6} + b_{2,8}^{6} unemployment_level_data_monthly_{t-6} + b_{2,9}^{6} all_employed_data_monthly_{t-6} + b_{2,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{2,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{2,12}^{6} m1_data_monthly_{t-6} + b_{2,13}^{6} m2_data_monthly_{t-6} + b_{2,14}^{6} m2_real_data_monthly_{t-6} + b_{2,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{2,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{2,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{2,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{2,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{2,20}^{6} job_openings_data_monthly_{t-6} + b_{2,21}^{6} job_hires_data_monthly_{t-6} + b_{2,22}^{6} job_separations_data_monthly_{t-6} + b_{2,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{2,24}^{6} gdp_data_quarterly_{t-6} + b_{2,25}^{6} gdp_real_data_quarterly_{t-6} + b_{2,26}^{6} cpi_inflation_data_annual_{t-6} + b_{2,27}^{6} median_income_real_data_annual_{t-6} + b_{2,28}^{6} median_income_data_annual_{t-6} + b_{2,29}^{6} Close_{t-6} + b_{2,30}^{6} Volume_{t-6} + c_{2} + e_{t}^{cpi_data_monthly} \\
cpi_fesl_data_monthly_{t} &= b_{3,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{3,2}^{1} cpi_data_monthly_{t-1} + b_{3,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{3,4}^{1} rec_sahm_data_monthly_{t-1} + b_{3,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{3,6}^{1} rec_nber_data_monthly_{t-1} + b_{3,7}^{1} unemployment_data_monthly_{t-1} + b_{3,8}^{1} unemployment_level_data_monthly_{t-1} + b_{3,9}^{1} all_employed_data_monthly_{t-1} + b_{3,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{3,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{3,12}^{1} m1_data_monthly_{t-1} + b_{3,13}^{1} m2_data_monthly_{t-1} + b_{3,14}^{1} m2_real_data_monthly_{t-1} + b_{3,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{3,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{3,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{3,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{3,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{3,20}^{1} job_openings_data_monthly_{t-1} + b_{3,21}^{1} job_hires_data_monthly_{t-1} + b_{3,22}^{1} job_separations_data_monthly_{t-1} + b_{3,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{3,24}^{1} gdp_data_quarterly_{t-1} + b_{3,25}^{1} gdp_real_data_quarterly_{t-1} + b_{3,26}^{1} cpi_inflation_data_annual_{t-1} + b_{3,27}^{1} median_income_real_data_annual_{t-1} + b_{3,28}^{1} median_income_data_annual_{t-1} + b_{3,29}^{1} Close_{t-1} + b_{3,30}^{1} Volume_{t-1} + b_{3,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{3,2}^{2} cpi_data_monthly_{t-2} + b_{3,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{3,4}^{2} rec_sahm_data_monthly_{t-2} + b_{3,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{3,6}^{2} rec_nber_data_monthly_{t-2} + b_{3,7}^{2} unemployment_data_monthly_{t-2} + b_{3,8}^{2} unemployment_level_data_monthly_{t-2} + b_{3,9}^{2} all_employed_data_monthly_{t-2} + b_{3,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{3,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{3,12}^{2} m1_data_monthly_{t-2} + b_{3,13}^{2} m2_data_monthly_{t-2} + b_{3,14}^{2} m2_real_data_monthly_{t-2} + b_{3,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{3,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{3,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{3,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{3,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{3,20}^{2} job_openings_data_monthly_{t-2} + b_{3,21}^{2} job_hires_data_monthly_{t-2} + b_{3,22}^{2} job_separations_data_monthly_{t-2} + b_{3,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{3,24}^{2} gdp_data_quarterly_{t-2} + b_{3,25}^{2} gdp_real_data_quarterly_{t-2} + b_{3,26}^{2} cpi_inflation_data_annual_{t-2} + b_{3,27}^{2} median_income_real_data_annual_{t-2} + b_{3,28}^{2} median_income_data_annual_{t-2} + b_{3,29}^{2} Close_{t-2} + b_{3,30}^{2} Volume_{t-2} + b_{3,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{3,2}^{3} cpi_data_monthly_{t-3} + b_{3,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{3,4}^{3} rec_sahm_data_monthly_{t-3} + b_{3,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{3,6}^{3} rec_nber_data_monthly_{t-3} + b_{3,7}^{3} unemployment_data_monthly_{t-3} + b_{3,8}^{3} unemployment_level_data_monthly_{t-3} + b_{3,9}^{3} all_employed_data_monthly_{t-3} + b_{3,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{3,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{3,12}^{3} m1_data_monthly_{t-3} + b_{3,13}^{3} m2_data_monthly_{t-3} + b_{3,14}^{3} m2_real_data_monthly_{t-3} + b_{3,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{3,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{3,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{3,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{3,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{3,20}^{3} job_openings_data_monthly_{t-3} + b_{3,21}^{3} job_hires_data_monthly_{t-3} + b_{3,22}^{3} job_separations_data_monthly_{t-3} + b_{3,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{3,24}^{3} gdp_data_quarterly_{t-3} + b_{3,25}^{3} gdp_real_data_quarterly_{t-3} + b_{3,26}^{3} cpi_inflation_data_annual_{t-3} + b_{3,27}^{3} median_income_real_data_annual_{t-3} + b_{3,28}^{3} median_income_data_annual_{t-3} + b_{3,29}^{3} Close_{t-3} + b_{3,30}^{3} Volume_{t-3} + b_{3,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{3,2}^{4} cpi_data_monthly_{t-4} + b_{3,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{3,4}^{4} rec_sahm_data_monthly_{t-4} + b_{3,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{3,6}^{4} rec_nber_data_monthly_{t-4} + b_{3,7}^{4} unemployment_data_monthly_{t-4} + b_{3,8}^{4} unemployment_level_data_monthly_{t-4} + b_{3,9}^{4} all_employed_data_monthly_{t-4} + b_{3,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{3,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{3,12}^{4} m1_data_monthly_{t-4} + b_{3,13}^{4} m2_data_monthly_{t-4} + b_{3,14}^{4} m2_real_data_monthly_{t-4} + b_{3,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{3,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{3,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{3,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{3,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{3,20}^{4} job_openings_data_monthly_{t-4} + b_{3,21}^{4} job_hires_data_monthly_{t-4} + b_{3,22}^{4} job_separations_data_monthly_{t-4} + b_{3,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{3,24}^{4} gdp_data_quarterly_{t-4} + b_{3,25}^{4} gdp_real_data_quarterly_{t-4} + b_{3,26}^{4} cpi_inflation_data_annual_{t-4} + b_{3,27}^{4} median_income_real_data_annual_{t-4} + b_{3,28}^{4} median_income_data_annual_{t-4} + b_{3,29}^{4} Close_{t-4} + b_{3,30}^{4} Volume_{t-4} + b_{3,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{3,2}^{5} cpi_data_monthly_{t-5} + b_{3,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{3,4}^{5} rec_sahm_data_monthly_{t-5} + b_{3,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{3,6}^{5} rec_nber_data_monthly_{t-5} + b_{3,7}^{5} unemployment_data_monthly_{t-5} + b_{3,8}^{5} unemployment_level_data_monthly_{t-5} + b_{3,9}^{5} all_employed_data_monthly_{t-5} + b_{3,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{3,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{3,12}^{5} m1_data_monthly_{t-5} + b_{3,13}^{5} m2_data_monthly_{t-5} + b_{3,14}^{5} m2_real_data_monthly_{t-5} + b_{3,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{3,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{3,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{3,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{3,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{3,20}^{5} job_openings_data_monthly_{t-5} + b_{3,21}^{5} job_hires_data_monthly_{t-5} + b_{3,22}^{5} job_separations_data_monthly_{t-5} + b_{3,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{3,24}^{5} gdp_data_quarterly_{t-5} + b_{3,25}^{5} gdp_real_data_quarterly_{t-5} + b_{3,26}^{5} cpi_inflation_data_annual_{t-5} + b_{3,27}^{5} median_income_real_data_annual_{t-5} + b_{3,28}^{5} median_income_data_annual_{t-5} + b_{3,29}^{5} Close_{t-5} + b_{3,30}^{5} Volume_{t-5} + b_{3,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{3,2}^{6} cpi_data_monthly_{t-6} + b_{3,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{3,4}^{6} rec_sahm_data_monthly_{t-6} + b_{3,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{3,6}^{6} rec_nber_data_monthly_{t-6} + b_{3,7}^{6} unemployment_data_monthly_{t-6} + b_{3,8}^{6} unemployment_level_data_monthly_{t-6} + b_{3,9}^{6} all_employed_data_monthly_{t-6} + b_{3,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{3,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{3,12}^{6} m1_data_monthly_{t-6} + b_{3,13}^{6} m2_data_monthly_{t-6} + b_{3,14}^{6} m2_real_data_monthly_{t-6} + b_{3,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{3,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{3,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{3,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{3,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{3,20}^{6} job_openings_data_monthly_{t-6} + b_{3,21}^{6} job_hires_data_monthly_{t-6} + b_{3,22}^{6} job_separations_data_monthly_{t-6} + b_{3,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{3,24}^{6} gdp_data_quarterly_{t-6} + b_{3,25}^{6} gdp_real_data_quarterly_{t-6} + b_{3,26}^{6} cpi_inflation_data_annual_{t-6} + b_{3,27}^{6} median_income_real_data_annual_{t-6} + b_{3,28}^{6} median_income_data_annual_{t-6} + b_{3,29}^{6} Close_{t-6} + b_{3,30}^{6} Volume_{t-6} + c_{3} + e_{t}^{cpi_fesl_data_monthly} \\
rec_sahm_data_monthly_{t} &= b_{4,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{4,2}^{1} cpi_data_monthly_{t-1} + b_{4,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{4,4}^{1} rec_sahm_data_monthly_{t-1} + b_{4,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{4,6}^{1} rec_nber_data_monthly_{t-1} + b_{4,7}^{1} unemployment_data_monthly_{t-1} + b_{4,8}^{1} unemployment_level_data_monthly_{t-1} + b_{4,9}^{1} all_employed_data_monthly_{t-1} + b_{4,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{4,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{4,12}^{1} m1_data_monthly_{t-1} + b_{4,13}^{1} m2_data_monthly_{t-1} + b_{4,14}^{1} m2_real_data_monthly_{t-1} + b_{4,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{4,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{4,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{4,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{4,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{4,20}^{1} job_openings_data_monthly_{t-1} + b_{4,21}^{1} job_hires_data_monthly_{t-1} + b_{4,22}^{1} job_separations_data_monthly_{t-1} + b_{4,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{4,24}^{1} gdp_data_quarterly_{t-1} + b_{4,25}^{1} gdp_real_data_quarterly_{t-1} + b_{4,26}^{1} cpi_inflation_data_annual_{t-1} + b_{4,27}^{1} median_income_real_data_annual_{t-1} + b_{4,28}^{1} median_income_data_annual_{t-1} + b_{4,29}^{1} Close_{t-1} + b_{4,30}^{1} Volume_{t-1} + b_{4,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{4,2}^{2} cpi_data_monthly_{t-2} + b_{4,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{4,4}^{2} rec_sahm_data_monthly_{t-2} + b_{4,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{4,6}^{2} rec_nber_data_monthly_{t-2} + b_{4,7}^{2} unemployment_data_monthly_{t-2} + b_{4,8}^{2} unemployment_level_data_monthly_{t-2} + b_{4,9}^{2} all_employed_data_monthly_{t-2} + b_{4,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{4,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{4,12}^{2} m1_data_monthly_{t-2} + b_{4,13}^{2} m2_data_monthly_{t-2} + b_{4,14}^{2} m2_real_data_monthly_{t-2} + b_{4,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{4,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{4,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{4,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{4,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{4,20}^{2} job_openings_data_monthly_{t-2} + b_{4,21}^{2} job_hires_data_monthly_{t-2} + b_{4,22}^{2} job_separations_data_monthly_{t-2} + b_{4,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{4,24}^{2} gdp_data_quarterly_{t-2} + b_{4,25}^{2} gdp_real_data_quarterly_{t-2} + b_{4,26}^{2} cpi_inflation_data_annual_{t-2} + b_{4,27}^{2} median_income_real_data_annual_{t-2} + b_{4,28}^{2} median_income_data_annual_{t-2} + b_{4,29}^{2} Close_{t-2} + b_{4,30}^{2} Volume_{t-2} + b_{4,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{4,2}^{3} cpi_data_monthly_{t-3} + b_{4,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{4,4}^{3} rec_sahm_data_monthly_{t-3} + b_{4,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{4,6}^{3} rec_nber_data_monthly_{t-3} + b_{4,7}^{3} unemployment_data_monthly_{t-3} + b_{4,8}^{3} unemployment_level_data_monthly_{t-3} + b_{4,9}^{3} all_employed_data_monthly_{t-3} + b_{4,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{4,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{4,12}^{3} m1_data_monthly_{t-3} + b_{4,13}^{3} m2_data_monthly_{t-3} + b_{4,14}^{3} m2_real_data_monthly_{t-3} + b_{4,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{4,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{4,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{4,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{4,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{4,20}^{3} job_openings_data_monthly_{t-3} + b_{4,21}^{3} job_hires_data_monthly_{t-3} + b_{4,22}^{3} job_separations_data_monthly_{t-3} + b_{4,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{4,24}^{3} gdp_data_quarterly_{t-3} + b_{4,25}^{3} gdp_real_data_quarterly_{t-3} + b_{4,26}^{3} cpi_inflation_data_annual_{t-3} + b_{4,27}^{3} median_income_real_data_annual_{t-3} + b_{4,28}^{3} median_income_data_annual_{t-3} + b_{4,29}^{3} Close_{t-3} + b_{4,30}^{3} Volume_{t-3} + b_{4,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{4,2}^{4} cpi_data_monthly_{t-4} + b_{4,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{4,4}^{4} rec_sahm_data_monthly_{t-4} + b_{4,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{4,6}^{4} rec_nber_data_monthly_{t-4} + b_{4,7}^{4} unemployment_data_monthly_{t-4} + b_{4,8}^{4} unemployment_level_data_monthly_{t-4} + b_{4,9}^{4} all_employed_data_monthly_{t-4} + b_{4,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{4,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{4,12}^{4} m1_data_monthly_{t-4} + b_{4,13}^{4} m2_data_monthly_{t-4} + b_{4,14}^{4} m2_real_data_monthly_{t-4} + b_{4,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{4,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{4,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{4,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{4,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{4,20}^{4} job_openings_data_monthly_{t-4} + b_{4,21}^{4} job_hires_data_monthly_{t-4} + b_{4,22}^{4} job_separations_data_monthly_{t-4} + b_{4,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{4,24}^{4} gdp_data_quarterly_{t-4} + b_{4,25}^{4} gdp_real_data_quarterly_{t-4} + b_{4,26}^{4} cpi_inflation_data_annual_{t-4} + b_{4,27}^{4} median_income_real_data_annual_{t-4} + b_{4,28}^{4} median_income_data_annual_{t-4} + b_{4,29}^{4} Close_{t-4} + b_{4,30}^{4} Volume_{t-4} + b_{4,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{4,2}^{5} cpi_data_monthly_{t-5} + b_{4,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{4,4}^{5} rec_sahm_data_monthly_{t-5} + b_{4,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{4,6}^{5} rec_nber_data_monthly_{t-5} + b_{4,7}^{5} unemployment_data_monthly_{t-5} + b_{4,8}^{5} unemployment_level_data_monthly_{t-5} + b_{4,9}^{5} all_employed_data_monthly_{t-5} + b_{4,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{4,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{4,12}^{5} m1_data_monthly_{t-5} + b_{4,13}^{5} m2_data_monthly_{t-5} + b_{4,14}^{5} m2_real_data_monthly_{t-5} + b_{4,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{4,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{4,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{4,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{4,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{4,20}^{5} job_openings_data_monthly_{t-5} + b_{4,21}^{5} job_hires_data_monthly_{t-5} + b_{4,22}^{5} job_separations_data_monthly_{t-5} + b_{4,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{4,24}^{5} gdp_data_quarterly_{t-5} + b_{4,25}^{5} gdp_real_data_quarterly_{t-5} + b_{4,26}^{5} cpi_inflation_data_annual_{t-5} + b_{4,27}^{5} median_income_real_data_annual_{t-5} + b_{4,28}^{5} median_income_data_annual_{t-5} + b_{4,29}^{5} Close_{t-5} + b_{4,30}^{5} Volume_{t-5} + b_{4,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{4,2}^{6} cpi_data_monthly_{t-6} + b_{4,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{4,4}^{6} rec_sahm_data_monthly_{t-6} + b_{4,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{4,6}^{6} rec_nber_data_monthly_{t-6} + b_{4,7}^{6} unemployment_data_monthly_{t-6} + b_{4,8}^{6} unemployment_level_data_monthly_{t-6} + b_{4,9}^{6} all_employed_data_monthly_{t-6} + b_{4,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{4,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{4,12}^{6} m1_data_monthly_{t-6} + b_{4,13}^{6} m2_data_monthly_{t-6} + b_{4,14}^{6} m2_real_data_monthly_{t-6} + b_{4,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{4,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{4,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{4,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{4,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{4,20}^{6} job_openings_data_monthly_{t-6} + b_{4,21}^{6} job_hires_data_monthly_{t-6} + b_{4,22}^{6} job_separations_data_monthly_{t-6} + b_{4,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{4,24}^{6} gdp_data_quarterly_{t-6} + b_{4,25}^{6} gdp_real_data_quarterly_{t-6} + b_{4,26}^{6} cpi_inflation_data_annual_{t-6} + b_{4,27}^{6} median_income_real_data_annual_{t-6} + b_{4,28}^{6} median_income_data_annual_{t-6} + b_{4,29}^{6} Close_{t-6} + b_{4,30}^{6} Volume_{t-6} + c_{4} + e_{t}^{rec_sahm_data_monthly} \\
rec_smooth_prob_data_monthly_{t} &= b_{5,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{5,2}^{1} cpi_data_monthly_{t-1} + b_{5,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{5,4}^{1} rec_sahm_data_monthly_{t-1} + b_{5,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{5,6}^{1} rec_nber_data_monthly_{t-1} + b_{5,7}^{1} unemployment_data_monthly_{t-1} + b_{5,8}^{1} unemployment_level_data_monthly_{t-1} + b_{5,9}^{1} all_employed_data_monthly_{t-1} + b_{5,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{5,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{5,12}^{1} m1_data_monthly_{t-1} + b_{5,13}^{1} m2_data_monthly_{t-1} + b_{5,14}^{1} m2_real_data_monthly_{t-1} + b_{5,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{5,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{5,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{5,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{5,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{5,20}^{1} job_openings_data_monthly_{t-1} + b_{5,21}^{1} job_hires_data_monthly_{t-1} + b_{5,22}^{1} job_separations_data_monthly_{t-1} + b_{5,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{5,24}^{1} gdp_data_quarterly_{t-1} + b_{5,25}^{1} gdp_real_data_quarterly_{t-1} + b_{5,26}^{1} cpi_inflation_data_annual_{t-1} + b_{5,27}^{1} median_income_real_data_annual_{t-1} + b_{5,28}^{1} median_income_data_annual_{t-1} + b_{5,29}^{1} Close_{t-1} + b_{5,30}^{1} Volume_{t-1} + b_{5,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{5,2}^{2} cpi_data_monthly_{t-2} + b_{5,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{5,4}^{2} rec_sahm_data_monthly_{t-2} + b_{5,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{5,6}^{2} rec_nber_data_monthly_{t-2} + b_{5,7}^{2} unemployment_data_monthly_{t-2} + b_{5,8}^{2} unemployment_level_data_monthly_{t-2} + b_{5,9}^{2} all_employed_data_monthly_{t-2} + b_{5,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{5,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{5,12}^{2} m1_data_monthly_{t-2} + b_{5,13}^{2} m2_data_monthly_{t-2} + b_{5,14}^{2} m2_real_data_monthly_{t-2} + b_{5,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{5,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{5,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{5,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{5,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{5,20}^{2} job_openings_data_monthly_{t-2} + b_{5,21}^{2} job_hires_data_monthly_{t-2} + b_{5,22}^{2} job_separations_data_monthly_{t-2} + b_{5,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{5,24}^{2} gdp_data_quarterly_{t-2} + b_{5,25}^{2} gdp_real_data_quarterly_{t-2} + b_{5,26}^{2} cpi_inflation_data_annual_{t-2} + b_{5,27}^{2} median_income_real_data_annual_{t-2} + b_{5,28}^{2} median_income_data_annual_{t-2} + b_{5,29}^{2} Close_{t-2} + b_{5,30}^{2} Volume_{t-2} + b_{5,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{5,2}^{3} cpi_data_monthly_{t-3} + b_{5,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{5,4}^{3} rec_sahm_data_monthly_{t-3} + b_{5,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{5,6}^{3} rec_nber_data_monthly_{t-3} + b_{5,7}^{3} unemployment_data_monthly_{t-3} + b_{5,8}^{3} unemployment_level_data_monthly_{t-3} + b_{5,9}^{3} all_employed_data_monthly_{t-3} + b_{5,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{5,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{5,12}^{3} m1_data_monthly_{t-3} + b_{5,13}^{3} m2_data_monthly_{t-3} + b_{5,14}^{3} m2_real_data_monthly_{t-3} + b_{5,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{5,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{5,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{5,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{5,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{5,20}^{3} job_openings_data_monthly_{t-3} + b_{5,21}^{3} job_hires_data_monthly_{t-3} + b_{5,22}^{3} job_separations_data_monthly_{t-3} + b_{5,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{5,24}^{3} gdp_data_quarterly_{t-3} + b_{5,25}^{3} gdp_real_data_quarterly_{t-3} + b_{5,26}^{3} cpi_inflation_data_annual_{t-3} + b_{5,27}^{3} median_income_real_data_annual_{t-3} + b_{5,28}^{3} median_income_data_annual_{t-3} + b_{5,29}^{3} Close_{t-3} + b_{5,30}^{3} Volume_{t-3} + b_{5,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{5,2}^{4} cpi_data_monthly_{t-4} + b_{5,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{5,4}^{4} rec_sahm_data_monthly_{t-4} + b_{5,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{5,6}^{4} rec_nber_data_monthly_{t-4} + b_{5,7}^{4} unemployment_data_monthly_{t-4} + b_{5,8}^{4} unemployment_level_data_monthly_{t-4} + b_{5,9}^{4} all_employed_data_monthly_{t-4} + b_{5,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{5,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{5,12}^{4} m1_data_monthly_{t-4} + b_{5,13}^{4} m2_data_monthly_{t-4} + b_{5,14}^{4} m2_real_data_monthly_{t-4} + b_{5,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{5,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{5,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{5,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{5,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{5,20}^{4} job_openings_data_monthly_{t-4} + b_{5,21}^{4} job_hires_data_monthly_{t-4} + b_{5,22}^{4} job_separations_data_monthly_{t-4} + b_{5,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{5,24}^{4} gdp_data_quarterly_{t-4} + b_{5,25}^{4} gdp_real_data_quarterly_{t-4} + b_{5,26}^{4} cpi_inflation_data_annual_{t-4} + b_{5,27}^{4} median_income_real_data_annual_{t-4} + b_{5,28}^{4} median_income_data_annual_{t-4} + b_{5,29}^{4} Close_{t-4} + b_{5,30}^{4} Volume_{t-4} + b_{5,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{5,2}^{5} cpi_data_monthly_{t-5} + b_{5,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{5,4}^{5} rec_sahm_data_monthly_{t-5} + b_{5,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{5,6}^{5} rec_nber_data_monthly_{t-5} + b_{5,7}^{5} unemployment_data_monthly_{t-5} + b_{5,8}^{5} unemployment_level_data_monthly_{t-5} + b_{5,9}^{5} all_employed_data_monthly_{t-5} + b_{5,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{5,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{5,12}^{5} m1_data_monthly_{t-5} + b_{5,13}^{5} m2_data_monthly_{t-5} + b_{5,14}^{5} m2_real_data_monthly_{t-5} + b_{5,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{5,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{5,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{5,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{5,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{5,20}^{5} job_openings_data_monthly_{t-5} + b_{5,21}^{5} job_hires_data_monthly_{t-5} + b_{5,22}^{5} job_separations_data_monthly_{t-5} + b_{5,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{5,24}^{5} gdp_data_quarterly_{t-5} + b_{5,25}^{5} gdp_real_data_quarterly_{t-5} + b_{5,26}^{5} cpi_inflation_data_annual_{t-5} + b_{5,27}^{5} median_income_real_data_annual_{t-5} + b_{5,28}^{5} median_income_data_annual_{t-5} + b_{5,29}^{5} Close_{t-5} + b_{5,30}^{5} Volume_{t-5} + b_{5,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{5,2}^{6} cpi_data_monthly_{t-6} + b_{5,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{5,4}^{6} rec_sahm_data_monthly_{t-6} + b_{5,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{5,6}^{6} rec_nber_data_monthly_{t-6} + b_{5,7}^{6} unemployment_data_monthly_{t-6} + b_{5,8}^{6} unemployment_level_data_monthly_{t-6} + b_{5,9}^{6} all_employed_data_monthly_{t-6} + b_{5,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{5,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{5,12}^{6} m1_data_monthly_{t-6} + b_{5,13}^{6} m2_data_monthly_{t-6} + b_{5,14}^{6} m2_real_data_monthly_{t-6} + b_{5,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{5,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{5,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{5,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{5,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{5,20}^{6} job_openings_data_monthly_{t-6} + b_{5,21}^{6} job_hires_data_monthly_{t-6} + b_{5,22}^{6} job_separations_data_monthly_{t-6} + b_{5,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{5,24}^{6} gdp_data_quarterly_{t-6} + b_{5,25}^{6} gdp_real_data_quarterly_{t-6} + b_{5,26}^{6} cpi_inflation_data_annual_{t-6} + b_{5,27}^{6} median_income_real_data_annual_{t-6} + b_{5,28}^{6} median_income_data_annual_{t-6} + b_{5,29}^{6} Close_{t-6} + b_{5,30}^{6} Volume_{t-6} + c_{5} + e_{t}^{rec_smooth_prob_data_monthly} \\
rec_nber_data_monthly_{t} &= b_{6,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{6,2}^{1} cpi_data_monthly_{t-1} + b_{6,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{6,4}^{1} rec_sahm_data_monthly_{t-1} + b_{6,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{6,6}^{1} rec_nber_data_monthly_{t-1} + b_{6,7}^{1} unemployment_data_monthly_{t-1} + b_{6,8}^{1} unemployment_level_data_monthly_{t-1} + b_{6,9}^{1} all_employed_data_monthly_{t-1} + b_{6,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{6,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{6,12}^{1} m1_data_monthly_{t-1} + b_{6,13}^{1} m2_data_monthly_{t-1} + b_{6,14}^{1} m2_real_data_monthly_{t-1} + b_{6,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{6,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{6,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{6,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{6,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{6,20}^{1} job_openings_data_monthly_{t-1} + b_{6,21}^{1} job_hires_data_monthly_{t-1} + b_{6,22}^{1} job_separations_data_monthly_{t-1} + b_{6,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{6,24}^{1} gdp_data_quarterly_{t-1} + b_{6,25}^{1} gdp_real_data_quarterly_{t-1} + b_{6,26}^{1} cpi_inflation_data_annual_{t-1} + b_{6,27}^{1} median_income_real_data_annual_{t-1} + b_{6,28}^{1} median_income_data_annual_{t-1} + b_{6,29}^{1} Close_{t-1} + b_{6,30}^{1} Volume_{t-1} + b_{6,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{6,2}^{2} cpi_data_monthly_{t-2} + b_{6,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{6,4}^{2} rec_sahm_data_monthly_{t-2} + b_{6,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{6,6}^{2} rec_nber_data_monthly_{t-2} + b_{6,7}^{2} unemployment_data_monthly_{t-2} + b_{6,8}^{2} unemployment_level_data_monthly_{t-2} + b_{6,9}^{2} all_employed_data_monthly_{t-2} + b_{6,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{6,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{6,12}^{2} m1_data_monthly_{t-2} + b_{6,13}^{2} m2_data_monthly_{t-2} + b_{6,14}^{2} m2_real_data_monthly_{t-2} + b_{6,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{6,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{6,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{6,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{6,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{6,20}^{2} job_openings_data_monthly_{t-2} + b_{6,21}^{2} job_hires_data_monthly_{t-2} + b_{6,22}^{2} job_separations_data_monthly_{t-2} + b_{6,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{6,24}^{2} gdp_data_quarterly_{t-2} + b_{6,25}^{2} gdp_real_data_quarterly_{t-2} + b_{6,26}^{2} cpi_inflation_data_annual_{t-2} + b_{6,27}^{2} median_income_real_data_annual_{t-2} + b_{6,28}^{2} median_income_data_annual_{t-2} + b_{6,29}^{2} Close_{t-2} + b_{6,30}^{2} Volume_{t-2} + b_{6,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{6,2}^{3} cpi_data_monthly_{t-3} + b_{6,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{6,4}^{3} rec_sahm_data_monthly_{t-3} + b_{6,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{6,6}^{3} rec_nber_data_monthly_{t-3} + b_{6,7}^{3} unemployment_data_monthly_{t-3} + b_{6,8}^{3} unemployment_level_data_monthly_{t-3} + b_{6,9}^{3} all_employed_data_monthly_{t-3} + b_{6,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{6,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{6,12}^{3} m1_data_monthly_{t-3} + b_{6,13}^{3} m2_data_monthly_{t-3} + b_{6,14}^{3} m2_real_data_monthly_{t-3} + b_{6,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{6,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{6,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{6,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{6,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{6,20}^{3} job_openings_data_monthly_{t-3} + b_{6,21}^{3} job_hires_data_monthly_{t-3} + b_{6,22}^{3} job_separations_data_monthly_{t-3} + b_{6,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{6,24}^{3} gdp_data_quarterly_{t-3} + b_{6,25}^{3} gdp_real_data_quarterly_{t-3} + b_{6,26}^{3} cpi_inflation_data_annual_{t-3} + b_{6,27}^{3} median_income_real_data_annual_{t-3} + b_{6,28}^{3} median_income_data_annual_{t-3} + b_{6,29}^{3} Close_{t-3} + b_{6,30}^{3} Volume_{t-3} + b_{6,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{6,2}^{4} cpi_data_monthly_{t-4} + b_{6,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{6,4}^{4} rec_sahm_data_monthly_{t-4} + b_{6,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{6,6}^{4} rec_nber_data_monthly_{t-4} + b_{6,7}^{4} unemployment_data_monthly_{t-4} + b_{6,8}^{4} unemployment_level_data_monthly_{t-4} + b_{6,9}^{4} all_employed_data_monthly_{t-4} + b_{6,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{6,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{6,12}^{4} m1_data_monthly_{t-4} + b_{6,13}^{4} m2_data_monthly_{t-4} + b_{6,14}^{4} m2_real_data_monthly_{t-4} + b_{6,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{6,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{6,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{6,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{6,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{6,20}^{4} job_openings_data_monthly_{t-4} + b_{6,21}^{4} job_hires_data_monthly_{t-4} + b_{6,22}^{4} job_separations_data_monthly_{t-4} + b_{6,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{6,24}^{4} gdp_data_quarterly_{t-4} + b_{6,25}^{4} gdp_real_data_quarterly_{t-4} + b_{6,26}^{4} cpi_inflation_data_annual_{t-4} + b_{6,27}^{4} median_income_real_data_annual_{t-4} + b_{6,28}^{4} median_income_data_annual_{t-4} + b_{6,29}^{4} Close_{t-4} + b_{6,30}^{4} Volume_{t-4} + b_{6,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{6,2}^{5} cpi_data_monthly_{t-5} + b_{6,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{6,4}^{5} rec_sahm_data_monthly_{t-5} + b_{6,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{6,6}^{5} rec_nber_data_monthly_{t-5} + b_{6,7}^{5} unemployment_data_monthly_{t-5} + b_{6,8}^{5} unemployment_level_data_monthly_{t-5} + b_{6,9}^{5} all_employed_data_monthly_{t-5} + b_{6,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{6,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{6,12}^{5} m1_data_monthly_{t-5} + b_{6,13}^{5} m2_data_monthly_{t-5} + b_{6,14}^{5} m2_real_data_monthly_{t-5} + b_{6,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{6,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{6,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{6,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{6,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{6,20}^{5} job_openings_data_monthly_{t-5} + b_{6,21}^{5} job_hires_data_monthly_{t-5} + b_{6,22}^{5} job_separations_data_monthly_{t-5} + b_{6,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{6,24}^{5} gdp_data_quarterly_{t-5} + b_{6,25}^{5} gdp_real_data_quarterly_{t-5} + b_{6,26}^{5} cpi_inflation_data_annual_{t-5} + b_{6,27}^{5} median_income_real_data_annual_{t-5} + b_{6,28}^{5} median_income_data_annual_{t-5} + b_{6,29}^{5} Close_{t-5} + b_{6,30}^{5} Volume_{t-5} + b_{6,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{6,2}^{6} cpi_data_monthly_{t-6} + b_{6,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{6,4}^{6} rec_sahm_data_monthly_{t-6} + b_{6,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{6,6}^{6} rec_nber_data_monthly_{t-6} + b_{6,7}^{6} unemployment_data_monthly_{t-6} + b_{6,8}^{6} unemployment_level_data_monthly_{t-6} + b_{6,9}^{6} all_employed_data_monthly_{t-6} + b_{6,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{6,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{6,12}^{6} m1_data_monthly_{t-6} + b_{6,13}^{6} m2_data_monthly_{t-6} + b_{6,14}^{6} m2_real_data_monthly_{t-6} + b_{6,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{6,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{6,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{6,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{6,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{6,20}^{6} job_openings_data_monthly_{t-6} + b_{6,21}^{6} job_hires_data_monthly_{t-6} + b_{6,22}^{6} job_separations_data_monthly_{t-6} + b_{6,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{6,24}^{6} gdp_data_quarterly_{t-6} + b_{6,25}^{6} gdp_real_data_quarterly_{t-6} + b_{6,26}^{6} cpi_inflation_data_annual_{t-6} + b_{6,27}^{6} median_income_real_data_annual_{t-6} + b_{6,28}^{6} median_income_data_annual_{t-6} + b_{6,29}^{6} Close_{t-6} + b_{6,30}^{6} Volume_{t-6} + c_{6} + e_{t}^{rec_nber_data_monthly} \\
unemployment_data_monthly_{t} &= b_{7,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{7,2}^{1} cpi_data_monthly_{t-1} + b_{7,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{7,4}^{1} rec_sahm_data_monthly_{t-1} + b_{7,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{7,6}^{1} rec_nber_data_monthly_{t-1} + b_{7,7}^{1} unemployment_data_monthly_{t-1} + b_{7,8}^{1} unemployment_level_data_monthly_{t-1} + b_{7,9}^{1} all_employed_data_monthly_{t-1} + b_{7,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{7,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{7,12}^{1} m1_data_monthly_{t-1} + b_{7,13}^{1} m2_data_monthly_{t-1} + b_{7,14}^{1} m2_real_data_monthly_{t-1} + b_{7,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{7,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{7,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{7,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{7,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{7,20}^{1} job_openings_data_monthly_{t-1} + b_{7,21}^{1} job_hires_data_monthly_{t-1} + b_{7,22}^{1} job_separations_data_monthly_{t-1} + b_{7,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{7,24}^{1} gdp_data_quarterly_{t-1} + b_{7,25}^{1} gdp_real_data_quarterly_{t-1} + b_{7,26}^{1} cpi_inflation_data_annual_{t-1} + b_{7,27}^{1} median_income_real_data_annual_{t-1} + b_{7,28}^{1} median_income_data_annual_{t-1} + b_{7,29}^{1} Close_{t-1} + b_{7,30}^{1} Volume_{t-1} + b_{7,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{7,2}^{2} cpi_data_monthly_{t-2} + b_{7,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{7,4}^{2} rec_sahm_data_monthly_{t-2} + b_{7,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{7,6}^{2} rec_nber_data_monthly_{t-2} + b_{7,7}^{2} unemployment_data_monthly_{t-2} + b_{7,8}^{2} unemployment_level_data_monthly_{t-2} + b_{7,9}^{2} all_employed_data_monthly_{t-2} + b_{7,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{7,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{7,12}^{2} m1_data_monthly_{t-2} + b_{7,13}^{2} m2_data_monthly_{t-2} + b_{7,14}^{2} m2_real_data_monthly_{t-2} + b_{7,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{7,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{7,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{7,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{7,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{7,20}^{2} job_openings_data_monthly_{t-2} + b_{7,21}^{2} job_hires_data_monthly_{t-2} + b_{7,22}^{2} job_separations_data_monthly_{t-2} + b_{7,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{7,24}^{2} gdp_data_quarterly_{t-2} + b_{7,25}^{2} gdp_real_data_quarterly_{t-2} + b_{7,26}^{2} cpi_inflation_data_annual_{t-2} + b_{7,27}^{2} median_income_real_data_annual_{t-2} + b_{7,28}^{2} median_income_data_annual_{t-2} + b_{7,29}^{2} Close_{t-2} + b_{7,30}^{2} Volume_{t-2} + b_{7,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{7,2}^{3} cpi_data_monthly_{t-3} + b_{7,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{7,4}^{3} rec_sahm_data_monthly_{t-3} + b_{7,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{7,6}^{3} rec_nber_data_monthly_{t-3} + b_{7,7}^{3} unemployment_data_monthly_{t-3} + b_{7,8}^{3} unemployment_level_data_monthly_{t-3} + b_{7,9}^{3} all_employed_data_monthly_{t-3} + b_{7,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{7,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{7,12}^{3} m1_data_monthly_{t-3} + b_{7,13}^{3} m2_data_monthly_{t-3} + b_{7,14}^{3} m2_real_data_monthly_{t-3} + b_{7,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{7,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{7,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{7,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{7,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{7,20}^{3} job_openings_data_monthly_{t-3} + b_{7,21}^{3} job_hires_data_monthly_{t-3} + b_{7,22}^{3} job_separations_data_monthly_{t-3} + b_{7,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{7,24}^{3} gdp_data_quarterly_{t-3} + b_{7,25}^{3} gdp_real_data_quarterly_{t-3} + b_{7,26}^{3} cpi_inflation_data_annual_{t-3} + b_{7,27}^{3} median_income_real_data_annual_{t-3} + b_{7,28}^{3} median_income_data_annual_{t-3} + b_{7,29}^{3} Close_{t-3} + b_{7,30}^{3} Volume_{t-3} + b_{7,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{7,2}^{4} cpi_data_monthly_{t-4} + b_{7,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{7,4}^{4} rec_sahm_data_monthly_{t-4} + b_{7,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{7,6}^{4} rec_nber_data_monthly_{t-4} + b_{7,7}^{4} unemployment_data_monthly_{t-4} + b_{7,8}^{4} unemployment_level_data_monthly_{t-4} + b_{7,9}^{4} all_employed_data_monthly_{t-4} + b_{7,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{7,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{7,12}^{4} m1_data_monthly_{t-4} + b_{7,13}^{4} m2_data_monthly_{t-4} + b_{7,14}^{4} m2_real_data_monthly_{t-4} + b_{7,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{7,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{7,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{7,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{7,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{7,20}^{4} job_openings_data_monthly_{t-4} + b_{7,21}^{4} job_hires_data_monthly_{t-4} + b_{7,22}^{4} job_separations_data_monthly_{t-4} + b_{7,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{7,24}^{4} gdp_data_quarterly_{t-4} + b_{7,25}^{4} gdp_real_data_quarterly_{t-4} + b_{7,26}^{4} cpi_inflation_data_annual_{t-4} + b_{7,27}^{4} median_income_real_data_annual_{t-4} + b_{7,28}^{4} median_income_data_annual_{t-4} + b_{7,29}^{4} Close_{t-4} + b_{7,30}^{4} Volume_{t-4} + b_{7,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{7,2}^{5} cpi_data_monthly_{t-5} + b_{7,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{7,4}^{5} rec_sahm_data_monthly_{t-5} + b_{7,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{7,6}^{5} rec_nber_data_monthly_{t-5} + b_{7,7}^{5} unemployment_data_monthly_{t-5} + b_{7,8}^{5} unemployment_level_data_monthly_{t-5} + b_{7,9}^{5} all_employed_data_monthly_{t-5} + b_{7,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{7,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{7,12}^{5} m1_data_monthly_{t-5} + b_{7,13}^{5} m2_data_monthly_{t-5} + b_{7,14}^{5} m2_real_data_monthly_{t-5} + b_{7,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{7,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{7,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{7,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{7,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{7,20}^{5} job_openings_data_monthly_{t-5} + b_{7,21}^{5} job_hires_data_monthly_{t-5} + b_{7,22}^{5} job_separations_data_monthly_{t-5} + b_{7,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{7,24}^{5} gdp_data_quarterly_{t-5} + b_{7,25}^{5} gdp_real_data_quarterly_{t-5} + b_{7,26}^{5} cpi_inflation_data_annual_{t-5} + b_{7,27}^{5} median_income_real_data_annual_{t-5} + b_{7,28}^{5} median_income_data_annual_{t-5} + b_{7,29}^{5} Close_{t-5} + b_{7,30}^{5} Volume_{t-5} + b_{7,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{7,2}^{6} cpi_data_monthly_{t-6} + b_{7,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{7,4}^{6} rec_sahm_data_monthly_{t-6} + b_{7,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{7,6}^{6} rec_nber_data_monthly_{t-6} + b_{7,7}^{6} unemployment_data_monthly_{t-6} + b_{7,8}^{6} unemployment_level_data_monthly_{t-6} + b_{7,9}^{6} all_employed_data_monthly_{t-6} + b_{7,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{7,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{7,12}^{6} m1_data_monthly_{t-6} + b_{7,13}^{6} m2_data_monthly_{t-6} + b_{7,14}^{6} m2_real_data_monthly_{t-6} + b_{7,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{7,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{7,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{7,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{7,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{7,20}^{6} job_openings_data_monthly_{t-6} + b_{7,21}^{6} job_hires_data_monthly_{t-6} + b_{7,22}^{6} job_separations_data_monthly_{t-6} + b_{7,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{7,24}^{6} gdp_data_quarterly_{t-6} + b_{7,25}^{6} gdp_real_data_quarterly_{t-6} + b_{7,26}^{6} cpi_inflation_data_annual_{t-6} + b_{7,27}^{6} median_income_real_data_annual_{t-6} + b_{7,28}^{6} median_income_data_annual_{t-6} + b_{7,29}^{6} Close_{t-6} + b_{7,30}^{6} Volume_{t-6} + c_{7} + e_{t}^{unemployment_data_monthly} \\
unemployment_level_data_monthly_{t} &= b_{8,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{8,2}^{1} cpi_data_monthly_{t-1} + b_{8,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{8,4}^{1} rec_sahm_data_monthly_{t-1} + b_{8,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{8,6}^{1} rec_nber_data_monthly_{t-1} + b_{8,7}^{1} unemployment_data_monthly_{t-1} + b_{8,8}^{1} unemployment_level_data_monthly_{t-1} + b_{8,9}^{1} all_employed_data_monthly_{t-1} + b_{8,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{8,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{8,12}^{1} m1_data_monthly_{t-1} + b_{8,13}^{1} m2_data_monthly_{t-1} + b_{8,14}^{1} m2_real_data_monthly_{t-1} + b_{8,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{8,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{8,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{8,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{8,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{8,20}^{1} job_openings_data_monthly_{t-1} + b_{8,21}^{1} job_hires_data_monthly_{t-1} + b_{8,22}^{1} job_separations_data_monthly_{t-1} + b_{8,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{8,24}^{1} gdp_data_quarterly_{t-1} + b_{8,25}^{1} gdp_real_data_quarterly_{t-1} + b_{8,26}^{1} cpi_inflation_data_annual_{t-1} + b_{8,27}^{1} median_income_real_data_annual_{t-1} + b_{8,28}^{1} median_income_data_annual_{t-1} + b_{8,29}^{1} Close_{t-1} + b_{8,30}^{1} Volume_{t-1} + b_{8,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{8,2}^{2} cpi_data_monthly_{t-2} + b_{8,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{8,4}^{2} rec_sahm_data_monthly_{t-2} + b_{8,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{8,6}^{2} rec_nber_data_monthly_{t-2} + b_{8,7}^{2} unemployment_data_monthly_{t-2} + b_{8,8}^{2} unemployment_level_data_monthly_{t-2} + b_{8,9}^{2} all_employed_data_monthly_{t-2} + b_{8,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{8,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{8,12}^{2} m1_data_monthly_{t-2} + b_{8,13}^{2} m2_data_monthly_{t-2} + b_{8,14}^{2} m2_real_data_monthly_{t-2} + b_{8,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{8,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{8,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{8,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{8,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{8,20}^{2} job_openings_data_monthly_{t-2} + b_{8,21}^{2} job_hires_data_monthly_{t-2} + b_{8,22}^{2} job_separations_data_monthly_{t-2} + b_{8,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{8,24}^{2} gdp_data_quarterly_{t-2} + b_{8,25}^{2} gdp_real_data_quarterly_{t-2} + b_{8,26}^{2} cpi_inflation_data_annual_{t-2} + b_{8,27}^{2} median_income_real_data_annual_{t-2} + b_{8,28}^{2} median_income_data_annual_{t-2} + b_{8,29}^{2} Close_{t-2} + b_{8,30}^{2} Volume_{t-2} + b_{8,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{8,2}^{3} cpi_data_monthly_{t-3} + b_{8,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{8,4}^{3} rec_sahm_data_monthly_{t-3} + b_{8,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{8,6}^{3} rec_nber_data_monthly_{t-3} + b_{8,7}^{3} unemployment_data_monthly_{t-3} + b_{8,8}^{3} unemployment_level_data_monthly_{t-3} + b_{8,9}^{3} all_employed_data_monthly_{t-3} + b_{8,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{8,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{8,12}^{3} m1_data_monthly_{t-3} + b_{8,13}^{3} m2_data_monthly_{t-3} + b_{8,14}^{3} m2_real_data_monthly_{t-3} + b_{8,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{8,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{8,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{8,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{8,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{8,20}^{3} job_openings_data_monthly_{t-3} + b_{8,21}^{3} job_hires_data_monthly_{t-3} + b_{8,22}^{3} job_separations_data_monthly_{t-3} + b_{8,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{8,24}^{3} gdp_data_quarterly_{t-3} + b_{8,25}^{3} gdp_real_data_quarterly_{t-3} + b_{8,26}^{3} cpi_inflation_data_annual_{t-3} + b_{8,27}^{3} median_income_real_data_annual_{t-3} + b_{8,28}^{3} median_income_data_annual_{t-3} + b_{8,29}^{3} Close_{t-3} + b_{8,30}^{3} Volume_{t-3} + b_{8,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{8,2}^{4} cpi_data_monthly_{t-4} + b_{8,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{8,4}^{4} rec_sahm_data_monthly_{t-4} + b_{8,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{8,6}^{4} rec_nber_data_monthly_{t-4} + b_{8,7}^{4} unemployment_data_monthly_{t-4} + b_{8,8}^{4} unemployment_level_data_monthly_{t-4} + b_{8,9}^{4} all_employed_data_monthly_{t-4} + b_{8,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{8,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{8,12}^{4} m1_data_monthly_{t-4} + b_{8,13}^{4} m2_data_monthly_{t-4} + b_{8,14}^{4} m2_real_data_monthly_{t-4} + b_{8,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{8,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{8,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{8,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{8,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{8,20}^{4} job_openings_data_monthly_{t-4} + b_{8,21}^{4} job_hires_data_monthly_{t-4} + b_{8,22}^{4} job_separations_data_monthly_{t-4} + b_{8,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{8,24}^{4} gdp_data_quarterly_{t-4} + b_{8,25}^{4} gdp_real_data_quarterly_{t-4} + b_{8,26}^{4} cpi_inflation_data_annual_{t-4} + b_{8,27}^{4} median_income_real_data_annual_{t-4} + b_{8,28}^{4} median_income_data_annual_{t-4} + b_{8,29}^{4} Close_{t-4} + b_{8,30}^{4} Volume_{t-4} + b_{8,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{8,2}^{5} cpi_data_monthly_{t-5} + b_{8,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{8,4}^{5} rec_sahm_data_monthly_{t-5} + b_{8,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{8,6}^{5} rec_nber_data_monthly_{t-5} + b_{8,7}^{5} unemployment_data_monthly_{t-5} + b_{8,8}^{5} unemployment_level_data_monthly_{t-5} + b_{8,9}^{5} all_employed_data_monthly_{t-5} + b_{8,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{8,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{8,12}^{5} m1_data_monthly_{t-5} + b_{8,13}^{5} m2_data_monthly_{t-5} + b_{8,14}^{5} m2_real_data_monthly_{t-5} + b_{8,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{8,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{8,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{8,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{8,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{8,20}^{5} job_openings_data_monthly_{t-5} + b_{8,21}^{5} job_hires_data_monthly_{t-5} + b_{8,22}^{5} job_separations_data_monthly_{t-5} + b_{8,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{8,24}^{5} gdp_data_quarterly_{t-5} + b_{8,25}^{5} gdp_real_data_quarterly_{t-5} + b_{8,26}^{5} cpi_inflation_data_annual_{t-5} + b_{8,27}^{5} median_income_real_data_annual_{t-5} + b_{8,28}^{5} median_income_data_annual_{t-5} + b_{8,29}^{5} Close_{t-5} + b_{8,30}^{5} Volume_{t-5} + b_{8,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{8,2}^{6} cpi_data_monthly_{t-6} + b_{8,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{8,4}^{6} rec_sahm_data_monthly_{t-6} + b_{8,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{8,6}^{6} rec_nber_data_monthly_{t-6} + b_{8,7}^{6} unemployment_data_monthly_{t-6} + b_{8,8}^{6} unemployment_level_data_monthly_{t-6} + b_{8,9}^{6} all_employed_data_monthly_{t-6} + b_{8,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{8,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{8,12}^{6} m1_data_monthly_{t-6} + b_{8,13}^{6} m2_data_monthly_{t-6} + b_{8,14}^{6} m2_real_data_monthly_{t-6} + b_{8,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{8,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{8,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{8,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{8,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{8,20}^{6} job_openings_data_monthly_{t-6} + b_{8,21}^{6} job_hires_data_monthly_{t-6} + b_{8,22}^{6} job_separations_data_monthly_{t-6} + b_{8,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{8,24}^{6} gdp_data_quarterly_{t-6} + b_{8,25}^{6} gdp_real_data_quarterly_{t-6} + b_{8,26}^{6} cpi_inflation_data_annual_{t-6} + b_{8,27}^{6} median_income_real_data_annual_{t-6} + b_{8,28}^{6} median_income_data_annual_{t-6} + b_{8,29}^{6} Close_{t-6} + b_{8,30}^{6} Volume_{t-6} + c_{8} + e_{t}^{unemployment_level_data_monthly} \\
all_employed_data_monthly_{t} &= b_{9,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{9,2}^{1} cpi_data_monthly_{t-1} + b_{9,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{9,4}^{1} rec_sahm_data_monthly_{t-1} + b_{9,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{9,6}^{1} rec_nber_data_monthly_{t-1} + b_{9,7}^{1} unemployment_data_monthly_{t-1} + b_{9,8}^{1} unemployment_level_data_monthly_{t-1} + b_{9,9}^{1} all_employed_data_monthly_{t-1} + b_{9,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{9,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{9,12}^{1} m1_data_monthly_{t-1} + b_{9,13}^{1} m2_data_monthly_{t-1} + b_{9,14}^{1} m2_real_data_monthly_{t-1} + b_{9,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{9,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{9,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{9,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{9,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{9,20}^{1} job_openings_data_monthly_{t-1} + b_{9,21}^{1} job_hires_data_monthly_{t-1} + b_{9,22}^{1} job_separations_data_monthly_{t-1} + b_{9,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{9,24}^{1} gdp_data_quarterly_{t-1} + b_{9,25}^{1} gdp_real_data_quarterly_{t-1} + b_{9,26}^{1} cpi_inflation_data_annual_{t-1} + b_{9,27}^{1} median_income_real_data_annual_{t-1} + b_{9,28}^{1} median_income_data_annual_{t-1} + b_{9,29}^{1} Close_{t-1} + b_{9,30}^{1} Volume_{t-1} + b_{9,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{9,2}^{2} cpi_data_monthly_{t-2} + b_{9,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{9,4}^{2} rec_sahm_data_monthly_{t-2} + b_{9,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{9,6}^{2} rec_nber_data_monthly_{t-2} + b_{9,7}^{2} unemployment_data_monthly_{t-2} + b_{9,8}^{2} unemployment_level_data_monthly_{t-2} + b_{9,9}^{2} all_employed_data_monthly_{t-2} + b_{9,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{9,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{9,12}^{2} m1_data_monthly_{t-2} + b_{9,13}^{2} m2_data_monthly_{t-2} + b_{9,14}^{2} m2_real_data_monthly_{t-2} + b_{9,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{9,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{9,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{9,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{9,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{9,20}^{2} job_openings_data_monthly_{t-2} + b_{9,21}^{2} job_hires_data_monthly_{t-2} + b_{9,22}^{2} job_separations_data_monthly_{t-2} + b_{9,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{9,24}^{2} gdp_data_quarterly_{t-2} + b_{9,25}^{2} gdp_real_data_quarterly_{t-2} + b_{9,26}^{2} cpi_inflation_data_annual_{t-2} + b_{9,27}^{2} median_income_real_data_annual_{t-2} + b_{9,28}^{2} median_income_data_annual_{t-2} + b_{9,29}^{2} Close_{t-2} + b_{9,30}^{2} Volume_{t-2} + b_{9,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{9,2}^{3} cpi_data_monthly_{t-3} + b_{9,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{9,4}^{3} rec_sahm_data_monthly_{t-3} + b_{9,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{9,6}^{3} rec_nber_data_monthly_{t-3} + b_{9,7}^{3} unemployment_data_monthly_{t-3} + b_{9,8}^{3} unemployment_level_data_monthly_{t-3} + b_{9,9}^{3} all_employed_data_monthly_{t-3} + b_{9,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{9,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{9,12}^{3} m1_data_monthly_{t-3} + b_{9,13}^{3} m2_data_monthly_{t-3} + b_{9,14}^{3} m2_real_data_monthly_{t-3} + b_{9,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{9,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{9,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{9,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{9,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{9,20}^{3} job_openings_data_monthly_{t-3} + b_{9,21}^{3} job_hires_data_monthly_{t-3} + b_{9,22}^{3} job_separations_data_monthly_{t-3} + b_{9,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{9,24}^{3} gdp_data_quarterly_{t-3} + b_{9,25}^{3} gdp_real_data_quarterly_{t-3} + b_{9,26}^{3} cpi_inflation_data_annual_{t-3} + b_{9,27}^{3} median_income_real_data_annual_{t-3} + b_{9,28}^{3} median_income_data_annual_{t-3} + b_{9,29}^{3} Close_{t-3} + b_{9,30}^{3} Volume_{t-3} + b_{9,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{9,2}^{4} cpi_data_monthly_{t-4} + b_{9,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{9,4}^{4} rec_sahm_data_monthly_{t-4} + b_{9,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{9,6}^{4} rec_nber_data_monthly_{t-4} + b_{9,7}^{4} unemployment_data_monthly_{t-4} + b_{9,8}^{4} unemployment_level_data_monthly_{t-4} + b_{9,9}^{4} all_employed_data_monthly_{t-4} + b_{9,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{9,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{9,12}^{4} m1_data_monthly_{t-4} + b_{9,13}^{4} m2_data_monthly_{t-4} + b_{9,14}^{4} m2_real_data_monthly_{t-4} + b_{9,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{9,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{9,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{9,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{9,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{9,20}^{4} job_openings_data_monthly_{t-4} + b_{9,21}^{4} job_hires_data_monthly_{t-4} + b_{9,22}^{4} job_separations_data_monthly_{t-4} + b_{9,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{9,24}^{4} gdp_data_quarterly_{t-4} + b_{9,25}^{4} gdp_real_data_quarterly_{t-4} + b_{9,26}^{4} cpi_inflation_data_annual_{t-4} + b_{9,27}^{4} median_income_real_data_annual_{t-4} + b_{9,28}^{4} median_income_data_annual_{t-4} + b_{9,29}^{4} Close_{t-4} + b_{9,30}^{4} Volume_{t-4} + b_{9,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{9,2}^{5} cpi_data_monthly_{t-5} + b_{9,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{9,4}^{5} rec_sahm_data_monthly_{t-5} + b_{9,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{9,6}^{5} rec_nber_data_monthly_{t-5} + b_{9,7}^{5} unemployment_data_monthly_{t-5} + b_{9,8}^{5} unemployment_level_data_monthly_{t-5} + b_{9,9}^{5} all_employed_data_monthly_{t-5} + b_{9,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{9,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{9,12}^{5} m1_data_monthly_{t-5} + b_{9,13}^{5} m2_data_monthly_{t-5} + b_{9,14}^{5} m2_real_data_monthly_{t-5} + b_{9,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{9,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{9,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{9,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{9,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{9,20}^{5} job_openings_data_monthly_{t-5} + b_{9,21}^{5} job_hires_data_monthly_{t-5} + b_{9,22}^{5} job_separations_data_monthly_{t-5} + b_{9,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{9,24}^{5} gdp_data_quarterly_{t-5} + b_{9,25}^{5} gdp_real_data_quarterly_{t-5} + b_{9,26}^{5} cpi_inflation_data_annual_{t-5} + b_{9,27}^{5} median_income_real_data_annual_{t-5} + b_{9,28}^{5} median_income_data_annual_{t-5} + b_{9,29}^{5} Close_{t-5} + b_{9,30}^{5} Volume_{t-5} + b_{9,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{9,2}^{6} cpi_data_monthly_{t-6} + b_{9,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{9,4}^{6} rec_sahm_data_monthly_{t-6} + b_{9,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{9,6}^{6} rec_nber_data_monthly_{t-6} + b_{9,7}^{6} unemployment_data_monthly_{t-6} + b_{9,8}^{6} unemployment_level_data_monthly_{t-6} + b_{9,9}^{6} all_employed_data_monthly_{t-6} + b_{9,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{9,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{9,12}^{6} m1_data_monthly_{t-6} + b_{9,13}^{6} m2_data_monthly_{t-6} + b_{9,14}^{6} m2_real_data_monthly_{t-6} + b_{9,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{9,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{9,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{9,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{9,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{9,20}^{6} job_openings_data_monthly_{t-6} + b_{9,21}^{6} job_hires_data_monthly_{t-6} + b_{9,22}^{6} job_separations_data_monthly_{t-6} + b_{9,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{9,24}^{6} gdp_data_quarterly_{t-6} + b_{9,25}^{6} gdp_real_data_quarterly_{t-6} + b_{9,26}^{6} cpi_inflation_data_annual_{t-6} + b_{9,27}^{6} median_income_real_data_annual_{t-6} + b_{9,28}^{6} median_income_data_annual_{t-6} + b_{9,29}^{6} Close_{t-6} + b_{9,30}^{6} Volume_{t-6} + c_{9} + e_{t}^{all_employed_data_monthly} \\
employment_population_ratio_data_monthly_{t} &= b_{10,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{10,2}^{1} cpi_data_monthly_{t-1} + b_{10,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{10,4}^{1} rec_sahm_data_monthly_{t-1} + b_{10,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{10,6}^{1} rec_nber_data_monthly_{t-1} + b_{10,7}^{1} unemployment_data_monthly_{t-1} + b_{10,8}^{1} unemployment_level_data_monthly_{t-1} + b_{10,9}^{1} all_employed_data_monthly_{t-1} + b_{10,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{10,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{10,12}^{1} m1_data_monthly_{t-1} + b_{10,13}^{1} m2_data_monthly_{t-1} + b_{10,14}^{1} m2_real_data_monthly_{t-1} + b_{10,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{10,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{10,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{10,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{10,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{10,20}^{1} job_openings_data_monthly_{t-1} + b_{10,21}^{1} job_hires_data_monthly_{t-1} + b_{10,22}^{1} job_separations_data_monthly_{t-1} + b_{10,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{10,24}^{1} gdp_data_quarterly_{t-1} + b_{10,25}^{1} gdp_real_data_quarterly_{t-1} + b_{10,26}^{1} cpi_inflation_data_annual_{t-1} + b_{10,27}^{1} median_income_real_data_annual_{t-1} + b_{10,28}^{1} median_income_data_annual_{t-1} + b_{10,29}^{1} Close_{t-1} + b_{10,30}^{1} Volume_{t-1} + b_{10,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{10,2}^{2} cpi_data_monthly_{t-2} + b_{10,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{10,4}^{2} rec_sahm_data_monthly_{t-2} + b_{10,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{10,6}^{2} rec_nber_data_monthly_{t-2} + b_{10,7}^{2} unemployment_data_monthly_{t-2} + b_{10,8}^{2} unemployment_level_data_monthly_{t-2} + b_{10,9}^{2} all_employed_data_monthly_{t-2} + b_{10,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{10,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{10,12}^{2} m1_data_monthly_{t-2} + b_{10,13}^{2} m2_data_monthly_{t-2} + b_{10,14}^{2} m2_real_data_monthly_{t-2} + b_{10,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{10,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{10,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{10,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{10,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{10,20}^{2} job_openings_data_monthly_{t-2} + b_{10,21}^{2} job_hires_data_monthly_{t-2} + b_{10,22}^{2} job_separations_data_monthly_{t-2} + b_{10,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{10,24}^{2} gdp_data_quarterly_{t-2} + b_{10,25}^{2} gdp_real_data_quarterly_{t-2} + b_{10,26}^{2} cpi_inflation_data_annual_{t-2} + b_{10,27}^{2} median_income_real_data_annual_{t-2} + b_{10,28}^{2} median_income_data_annual_{t-2} + b_{10,29}^{2} Close_{t-2} + b_{10,30}^{2} Volume_{t-2} + b_{10,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{10,2}^{3} cpi_data_monthly_{t-3} + b_{10,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{10,4}^{3} rec_sahm_data_monthly_{t-3} + b_{10,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{10,6}^{3} rec_nber_data_monthly_{t-3} + b_{10,7}^{3} unemployment_data_monthly_{t-3} + b_{10,8}^{3} unemployment_level_data_monthly_{t-3} + b_{10,9}^{3} all_employed_data_monthly_{t-3} + b_{10,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{10,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{10,12}^{3} m1_data_monthly_{t-3} + b_{10,13}^{3} m2_data_monthly_{t-3} + b_{10,14}^{3} m2_real_data_monthly_{t-3} + b_{10,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{10,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{10,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{10,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{10,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{10,20}^{3} job_openings_data_monthly_{t-3} + b_{10,21}^{3} job_hires_data_monthly_{t-3} + b_{10,22}^{3} job_separations_data_monthly_{t-3} + b_{10,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{10,24}^{3} gdp_data_quarterly_{t-3} + b_{10,25}^{3} gdp_real_data_quarterly_{t-3} + b_{10,26}^{3} cpi_inflation_data_annual_{t-3} + b_{10,27}^{3} median_income_real_data_annual_{t-3} + b_{10,28}^{3} median_income_data_annual_{t-3} + b_{10,29}^{3} Close_{t-3} + b_{10,30}^{3} Volume_{t-3} + b_{10,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{10,2}^{4} cpi_data_monthly_{t-4} + b_{10,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{10,4}^{4} rec_sahm_data_monthly_{t-4} + b_{10,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{10,6}^{4} rec_nber_data_monthly_{t-4} + b_{10,7}^{4} unemployment_data_monthly_{t-4} + b_{10,8}^{4} unemployment_level_data_monthly_{t-4} + b_{10,9}^{4} all_employed_data_monthly_{t-4} + b_{10,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{10,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{10,12}^{4} m1_data_monthly_{t-4} + b_{10,13}^{4} m2_data_monthly_{t-4} + b_{10,14}^{4} m2_real_data_monthly_{t-4} + b_{10,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{10,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{10,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{10,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{10,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{10,20}^{4} job_openings_data_monthly_{t-4} + b_{10,21}^{4} job_hires_data_monthly_{t-4} + b_{10,22}^{4} job_separations_data_monthly_{t-4} + b_{10,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{10,24}^{4} gdp_data_quarterly_{t-4} + b_{10,25}^{4} gdp_real_data_quarterly_{t-4} + b_{10,26}^{4} cpi_inflation_data_annual_{t-4} + b_{10,27}^{4} median_income_real_data_annual_{t-4} + b_{10,28}^{4} median_income_data_annual_{t-4} + b_{10,29}^{4} Close_{t-4} + b_{10,30}^{4} Volume_{t-4} + b_{10,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{10,2}^{5} cpi_data_monthly_{t-5} + b_{10,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{10,4}^{5} rec_sahm_data_monthly_{t-5} + b_{10,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{10,6}^{5} rec_nber_data_monthly_{t-5} + b_{10,7}^{5} unemployment_data_monthly_{t-5} + b_{10,8}^{5} unemployment_level_data_monthly_{t-5} + b_{10,9}^{5} all_employed_data_monthly_{t-5} + b_{10,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{10,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{10,12}^{5} m1_data_monthly_{t-5} + b_{10,13}^{5} m2_data_monthly_{t-5} + b_{10,14}^{5} m2_real_data_monthly_{t-5} + b_{10,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{10,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{10,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{10,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{10,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{10,20}^{5} job_openings_data_monthly_{t-5} + b_{10,21}^{5} job_hires_data_monthly_{t-5} + b_{10,22}^{5} job_separations_data_monthly_{t-5} + b_{10,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{10,24}^{5} gdp_data_quarterly_{t-5} + b_{10,25}^{5} gdp_real_data_quarterly_{t-5} + b_{10,26}^{5} cpi_inflation_data_annual_{t-5} + b_{10,27}^{5} median_income_real_data_annual_{t-5} + b_{10,28}^{5} median_income_data_annual_{t-5} + b_{10,29}^{5} Close_{t-5} + b_{10,30}^{5} Volume_{t-5} + b_{10,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{10,2}^{6} cpi_data_monthly_{t-6} + b_{10,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{10,4}^{6} rec_sahm_data_monthly_{t-6} + b_{10,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{10,6}^{6} rec_nber_data_monthly_{t-6} + b_{10,7}^{6} unemployment_data_monthly_{t-6} + b_{10,8}^{6} unemployment_level_data_monthly_{t-6} + b_{10,9}^{6} all_employed_data_monthly_{t-6} + b_{10,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{10,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{10,12}^{6} m1_data_monthly_{t-6} + b_{10,13}^{6} m2_data_monthly_{t-6} + b_{10,14}^{6} m2_real_data_monthly_{t-6} + b_{10,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{10,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{10,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{10,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{10,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{10,20}^{6} job_openings_data_monthly_{t-6} + b_{10,21}^{6} job_hires_data_monthly_{t-6} + b_{10,22}^{6} job_separations_data_monthly_{t-6} + b_{10,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{10,24}^{6} gdp_data_quarterly_{t-6} + b_{10,25}^{6} gdp_real_data_quarterly_{t-6} + b_{10,26}^{6} cpi_inflation_data_annual_{t-6} + b_{10,27}^{6} median_income_real_data_annual_{t-6} + b_{10,28}^{6} median_income_data_annual_{t-6} + b_{10,29}^{6} Close_{t-6} + b_{10,30}^{6} Volume_{t-6} + c_{10} + e_{t}^{employment_population_ratio_data_monthly} \\
labor_force_participation_data_monthly_{t} &= b_{11,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{11,2}^{1} cpi_data_monthly_{t-1} + b_{11,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{11,4}^{1} rec_sahm_data_monthly_{t-1} + b_{11,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{11,6}^{1} rec_nber_data_monthly_{t-1} + b_{11,7}^{1} unemployment_data_monthly_{t-1} + b_{11,8}^{1} unemployment_level_data_monthly_{t-1} + b_{11,9}^{1} all_employed_data_monthly_{t-1} + b_{11,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{11,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{11,12}^{1} m1_data_monthly_{t-1} + b_{11,13}^{1} m2_data_monthly_{t-1} + b_{11,14}^{1} m2_real_data_monthly_{t-1} + b_{11,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{11,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{11,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{11,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{11,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{11,20}^{1} job_openings_data_monthly_{t-1} + b_{11,21}^{1} job_hires_data_monthly_{t-1} + b_{11,22}^{1} job_separations_data_monthly_{t-1} + b_{11,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{11,24}^{1} gdp_data_quarterly_{t-1} + b_{11,25}^{1} gdp_real_data_quarterly_{t-1} + b_{11,26}^{1} cpi_inflation_data_annual_{t-1} + b_{11,27}^{1} median_income_real_data_annual_{t-1} + b_{11,28}^{1} median_income_data_annual_{t-1} + b_{11,29}^{1} Close_{t-1} + b_{11,30}^{1} Volume_{t-1} + b_{11,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{11,2}^{2} cpi_data_monthly_{t-2} + b_{11,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{11,4}^{2} rec_sahm_data_monthly_{t-2} + b_{11,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{11,6}^{2} rec_nber_data_monthly_{t-2} + b_{11,7}^{2} unemployment_data_monthly_{t-2} + b_{11,8}^{2} unemployment_level_data_monthly_{t-2} + b_{11,9}^{2} all_employed_data_monthly_{t-2} + b_{11,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{11,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{11,12}^{2} m1_data_monthly_{t-2} + b_{11,13}^{2} m2_data_monthly_{t-2} + b_{11,14}^{2} m2_real_data_monthly_{t-2} + b_{11,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{11,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{11,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{11,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{11,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{11,20}^{2} job_openings_data_monthly_{t-2} + b_{11,21}^{2} job_hires_data_monthly_{t-2} + b_{11,22}^{2} job_separations_data_monthly_{t-2} + b_{11,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{11,24}^{2} gdp_data_quarterly_{t-2} + b_{11,25}^{2} gdp_real_data_quarterly_{t-2} + b_{11,26}^{2} cpi_inflation_data_annual_{t-2} + b_{11,27}^{2} median_income_real_data_annual_{t-2} + b_{11,28}^{2} median_income_data_annual_{t-2} + b_{11,29}^{2} Close_{t-2} + b_{11,30}^{2} Volume_{t-2} + b_{11,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{11,2}^{3} cpi_data_monthly_{t-3} + b_{11,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{11,4}^{3} rec_sahm_data_monthly_{t-3} + b_{11,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{11,6}^{3} rec_nber_data_monthly_{t-3} + b_{11,7}^{3} unemployment_data_monthly_{t-3} + b_{11,8}^{3} unemployment_level_data_monthly_{t-3} + b_{11,9}^{3} all_employed_data_monthly_{t-3} + b_{11,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{11,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{11,12}^{3} m1_data_monthly_{t-3} + b_{11,13}^{3} m2_data_monthly_{t-3} + b_{11,14}^{3} m2_real_data_monthly_{t-3} + b_{11,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{11,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{11,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{11,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{11,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{11,20}^{3} job_openings_data_monthly_{t-3} + b_{11,21}^{3} job_hires_data_monthly_{t-3} + b_{11,22}^{3} job_separations_data_monthly_{t-3} + b_{11,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{11,24}^{3} gdp_data_quarterly_{t-3} + b_{11,25}^{3} gdp_real_data_quarterly_{t-3} + b_{11,26}^{3} cpi_inflation_data_annual_{t-3} + b_{11,27}^{3} median_income_real_data_annual_{t-3} + b_{11,28}^{3} median_income_data_annual_{t-3} + b_{11,29}^{3} Close_{t-3} + b_{11,30}^{3} Volume_{t-3} + b_{11,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{11,2}^{4} cpi_data_monthly_{t-4} + b_{11,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{11,4}^{4} rec_sahm_data_monthly_{t-4} + b_{11,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{11,6}^{4} rec_nber_data_monthly_{t-4} + b_{11,7}^{4} unemployment_data_monthly_{t-4} + b_{11,8}^{4} unemployment_level_data_monthly_{t-4} + b_{11,9}^{4} all_employed_data_monthly_{t-4} + b_{11,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{11,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{11,12}^{4} m1_data_monthly_{t-4} + b_{11,13}^{4} m2_data_monthly_{t-4} + b_{11,14}^{4} m2_real_data_monthly_{t-4} + b_{11,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{11,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{11,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{11,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{11,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{11,20}^{4} job_openings_data_monthly_{t-4} + b_{11,21}^{4} job_hires_data_monthly_{t-4} + b_{11,22}^{4} job_separations_data_monthly_{t-4} + b_{11,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{11,24}^{4} gdp_data_quarterly_{t-4} + b_{11,25}^{4} gdp_real_data_quarterly_{t-4} + b_{11,26}^{4} cpi_inflation_data_annual_{t-4} + b_{11,27}^{4} median_income_real_data_annual_{t-4} + b_{11,28}^{4} median_income_data_annual_{t-4} + b_{11,29}^{4} Close_{t-4} + b_{11,30}^{4} Volume_{t-4} + b_{11,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{11,2}^{5} cpi_data_monthly_{t-5} + b_{11,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{11,4}^{5} rec_sahm_data_monthly_{t-5} + b_{11,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{11,6}^{5} rec_nber_data_monthly_{t-5} + b_{11,7}^{5} unemployment_data_monthly_{t-5} + b_{11,8}^{5} unemployment_level_data_monthly_{t-5} + b_{11,9}^{5} all_employed_data_monthly_{t-5} + b_{11,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{11,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{11,12}^{5} m1_data_monthly_{t-5} + b_{11,13}^{5} m2_data_monthly_{t-5} + b_{11,14}^{5} m2_real_data_monthly_{t-5} + b_{11,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{11,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{11,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{11,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{11,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{11,20}^{5} job_openings_data_monthly_{t-5} + b_{11,21}^{5} job_hires_data_monthly_{t-5} + b_{11,22}^{5} job_separations_data_monthly_{t-5} + b_{11,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{11,24}^{5} gdp_data_quarterly_{t-5} + b_{11,25}^{5} gdp_real_data_quarterly_{t-5} + b_{11,26}^{5} cpi_inflation_data_annual_{t-5} + b_{11,27}^{5} median_income_real_data_annual_{t-5} + b_{11,28}^{5} median_income_data_annual_{t-5} + b_{11,29}^{5} Close_{t-5} + b_{11,30}^{5} Volume_{t-5} + b_{11,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{11,2}^{6} cpi_data_monthly_{t-6} + b_{11,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{11,4}^{6} rec_sahm_data_monthly_{t-6} + b_{11,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{11,6}^{6} rec_nber_data_monthly_{t-6} + b_{11,7}^{6} unemployment_data_monthly_{t-6} + b_{11,8}^{6} unemployment_level_data_monthly_{t-6} + b_{11,9}^{6} all_employed_data_monthly_{t-6} + b_{11,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{11,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{11,12}^{6} m1_data_monthly_{t-6} + b_{11,13}^{6} m2_data_monthly_{t-6} + b_{11,14}^{6} m2_real_data_monthly_{t-6} + b_{11,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{11,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{11,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{11,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{11,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{11,20}^{6} job_openings_data_monthly_{t-6} + b_{11,21}^{6} job_hires_data_monthly_{t-6} + b_{11,22}^{6} job_separations_data_monthly_{t-6} + b_{11,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{11,24}^{6} gdp_data_quarterly_{t-6} + b_{11,25}^{6} gdp_real_data_quarterly_{t-6} + b_{11,26}^{6} cpi_inflation_data_annual_{t-6} + b_{11,27}^{6} median_income_real_data_annual_{t-6} + b_{11,28}^{6} median_income_data_annual_{t-6} + b_{11,29}^{6} Close_{t-6} + b_{11,30}^{6} Volume_{t-6} + c_{11} + e_{t}^{labor_force_participation_data_monthly} \\
m1_data_monthly_{t} &= b_{12,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{12,2}^{1} cpi_data_monthly_{t-1} + b_{12,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{12,4}^{1} rec_sahm_data_monthly_{t-1} + b_{12,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{12,6}^{1} rec_nber_data_monthly_{t-1} + b_{12,7}^{1} unemployment_data_monthly_{t-1} + b_{12,8}^{1} unemployment_level_data_monthly_{t-1} + b_{12,9}^{1} all_employed_data_monthly_{t-1} + b_{12,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{12,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{12,12}^{1} m1_data_monthly_{t-1} + b_{12,13}^{1} m2_data_monthly_{t-1} + b_{12,14}^{1} m2_real_data_monthly_{t-1} + b_{12,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{12,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{12,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{12,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{12,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{12,20}^{1} job_openings_data_monthly_{t-1} + b_{12,21}^{1} job_hires_data_monthly_{t-1} + b_{12,22}^{1} job_separations_data_monthly_{t-1} + b_{12,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{12,24}^{1} gdp_data_quarterly_{t-1} + b_{12,25}^{1} gdp_real_data_quarterly_{t-1} + b_{12,26}^{1} cpi_inflation_data_annual_{t-1} + b_{12,27}^{1} median_income_real_data_annual_{t-1} + b_{12,28}^{1} median_income_data_annual_{t-1} + b_{12,29}^{1} Close_{t-1} + b_{12,30}^{1} Volume_{t-1} + b_{12,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{12,2}^{2} cpi_data_monthly_{t-2} + b_{12,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{12,4}^{2} rec_sahm_data_monthly_{t-2} + b_{12,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{12,6}^{2} rec_nber_data_monthly_{t-2} + b_{12,7}^{2} unemployment_data_monthly_{t-2} + b_{12,8}^{2} unemployment_level_data_monthly_{t-2} + b_{12,9}^{2} all_employed_data_monthly_{t-2} + b_{12,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{12,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{12,12}^{2} m1_data_monthly_{t-2} + b_{12,13}^{2} m2_data_monthly_{t-2} + b_{12,14}^{2} m2_real_data_monthly_{t-2} + b_{12,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{12,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{12,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{12,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{12,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{12,20}^{2} job_openings_data_monthly_{t-2} + b_{12,21}^{2} job_hires_data_monthly_{t-2} + b_{12,22}^{2} job_separations_data_monthly_{t-2} + b_{12,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{12,24}^{2} gdp_data_quarterly_{t-2} + b_{12,25}^{2} gdp_real_data_quarterly_{t-2} + b_{12,26}^{2} cpi_inflation_data_annual_{t-2} + b_{12,27}^{2} median_income_real_data_annual_{t-2} + b_{12,28}^{2} median_income_data_annual_{t-2} + b_{12,29}^{2} Close_{t-2} + b_{12,30}^{2} Volume_{t-2} + b_{12,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{12,2}^{3} cpi_data_monthly_{t-3} + b_{12,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{12,4}^{3} rec_sahm_data_monthly_{t-3} + b_{12,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{12,6}^{3} rec_nber_data_monthly_{t-3} + b_{12,7}^{3} unemployment_data_monthly_{t-3} + b_{12,8}^{3} unemployment_level_data_monthly_{t-3} + b_{12,9}^{3} all_employed_data_monthly_{t-3} + b_{12,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{12,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{12,12}^{3} m1_data_monthly_{t-3} + b_{12,13}^{3} m2_data_monthly_{t-3} + b_{12,14}^{3} m2_real_data_monthly_{t-3} + b_{12,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{12,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{12,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{12,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{12,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{12,20}^{3} job_openings_data_monthly_{t-3} + b_{12,21}^{3} job_hires_data_monthly_{t-3} + b_{12,22}^{3} job_separations_data_monthly_{t-3} + b_{12,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{12,24}^{3} gdp_data_quarterly_{t-3} + b_{12,25}^{3} gdp_real_data_quarterly_{t-3} + b_{12,26}^{3} cpi_inflation_data_annual_{t-3} + b_{12,27}^{3} median_income_real_data_annual_{t-3} + b_{12,28}^{3} median_income_data_annual_{t-3} + b_{12,29}^{3} Close_{t-3} + b_{12,30}^{3} Volume_{t-3} + b_{12,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{12,2}^{4} cpi_data_monthly_{t-4} + b_{12,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{12,4}^{4} rec_sahm_data_monthly_{t-4} + b_{12,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{12,6}^{4} rec_nber_data_monthly_{t-4} + b_{12,7}^{4} unemployment_data_monthly_{t-4} + b_{12,8}^{4} unemployment_level_data_monthly_{t-4} + b_{12,9}^{4} all_employed_data_monthly_{t-4} + b_{12,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{12,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{12,12}^{4} m1_data_monthly_{t-4} + b_{12,13}^{4} m2_data_monthly_{t-4} + b_{12,14}^{4} m2_real_data_monthly_{t-4} + b_{12,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{12,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{12,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{12,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{12,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{12,20}^{4} job_openings_data_monthly_{t-4} + b_{12,21}^{4} job_hires_data_monthly_{t-4} + b_{12,22}^{4} job_separations_data_monthly_{t-4} + b_{12,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{12,24}^{4} gdp_data_quarterly_{t-4} + b_{12,25}^{4} gdp_real_data_quarterly_{t-4} + b_{12,26}^{4} cpi_inflation_data_annual_{t-4} + b_{12,27}^{4} median_income_real_data_annual_{t-4} + b_{12,28}^{4} median_income_data_annual_{t-4} + b_{12,29}^{4} Close_{t-4} + b_{12,30}^{4} Volume_{t-4} + b_{12,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{12,2}^{5} cpi_data_monthly_{t-5} + b_{12,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{12,4}^{5} rec_sahm_data_monthly_{t-5} + b_{12,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{12,6}^{5} rec_nber_data_monthly_{t-5} + b_{12,7}^{5} unemployment_data_monthly_{t-5} + b_{12,8}^{5} unemployment_level_data_monthly_{t-5} + b_{12,9}^{5} all_employed_data_monthly_{t-5} + b_{12,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{12,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{12,12}^{5} m1_data_monthly_{t-5} + b_{12,13}^{5} m2_data_monthly_{t-5} + b_{12,14}^{5} m2_real_data_monthly_{t-5} + b_{12,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{12,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{12,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{12,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{12,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{12,20}^{5} job_openings_data_monthly_{t-5} + b_{12,21}^{5} job_hires_data_monthly_{t-5} + b_{12,22}^{5} job_separations_data_monthly_{t-5} + b_{12,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{12,24}^{5} gdp_data_quarterly_{t-5} + b_{12,25}^{5} gdp_real_data_quarterly_{t-5} + b_{12,26}^{5} cpi_inflation_data_annual_{t-5} + b_{12,27}^{5} median_income_real_data_annual_{t-5} + b_{12,28}^{5} median_income_data_annual_{t-5} + b_{12,29}^{5} Close_{t-5} + b_{12,30}^{5} Volume_{t-5} + b_{12,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{12,2}^{6} cpi_data_monthly_{t-6} + b_{12,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{12,4}^{6} rec_sahm_data_monthly_{t-6} + b_{12,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{12,6}^{6} rec_nber_data_monthly_{t-6} + b_{12,7}^{6} unemployment_data_monthly_{t-6} + b_{12,8}^{6} unemployment_level_data_monthly_{t-6} + b_{12,9}^{6} all_employed_data_monthly_{t-6} + b_{12,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{12,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{12,12}^{6} m1_data_monthly_{t-6} + b_{12,13}^{6} m2_data_monthly_{t-6} + b_{12,14}^{6} m2_real_data_monthly_{t-6} + b_{12,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{12,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{12,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{12,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{12,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{12,20}^{6} job_openings_data_monthly_{t-6} + b_{12,21}^{6} job_hires_data_monthly_{t-6} + b_{12,22}^{6} job_separations_data_monthly_{t-6} + b_{12,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{12,24}^{6} gdp_data_quarterly_{t-6} + b_{12,25}^{6} gdp_real_data_quarterly_{t-6} + b_{12,26}^{6} cpi_inflation_data_annual_{t-6} + b_{12,27}^{6} median_income_real_data_annual_{t-6} + b_{12,28}^{6} median_income_data_annual_{t-6} + b_{12,29}^{6} Close_{t-6} + b_{12,30}^{6} Volume_{t-6} + c_{12} + e_{t}^{m1_data_monthly} \\
m2_data_monthly_{t} &= b_{13,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{13,2}^{1} cpi_data_monthly_{t-1} + b_{13,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{13,4}^{1} rec_sahm_data_monthly_{t-1} + b_{13,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{13,6}^{1} rec_nber_data_monthly_{t-1} + b_{13,7}^{1} unemployment_data_monthly_{t-1} + b_{13,8}^{1} unemployment_level_data_monthly_{t-1} + b_{13,9}^{1} all_employed_data_monthly_{t-1} + b_{13,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{13,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{13,12}^{1} m1_data_monthly_{t-1} + b_{13,13}^{1} m2_data_monthly_{t-1} + b_{13,14}^{1} m2_real_data_monthly_{t-1} + b_{13,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{13,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{13,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{13,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{13,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{13,20}^{1} job_openings_data_monthly_{t-1} + b_{13,21}^{1} job_hires_data_monthly_{t-1} + b_{13,22}^{1} job_separations_data_monthly_{t-1} + b_{13,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{13,24}^{1} gdp_data_quarterly_{t-1} + b_{13,25}^{1} gdp_real_data_quarterly_{t-1} + b_{13,26}^{1} cpi_inflation_data_annual_{t-1} + b_{13,27}^{1} median_income_real_data_annual_{t-1} + b_{13,28}^{1} median_income_data_annual_{t-1} + b_{13,29}^{1} Close_{t-1} + b_{13,30}^{1} Volume_{t-1} + b_{13,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{13,2}^{2} cpi_data_monthly_{t-2} + b_{13,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{13,4}^{2} rec_sahm_data_monthly_{t-2} + b_{13,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{13,6}^{2} rec_nber_data_monthly_{t-2} + b_{13,7}^{2} unemployment_data_monthly_{t-2} + b_{13,8}^{2} unemployment_level_data_monthly_{t-2} + b_{13,9}^{2} all_employed_data_monthly_{t-2} + b_{13,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{13,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{13,12}^{2} m1_data_monthly_{t-2} + b_{13,13}^{2} m2_data_monthly_{t-2} + b_{13,14}^{2} m2_real_data_monthly_{t-2} + b_{13,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{13,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{13,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{13,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{13,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{13,20}^{2} job_openings_data_monthly_{t-2} + b_{13,21}^{2} job_hires_data_monthly_{t-2} + b_{13,22}^{2} job_separations_data_monthly_{t-2} + b_{13,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{13,24}^{2} gdp_data_quarterly_{t-2} + b_{13,25}^{2} gdp_real_data_quarterly_{t-2} + b_{13,26}^{2} cpi_inflation_data_annual_{t-2} + b_{13,27}^{2} median_income_real_data_annual_{t-2} + b_{13,28}^{2} median_income_data_annual_{t-2} + b_{13,29}^{2} Close_{t-2} + b_{13,30}^{2} Volume_{t-2} + b_{13,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{13,2}^{3} cpi_data_monthly_{t-3} + b_{13,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{13,4}^{3} rec_sahm_data_monthly_{t-3} + b_{13,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{13,6}^{3} rec_nber_data_monthly_{t-3} + b_{13,7}^{3} unemployment_data_monthly_{t-3} + b_{13,8}^{3} unemployment_level_data_monthly_{t-3} + b_{13,9}^{3} all_employed_data_monthly_{t-3} + b_{13,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{13,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{13,12}^{3} m1_data_monthly_{t-3} + b_{13,13}^{3} m2_data_monthly_{t-3} + b_{13,14}^{3} m2_real_data_monthly_{t-3} + b_{13,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{13,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{13,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{13,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{13,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{13,20}^{3} job_openings_data_monthly_{t-3} + b_{13,21}^{3} job_hires_data_monthly_{t-3} + b_{13,22}^{3} job_separations_data_monthly_{t-3} + b_{13,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{13,24}^{3} gdp_data_quarterly_{t-3} + b_{13,25}^{3} gdp_real_data_quarterly_{t-3} + b_{13,26}^{3} cpi_inflation_data_annual_{t-3} + b_{13,27}^{3} median_income_real_data_annual_{t-3} + b_{13,28}^{3} median_income_data_annual_{t-3} + b_{13,29}^{3} Close_{t-3} + b_{13,30}^{3} Volume_{t-3} + b_{13,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{13,2}^{4} cpi_data_monthly_{t-4} + b_{13,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{13,4}^{4} rec_sahm_data_monthly_{t-4} + b_{13,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{13,6}^{4} rec_nber_data_monthly_{t-4} + b_{13,7}^{4} unemployment_data_monthly_{t-4} + b_{13,8}^{4} unemployment_level_data_monthly_{t-4} + b_{13,9}^{4} all_employed_data_monthly_{t-4} + b_{13,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{13,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{13,12}^{4} m1_data_monthly_{t-4} + b_{13,13}^{4} m2_data_monthly_{t-4} + b_{13,14}^{4} m2_real_data_monthly_{t-4} + b_{13,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{13,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{13,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{13,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{13,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{13,20}^{4} job_openings_data_monthly_{t-4} + b_{13,21}^{4} job_hires_data_monthly_{t-4} + b_{13,22}^{4} job_separations_data_monthly_{t-4} + b_{13,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{13,24}^{4} gdp_data_quarterly_{t-4} + b_{13,25}^{4} gdp_real_data_quarterly_{t-4} + b_{13,26}^{4} cpi_inflation_data_annual_{t-4} + b_{13,27}^{4} median_income_real_data_annual_{t-4} + b_{13,28}^{4} median_income_data_annual_{t-4} + b_{13,29}^{4} Close_{t-4} + b_{13,30}^{4} Volume_{t-4} + b_{13,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{13,2}^{5} cpi_data_monthly_{t-5} + b_{13,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{13,4}^{5} rec_sahm_data_monthly_{t-5} + b_{13,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{13,6}^{5} rec_nber_data_monthly_{t-5} + b_{13,7}^{5} unemployment_data_monthly_{t-5} + b_{13,8}^{5} unemployment_level_data_monthly_{t-5} + b_{13,9}^{5} all_employed_data_monthly_{t-5} + b_{13,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{13,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{13,12}^{5} m1_data_monthly_{t-5} + b_{13,13}^{5} m2_data_monthly_{t-5} + b_{13,14}^{5} m2_real_data_monthly_{t-5} + b_{13,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{13,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{13,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{13,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{13,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{13,20}^{5} job_openings_data_monthly_{t-5} + b_{13,21}^{5} job_hires_data_monthly_{t-5} + b_{13,22}^{5} job_separations_data_monthly_{t-5} + b_{13,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{13,24}^{5} gdp_data_quarterly_{t-5} + b_{13,25}^{5} gdp_real_data_quarterly_{t-5} + b_{13,26}^{5} cpi_inflation_data_annual_{t-5} + b_{13,27}^{5} median_income_real_data_annual_{t-5} + b_{13,28}^{5} median_income_data_annual_{t-5} + b_{13,29}^{5} Close_{t-5} + b_{13,30}^{5} Volume_{t-5} + b_{13,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{13,2}^{6} cpi_data_monthly_{t-6} + b_{13,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{13,4}^{6} rec_sahm_data_monthly_{t-6} + b_{13,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{13,6}^{6} rec_nber_data_monthly_{t-6} + b_{13,7}^{6} unemployment_data_monthly_{t-6} + b_{13,8}^{6} unemployment_level_data_monthly_{t-6} + b_{13,9}^{6} all_employed_data_monthly_{t-6} + b_{13,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{13,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{13,12}^{6} m1_data_monthly_{t-6} + b_{13,13}^{6} m2_data_monthly_{t-6} + b_{13,14}^{6} m2_real_data_monthly_{t-6} + b_{13,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{13,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{13,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{13,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{13,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{13,20}^{6} job_openings_data_monthly_{t-6} + b_{13,21}^{6} job_hires_data_monthly_{t-6} + b_{13,22}^{6} job_separations_data_monthly_{t-6} + b_{13,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{13,24}^{6} gdp_data_quarterly_{t-6} + b_{13,25}^{6} gdp_real_data_quarterly_{t-6} + b_{13,26}^{6} cpi_inflation_data_annual_{t-6} + b_{13,27}^{6} median_income_real_data_annual_{t-6} + b_{13,28}^{6} median_income_data_annual_{t-6} + b_{13,29}^{6} Close_{t-6} + b_{13,30}^{6} Volume_{t-6} + c_{13} + e_{t}^{m2_data_monthly} \\
m2_real_data_monthly_{t} &= b_{14,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{14,2}^{1} cpi_data_monthly_{t-1} + b_{14,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{14,4}^{1} rec_sahm_data_monthly_{t-1} + b_{14,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{14,6}^{1} rec_nber_data_monthly_{t-1} + b_{14,7}^{1} unemployment_data_monthly_{t-1} + b_{14,8}^{1} unemployment_level_data_monthly_{t-1} + b_{14,9}^{1} all_employed_data_monthly_{t-1} + b_{14,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{14,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{14,12}^{1} m1_data_monthly_{t-1} + b_{14,13}^{1} m2_data_monthly_{t-1} + b_{14,14}^{1} m2_real_data_monthly_{t-1} + b_{14,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{14,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{14,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{14,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{14,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{14,20}^{1} job_openings_data_monthly_{t-1} + b_{14,21}^{1} job_hires_data_monthly_{t-1} + b_{14,22}^{1} job_separations_data_monthly_{t-1} + b_{14,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{14,24}^{1} gdp_data_quarterly_{t-1} + b_{14,25}^{1} gdp_real_data_quarterly_{t-1} + b_{14,26}^{1} cpi_inflation_data_annual_{t-1} + b_{14,27}^{1} median_income_real_data_annual_{t-1} + b_{14,28}^{1} median_income_data_annual_{t-1} + b_{14,29}^{1} Close_{t-1} + b_{14,30}^{1} Volume_{t-1} + b_{14,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{14,2}^{2} cpi_data_monthly_{t-2} + b_{14,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{14,4}^{2} rec_sahm_data_monthly_{t-2} + b_{14,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{14,6}^{2} rec_nber_data_monthly_{t-2} + b_{14,7}^{2} unemployment_data_monthly_{t-2} + b_{14,8}^{2} unemployment_level_data_monthly_{t-2} + b_{14,9}^{2} all_employed_data_monthly_{t-2} + b_{14,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{14,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{14,12}^{2} m1_data_monthly_{t-2} + b_{14,13}^{2} m2_data_monthly_{t-2} + b_{14,14}^{2} m2_real_data_monthly_{t-2} + b_{14,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{14,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{14,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{14,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{14,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{14,20}^{2} job_openings_data_monthly_{t-2} + b_{14,21}^{2} job_hires_data_monthly_{t-2} + b_{14,22}^{2} job_separations_data_monthly_{t-2} + b_{14,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{14,24}^{2} gdp_data_quarterly_{t-2} + b_{14,25}^{2} gdp_real_data_quarterly_{t-2} + b_{14,26}^{2} cpi_inflation_data_annual_{t-2} + b_{14,27}^{2} median_income_real_data_annual_{t-2} + b_{14,28}^{2} median_income_data_annual_{t-2} + b_{14,29}^{2} Close_{t-2} + b_{14,30}^{2} Volume_{t-2} + b_{14,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{14,2}^{3} cpi_data_monthly_{t-3} + b_{14,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{14,4}^{3} rec_sahm_data_monthly_{t-3} + b_{14,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{14,6}^{3} rec_nber_data_monthly_{t-3} + b_{14,7}^{3} unemployment_data_monthly_{t-3} + b_{14,8}^{3} unemployment_level_data_monthly_{t-3} + b_{14,9}^{3} all_employed_data_monthly_{t-3} + b_{14,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{14,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{14,12}^{3} m1_data_monthly_{t-3} + b_{14,13}^{3} m2_data_monthly_{t-3} + b_{14,14}^{3} m2_real_data_monthly_{t-3} + b_{14,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{14,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{14,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{14,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{14,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{14,20}^{3} job_openings_data_monthly_{t-3} + b_{14,21}^{3} job_hires_data_monthly_{t-3} + b_{14,22}^{3} job_separations_data_monthly_{t-3} + b_{14,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{14,24}^{3} gdp_data_quarterly_{t-3} + b_{14,25}^{3} gdp_real_data_quarterly_{t-3} + b_{14,26}^{3} cpi_inflation_data_annual_{t-3} + b_{14,27}^{3} median_income_real_data_annual_{t-3} + b_{14,28}^{3} median_income_data_annual_{t-3} + b_{14,29}^{3} Close_{t-3} + b_{14,30}^{3} Volume_{t-3} + b_{14,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{14,2}^{4} cpi_data_monthly_{t-4} + b_{14,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{14,4}^{4} rec_sahm_data_monthly_{t-4} + b_{14,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{14,6}^{4} rec_nber_data_monthly_{t-4} + b_{14,7}^{4} unemployment_data_monthly_{t-4} + b_{14,8}^{4} unemployment_level_data_monthly_{t-4} + b_{14,9}^{4} all_employed_data_monthly_{t-4} + b_{14,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{14,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{14,12}^{4} m1_data_monthly_{t-4} + b_{14,13}^{4} m2_data_monthly_{t-4} + b_{14,14}^{4} m2_real_data_monthly_{t-4} + b_{14,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{14,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{14,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{14,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{14,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{14,20}^{4} job_openings_data_monthly_{t-4} + b_{14,21}^{4} job_hires_data_monthly_{t-4} + b_{14,22}^{4} job_separations_data_monthly_{t-4} + b_{14,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{14,24}^{4} gdp_data_quarterly_{t-4} + b_{14,25}^{4} gdp_real_data_quarterly_{t-4} + b_{14,26}^{4} cpi_inflation_data_annual_{t-4} + b_{14,27}^{4} median_income_real_data_annual_{t-4} + b_{14,28}^{4} median_income_data_annual_{t-4} + b_{14,29}^{4} Close_{t-4} + b_{14,30}^{4} Volume_{t-4} + b_{14,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{14,2}^{5} cpi_data_monthly_{t-5} + b_{14,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{14,4}^{5} rec_sahm_data_monthly_{t-5} + b_{14,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{14,6}^{5} rec_nber_data_monthly_{t-5} + b_{14,7}^{5} unemployment_data_monthly_{t-5} + b_{14,8}^{5} unemployment_level_data_monthly_{t-5} + b_{14,9}^{5} all_employed_data_monthly_{t-5} + b_{14,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{14,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{14,12}^{5} m1_data_monthly_{t-5} + b_{14,13}^{5} m2_data_monthly_{t-5} + b_{14,14}^{5} m2_real_data_monthly_{t-5} + b_{14,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{14,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{14,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{14,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{14,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{14,20}^{5} job_openings_data_monthly_{t-5} + b_{14,21}^{5} job_hires_data_monthly_{t-5} + b_{14,22}^{5} job_separations_data_monthly_{t-5} + b_{14,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{14,24}^{5} gdp_data_quarterly_{t-5} + b_{14,25}^{5} gdp_real_data_quarterly_{t-5} + b_{14,26}^{5} cpi_inflation_data_annual_{t-5} + b_{14,27}^{5} median_income_real_data_annual_{t-5} + b_{14,28}^{5} median_income_data_annual_{t-5} + b_{14,29}^{5} Close_{t-5} + b_{14,30}^{5} Volume_{t-5} + b_{14,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{14,2}^{6} cpi_data_monthly_{t-6} + b_{14,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{14,4}^{6} rec_sahm_data_monthly_{t-6} + b_{14,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{14,6}^{6} rec_nber_data_monthly_{t-6} + b_{14,7}^{6} unemployment_data_monthly_{t-6} + b_{14,8}^{6} unemployment_level_data_monthly_{t-6} + b_{14,9}^{6} all_employed_data_monthly_{t-6} + b_{14,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{14,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{14,12}^{6} m1_data_monthly_{t-6} + b_{14,13}^{6} m2_data_monthly_{t-6} + b_{14,14}^{6} m2_real_data_monthly_{t-6} + b_{14,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{14,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{14,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{14,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{14,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{14,20}^{6} job_openings_data_monthly_{t-6} + b_{14,21}^{6} job_hires_data_monthly_{t-6} + b_{14,22}^{6} job_separations_data_monthly_{t-6} + b_{14,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{14,24}^{6} gdp_data_quarterly_{t-6} + b_{14,25}^{6} gdp_real_data_quarterly_{t-6} + b_{14,26}^{6} cpi_inflation_data_annual_{t-6} + b_{14,27}^{6} median_income_real_data_annual_{t-6} + b_{14,28}^{6} median_income_data_annual_{t-6} + b_{14,29}^{6} Close_{t-6} + b_{14,30}^{6} Volume_{t-6} + c_{14} + e_{t}^{m2_real_data_monthly} \\
interest_rate_fedfunds_data_monthly_{t} &= b_{15,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{15,2}^{1} cpi_data_monthly_{t-1} + b_{15,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{15,4}^{1} rec_sahm_data_monthly_{t-1} + b_{15,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{15,6}^{1} rec_nber_data_monthly_{t-1} + b_{15,7}^{1} unemployment_data_monthly_{t-1} + b_{15,8}^{1} unemployment_level_data_monthly_{t-1} + b_{15,9}^{1} all_employed_data_monthly_{t-1} + b_{15,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{15,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{15,12}^{1} m1_data_monthly_{t-1} + b_{15,13}^{1} m2_data_monthly_{t-1} + b_{15,14}^{1} m2_real_data_monthly_{t-1} + b_{15,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{15,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{15,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{15,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{15,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{15,20}^{1} job_openings_data_monthly_{t-1} + b_{15,21}^{1} job_hires_data_monthly_{t-1} + b_{15,22}^{1} job_separations_data_monthly_{t-1} + b_{15,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{15,24}^{1} gdp_data_quarterly_{t-1} + b_{15,25}^{1} gdp_real_data_quarterly_{t-1} + b_{15,26}^{1} cpi_inflation_data_annual_{t-1} + b_{15,27}^{1} median_income_real_data_annual_{t-1} + b_{15,28}^{1} median_income_data_annual_{t-1} + b_{15,29}^{1} Close_{t-1} + b_{15,30}^{1} Volume_{t-1} + b_{15,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{15,2}^{2} cpi_data_monthly_{t-2} + b_{15,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{15,4}^{2} rec_sahm_data_monthly_{t-2} + b_{15,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{15,6}^{2} rec_nber_data_monthly_{t-2} + b_{15,7}^{2} unemployment_data_monthly_{t-2} + b_{15,8}^{2} unemployment_level_data_monthly_{t-2} + b_{15,9}^{2} all_employed_data_monthly_{t-2} + b_{15,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{15,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{15,12}^{2} m1_data_monthly_{t-2} + b_{15,13}^{2} m2_data_monthly_{t-2} + b_{15,14}^{2} m2_real_data_monthly_{t-2} + b_{15,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{15,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{15,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{15,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{15,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{15,20}^{2} job_openings_data_monthly_{t-2} + b_{15,21}^{2} job_hires_data_monthly_{t-2} + b_{15,22}^{2} job_separations_data_monthly_{t-2} + b_{15,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{15,24}^{2} gdp_data_quarterly_{t-2} + b_{15,25}^{2} gdp_real_data_quarterly_{t-2} + b_{15,26}^{2} cpi_inflation_data_annual_{t-2} + b_{15,27}^{2} median_income_real_data_annual_{t-2} + b_{15,28}^{2} median_income_data_annual_{t-2} + b_{15,29}^{2} Close_{t-2} + b_{15,30}^{2} Volume_{t-2} + b_{15,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{15,2}^{3} cpi_data_monthly_{t-3} + b_{15,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{15,4}^{3} rec_sahm_data_monthly_{t-3} + b_{15,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{15,6}^{3} rec_nber_data_monthly_{t-3} + b_{15,7}^{3} unemployment_data_monthly_{t-3} + b_{15,8}^{3} unemployment_level_data_monthly_{t-3} + b_{15,9}^{3} all_employed_data_monthly_{t-3} + b_{15,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{15,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{15,12}^{3} m1_data_monthly_{t-3} + b_{15,13}^{3} m2_data_monthly_{t-3} + b_{15,14}^{3} m2_real_data_monthly_{t-3} + b_{15,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{15,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{15,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{15,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{15,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{15,20}^{3} job_openings_data_monthly_{t-3} + b_{15,21}^{3} job_hires_data_monthly_{t-3} + b_{15,22}^{3} job_separations_data_monthly_{t-3} + b_{15,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{15,24}^{3} gdp_data_quarterly_{t-3} + b_{15,25}^{3} gdp_real_data_quarterly_{t-3} + b_{15,26}^{3} cpi_inflation_data_annual_{t-3} + b_{15,27}^{3} median_income_real_data_annual_{t-3} + b_{15,28}^{3} median_income_data_annual_{t-3} + b_{15,29}^{3} Close_{t-3} + b_{15,30}^{3} Volume_{t-3} + b_{15,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{15,2}^{4} cpi_data_monthly_{t-4} + b_{15,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{15,4}^{4} rec_sahm_data_monthly_{t-4} + b_{15,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{15,6}^{4} rec_nber_data_monthly_{t-4} + b_{15,7}^{4} unemployment_data_monthly_{t-4} + b_{15,8}^{4} unemployment_level_data_monthly_{t-4} + b_{15,9}^{4} all_employed_data_monthly_{t-4} + b_{15,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{15,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{15,12}^{4} m1_data_monthly_{t-4} + b_{15,13}^{4} m2_data_monthly_{t-4} + b_{15,14}^{4} m2_real_data_monthly_{t-4} + b_{15,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{15,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{15,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{15,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{15,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{15,20}^{4} job_openings_data_monthly_{t-4} + b_{15,21}^{4} job_hires_data_monthly_{t-4} + b_{15,22}^{4} job_separations_data_monthly_{t-4} + b_{15,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{15,24}^{4} gdp_data_quarterly_{t-4} + b_{15,25}^{4} gdp_real_data_quarterly_{t-4} + b_{15,26}^{4} cpi_inflation_data_annual_{t-4} + b_{15,27}^{4} median_income_real_data_annual_{t-4} + b_{15,28}^{4} median_income_data_annual_{t-4} + b_{15,29}^{4} Close_{t-4} + b_{15,30}^{4} Volume_{t-4} + b_{15,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{15,2}^{5} cpi_data_monthly_{t-5} + b_{15,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{15,4}^{5} rec_sahm_data_monthly_{t-5} + b_{15,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{15,6}^{5} rec_nber_data_monthly_{t-5} + b_{15,7}^{5} unemployment_data_monthly_{t-5} + b_{15,8}^{5} unemployment_level_data_monthly_{t-5} + b_{15,9}^{5} all_employed_data_monthly_{t-5} + b_{15,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{15,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{15,12}^{5} m1_data_monthly_{t-5} + b_{15,13}^{5} m2_data_monthly_{t-5} + b_{15,14}^{5} m2_real_data_monthly_{t-5} + b_{15,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{15,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{15,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{15,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{15,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{15,20}^{5} job_openings_data_monthly_{t-5} + b_{15,21}^{5} job_hires_data_monthly_{t-5} + b_{15,22}^{5} job_separations_data_monthly_{t-5} + b_{15,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{15,24}^{5} gdp_data_quarterly_{t-5} + b_{15,25}^{5} gdp_real_data_quarterly_{t-5} + b_{15,26}^{5} cpi_inflation_data_annual_{t-5} + b_{15,27}^{5} median_income_real_data_annual_{t-5} + b_{15,28}^{5} median_income_data_annual_{t-5} + b_{15,29}^{5} Close_{t-5} + b_{15,30}^{5} Volume_{t-5} + b_{15,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{15,2}^{6} cpi_data_monthly_{t-6} + b_{15,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{15,4}^{6} rec_sahm_data_monthly_{t-6} + b_{15,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{15,6}^{6} rec_nber_data_monthly_{t-6} + b_{15,7}^{6} unemployment_data_monthly_{t-6} + b_{15,8}^{6} unemployment_level_data_monthly_{t-6} + b_{15,9}^{6} all_employed_data_monthly_{t-6} + b_{15,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{15,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{15,12}^{6} m1_data_monthly_{t-6} + b_{15,13}^{6} m2_data_monthly_{t-6} + b_{15,14}^{6} m2_real_data_monthly_{t-6} + b_{15,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{15,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{15,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{15,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{15,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{15,20}^{6} job_openings_data_monthly_{t-6} + b_{15,21}^{6} job_hires_data_monthly_{t-6} + b_{15,22}^{6} job_separations_data_monthly_{t-6} + b_{15,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{15,24}^{6} gdp_data_quarterly_{t-6} + b_{15,25}^{6} gdp_real_data_quarterly_{t-6} + b_{15,26}^{6} cpi_inflation_data_annual_{t-6} + b_{15,27}^{6} median_income_real_data_annual_{t-6} + b_{15,28}^{6} median_income_data_annual_{t-6} + b_{15,29}^{6} Close_{t-6} + b_{15,30}^{6} Volume_{t-6} + c_{15} + e_{t}^{interest_rate_fedfunds_data_monthly} \\
consumer_sentiment_data_monthly_{t} &= b_{16,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{16,2}^{1} cpi_data_monthly_{t-1} + b_{16,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{16,4}^{1} rec_sahm_data_monthly_{t-1} + b_{16,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{16,6}^{1} rec_nber_data_monthly_{t-1} + b_{16,7}^{1} unemployment_data_monthly_{t-1} + b_{16,8}^{1} unemployment_level_data_monthly_{t-1} + b_{16,9}^{1} all_employed_data_monthly_{t-1} + b_{16,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{16,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{16,12}^{1} m1_data_monthly_{t-1} + b_{16,13}^{1} m2_data_monthly_{t-1} + b_{16,14}^{1} m2_real_data_monthly_{t-1} + b_{16,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{16,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{16,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{16,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{16,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{16,20}^{1} job_openings_data_monthly_{t-1} + b_{16,21}^{1} job_hires_data_monthly_{t-1} + b_{16,22}^{1} job_separations_data_monthly_{t-1} + b_{16,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{16,24}^{1} gdp_data_quarterly_{t-1} + b_{16,25}^{1} gdp_real_data_quarterly_{t-1} + b_{16,26}^{1} cpi_inflation_data_annual_{t-1} + b_{16,27}^{1} median_income_real_data_annual_{t-1} + b_{16,28}^{1} median_income_data_annual_{t-1} + b_{16,29}^{1} Close_{t-1} + b_{16,30}^{1} Volume_{t-1} + b_{16,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{16,2}^{2} cpi_data_monthly_{t-2} + b_{16,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{16,4}^{2} rec_sahm_data_monthly_{t-2} + b_{16,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{16,6}^{2} rec_nber_data_monthly_{t-2} + b_{16,7}^{2} unemployment_data_monthly_{t-2} + b_{16,8}^{2} unemployment_level_data_monthly_{t-2} + b_{16,9}^{2} all_employed_data_monthly_{t-2} + b_{16,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{16,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{16,12}^{2} m1_data_monthly_{t-2} + b_{16,13}^{2} m2_data_monthly_{t-2} + b_{16,14}^{2} m2_real_data_monthly_{t-2} + b_{16,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{16,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{16,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{16,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{16,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{16,20}^{2} job_openings_data_monthly_{t-2} + b_{16,21}^{2} job_hires_data_monthly_{t-2} + b_{16,22}^{2} job_separations_data_monthly_{t-2} + b_{16,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{16,24}^{2} gdp_data_quarterly_{t-2} + b_{16,25}^{2} gdp_real_data_quarterly_{t-2} + b_{16,26}^{2} cpi_inflation_data_annual_{t-2} + b_{16,27}^{2} median_income_real_data_annual_{t-2} + b_{16,28}^{2} median_income_data_annual_{t-2} + b_{16,29}^{2} Close_{t-2} + b_{16,30}^{2} Volume_{t-2} + b_{16,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{16,2}^{3} cpi_data_monthly_{t-3} + b_{16,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{16,4}^{3} rec_sahm_data_monthly_{t-3} + b_{16,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{16,6}^{3} rec_nber_data_monthly_{t-3} + b_{16,7}^{3} unemployment_data_monthly_{t-3} + b_{16,8}^{3} unemployment_level_data_monthly_{t-3} + b_{16,9}^{3} all_employed_data_monthly_{t-3} + b_{16,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{16,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{16,12}^{3} m1_data_monthly_{t-3} + b_{16,13}^{3} m2_data_monthly_{t-3} + b_{16,14}^{3} m2_real_data_monthly_{t-3} + b_{16,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{16,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{16,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{16,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{16,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{16,20}^{3} job_openings_data_monthly_{t-3} + b_{16,21}^{3} job_hires_data_monthly_{t-3} + b_{16,22}^{3} job_separations_data_monthly_{t-3} + b_{16,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{16,24}^{3} gdp_data_quarterly_{t-3} + b_{16,25}^{3} gdp_real_data_quarterly_{t-3} + b_{16,26}^{3} cpi_inflation_data_annual_{t-3} + b_{16,27}^{3} median_income_real_data_annual_{t-3} + b_{16,28}^{3} median_income_data_annual_{t-3} + b_{16,29}^{3} Close_{t-3} + b_{16,30}^{3} Volume_{t-3} + b_{16,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{16,2}^{4} cpi_data_monthly_{t-4} + b_{16,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{16,4}^{4} rec_sahm_data_monthly_{t-4} + b_{16,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{16,6}^{4} rec_nber_data_monthly_{t-4} + b_{16,7}^{4} unemployment_data_monthly_{t-4} + b_{16,8}^{4} unemployment_level_data_monthly_{t-4} + b_{16,9}^{4} all_employed_data_monthly_{t-4} + b_{16,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{16,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{16,12}^{4} m1_data_monthly_{t-4} + b_{16,13}^{4} m2_data_monthly_{t-4} + b_{16,14}^{4} m2_real_data_monthly_{t-4} + b_{16,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{16,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{16,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{16,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{16,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{16,20}^{4} job_openings_data_monthly_{t-4} + b_{16,21}^{4} job_hires_data_monthly_{t-4} + b_{16,22}^{4} job_separations_data_monthly_{t-4} + b_{16,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{16,24}^{4} gdp_data_quarterly_{t-4} + b_{16,25}^{4} gdp_real_data_quarterly_{t-4} + b_{16,26}^{4} cpi_inflation_data_annual_{t-4} + b_{16,27}^{4} median_income_real_data_annual_{t-4} + b_{16,28}^{4} median_income_data_annual_{t-4} + b_{16,29}^{4} Close_{t-4} + b_{16,30}^{4} Volume_{t-4} + b_{16,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{16,2}^{5} cpi_data_monthly_{t-5} + b_{16,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{16,4}^{5} rec_sahm_data_monthly_{t-5} + b_{16,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{16,6}^{5} rec_nber_data_monthly_{t-5} + b_{16,7}^{5} unemployment_data_monthly_{t-5} + b_{16,8}^{5} unemployment_level_data_monthly_{t-5} + b_{16,9}^{5} all_employed_data_monthly_{t-5} + b_{16,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{16,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{16,12}^{5} m1_data_monthly_{t-5} + b_{16,13}^{5} m2_data_monthly_{t-5} + b_{16,14}^{5} m2_real_data_monthly_{t-5} + b_{16,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{16,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{16,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{16,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{16,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{16,20}^{5} job_openings_data_monthly_{t-5} + b_{16,21}^{5} job_hires_data_monthly_{t-5} + b_{16,22}^{5} job_separations_data_monthly_{t-5} + b_{16,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{16,24}^{5} gdp_data_quarterly_{t-5} + b_{16,25}^{5} gdp_real_data_quarterly_{t-5} + b_{16,26}^{5} cpi_inflation_data_annual_{t-5} + b_{16,27}^{5} median_income_real_data_annual_{t-5} + b_{16,28}^{5} median_income_data_annual_{t-5} + b_{16,29}^{5} Close_{t-5} + b_{16,30}^{5} Volume_{t-5} + b_{16,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{16,2}^{6} cpi_data_monthly_{t-6} + b_{16,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{16,4}^{6} rec_sahm_data_monthly_{t-6} + b_{16,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{16,6}^{6} rec_nber_data_monthly_{t-6} + b_{16,7}^{6} unemployment_data_monthly_{t-6} + b_{16,8}^{6} unemployment_level_data_monthly_{t-6} + b_{16,9}^{6} all_employed_data_monthly_{t-6} + b_{16,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{16,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{16,12}^{6} m1_data_monthly_{t-6} + b_{16,13}^{6} m2_data_monthly_{t-6} + b_{16,14}^{6} m2_real_data_monthly_{t-6} + b_{16,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{16,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{16,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{16,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{16,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{16,20}^{6} job_openings_data_monthly_{t-6} + b_{16,21}^{6} job_hires_data_monthly_{t-6} + b_{16,22}^{6} job_separations_data_monthly_{t-6} + b_{16,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{16,24}^{6} gdp_data_quarterly_{t-6} + b_{16,25}^{6} gdp_real_data_quarterly_{t-6} + b_{16,26}^{6} cpi_inflation_data_annual_{t-6} + b_{16,27}^{6} median_income_real_data_annual_{t-6} + b_{16,28}^{6} median_income_data_annual_{t-6} + b_{16,29}^{6} Close_{t-6} + b_{16,30}^{6} Volume_{t-6} + c_{16} + e_{t}^{consumer_sentiment_data_monthly} \\
consumer_sentiment_inflation_data_monthly_{t} &= b_{17,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{17,2}^{1} cpi_data_monthly_{t-1} + b_{17,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{17,4}^{1} rec_sahm_data_monthly_{t-1} + b_{17,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{17,6}^{1} rec_nber_data_monthly_{t-1} + b_{17,7}^{1} unemployment_data_monthly_{t-1} + b_{17,8}^{1} unemployment_level_data_monthly_{t-1} + b_{17,9}^{1} all_employed_data_monthly_{t-1} + b_{17,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{17,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{17,12}^{1} m1_data_monthly_{t-1} + b_{17,13}^{1} m2_data_monthly_{t-1} + b_{17,14}^{1} m2_real_data_monthly_{t-1} + b_{17,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{17,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{17,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{17,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{17,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{17,20}^{1} job_openings_data_monthly_{t-1} + b_{17,21}^{1} job_hires_data_monthly_{t-1} + b_{17,22}^{1} job_separations_data_monthly_{t-1} + b_{17,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{17,24}^{1} gdp_data_quarterly_{t-1} + b_{17,25}^{1} gdp_real_data_quarterly_{t-1} + b_{17,26}^{1} cpi_inflation_data_annual_{t-1} + b_{17,27}^{1} median_income_real_data_annual_{t-1} + b_{17,28}^{1} median_income_data_annual_{t-1} + b_{17,29}^{1} Close_{t-1} + b_{17,30}^{1} Volume_{t-1} + b_{17,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{17,2}^{2} cpi_data_monthly_{t-2} + b_{17,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{17,4}^{2} rec_sahm_data_monthly_{t-2} + b_{17,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{17,6}^{2} rec_nber_data_monthly_{t-2} + b_{17,7}^{2} unemployment_data_monthly_{t-2} + b_{17,8}^{2} unemployment_level_data_monthly_{t-2} + b_{17,9}^{2} all_employed_data_monthly_{t-2} + b_{17,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{17,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{17,12}^{2} m1_data_monthly_{t-2} + b_{17,13}^{2} m2_data_monthly_{t-2} + b_{17,14}^{2} m2_real_data_monthly_{t-2} + b_{17,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{17,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{17,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{17,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{17,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{17,20}^{2} job_openings_data_monthly_{t-2} + b_{17,21}^{2} job_hires_data_monthly_{t-2} + b_{17,22}^{2} job_separations_data_monthly_{t-2} + b_{17,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{17,24}^{2} gdp_data_quarterly_{t-2} + b_{17,25}^{2} gdp_real_data_quarterly_{t-2} + b_{17,26}^{2} cpi_inflation_data_annual_{t-2} + b_{17,27}^{2} median_income_real_data_annual_{t-2} + b_{17,28}^{2} median_income_data_annual_{t-2} + b_{17,29}^{2} Close_{t-2} + b_{17,30}^{2} Volume_{t-2} + b_{17,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{17,2}^{3} cpi_data_monthly_{t-3} + b_{17,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{17,4}^{3} rec_sahm_data_monthly_{t-3} + b_{17,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{17,6}^{3} rec_nber_data_monthly_{t-3} + b_{17,7}^{3} unemployment_data_monthly_{t-3} + b_{17,8}^{3} unemployment_level_data_monthly_{t-3} + b_{17,9}^{3} all_employed_data_monthly_{t-3} + b_{17,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{17,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{17,12}^{3} m1_data_monthly_{t-3} + b_{17,13}^{3} m2_data_monthly_{t-3} + b_{17,14}^{3} m2_real_data_monthly_{t-3} + b_{17,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{17,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{17,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{17,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{17,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{17,20}^{3} job_openings_data_monthly_{t-3} + b_{17,21}^{3} job_hires_data_monthly_{t-3} + b_{17,22}^{3} job_separations_data_monthly_{t-3} + b_{17,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{17,24}^{3} gdp_data_quarterly_{t-3} + b_{17,25}^{3} gdp_real_data_quarterly_{t-3} + b_{17,26}^{3} cpi_inflation_data_annual_{t-3} + b_{17,27}^{3} median_income_real_data_annual_{t-3} + b_{17,28}^{3} median_income_data_annual_{t-3} + b_{17,29}^{3} Close_{t-3} + b_{17,30}^{3} Volume_{t-3} + b_{17,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{17,2}^{4} cpi_data_monthly_{t-4} + b_{17,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{17,4}^{4} rec_sahm_data_monthly_{t-4} + b_{17,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{17,6}^{4} rec_nber_data_monthly_{t-4} + b_{17,7}^{4} unemployment_data_monthly_{t-4} + b_{17,8}^{4} unemployment_level_data_monthly_{t-4} + b_{17,9}^{4} all_employed_data_monthly_{t-4} + b_{17,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{17,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{17,12}^{4} m1_data_monthly_{t-4} + b_{17,13}^{4} m2_data_monthly_{t-4} + b_{17,14}^{4} m2_real_data_monthly_{t-4} + b_{17,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{17,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{17,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{17,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{17,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{17,20}^{4} job_openings_data_monthly_{t-4} + b_{17,21}^{4} job_hires_data_monthly_{t-4} + b_{17,22}^{4} job_separations_data_monthly_{t-4} + b_{17,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{17,24}^{4} gdp_data_quarterly_{t-4} + b_{17,25}^{4} gdp_real_data_quarterly_{t-4} + b_{17,26}^{4} cpi_inflation_data_annual_{t-4} + b_{17,27}^{4} median_income_real_data_annual_{t-4} + b_{17,28}^{4} median_income_data_annual_{t-4} + b_{17,29}^{4} Close_{t-4} + b_{17,30}^{4} Volume_{t-4} + b_{17,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{17,2}^{5} cpi_data_monthly_{t-5} + b_{17,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{17,4}^{5} rec_sahm_data_monthly_{t-5} + b_{17,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{17,6}^{5} rec_nber_data_monthly_{t-5} + b_{17,7}^{5} unemployment_data_monthly_{t-5} + b_{17,8}^{5} unemployment_level_data_monthly_{t-5} + b_{17,9}^{5} all_employed_data_monthly_{t-5} + b_{17,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{17,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{17,12}^{5} m1_data_monthly_{t-5} + b_{17,13}^{5} m2_data_monthly_{t-5} + b_{17,14}^{5} m2_real_data_monthly_{t-5} + b_{17,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{17,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{17,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{17,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{17,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{17,20}^{5} job_openings_data_monthly_{t-5} + b_{17,21}^{5} job_hires_data_monthly_{t-5} + b_{17,22}^{5} job_separations_data_monthly_{t-5} + b_{17,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{17,24}^{5} gdp_data_quarterly_{t-5} + b_{17,25}^{5} gdp_real_data_quarterly_{t-5} + b_{17,26}^{5} cpi_inflation_data_annual_{t-5} + b_{17,27}^{5} median_income_real_data_annual_{t-5} + b_{17,28}^{5} median_income_data_annual_{t-5} + b_{17,29}^{5} Close_{t-5} + b_{17,30}^{5} Volume_{t-5} + b_{17,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{17,2}^{6} cpi_data_monthly_{t-6} + b_{17,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{17,4}^{6} rec_sahm_data_monthly_{t-6} + b_{17,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{17,6}^{6} rec_nber_data_monthly_{t-6} + b_{17,7}^{6} unemployment_data_monthly_{t-6} + b_{17,8}^{6} unemployment_level_data_monthly_{t-6} + b_{17,9}^{6} all_employed_data_monthly_{t-6} + b_{17,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{17,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{17,12}^{6} m1_data_monthly_{t-6} + b_{17,13}^{6} m2_data_monthly_{t-6} + b_{17,14}^{6} m2_real_data_monthly_{t-6} + b_{17,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{17,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{17,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{17,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{17,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{17,20}^{6} job_openings_data_monthly_{t-6} + b_{17,21}^{6} job_hires_data_monthly_{t-6} + b_{17,22}^{6} job_separations_data_monthly_{t-6} + b_{17,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{17,24}^{6} gdp_data_quarterly_{t-6} + b_{17,25}^{6} gdp_real_data_quarterly_{t-6} + b_{17,26}^{6} cpi_inflation_data_annual_{t-6} + b_{17,27}^{6} median_income_real_data_annual_{t-6} + b_{17,28}^{6} median_income_data_annual_{t-6} + b_{17,29}^{6} Close_{t-6} + b_{17,30}^{6} Volume_{t-6} + c_{17} + e_{t}^{consumer_sentiment_inflation_data_monthly} \\
consumer_sentiment_composite_data_monthly_{t} &= b_{18,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{18,2}^{1} cpi_data_monthly_{t-1} + b_{18,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{18,4}^{1} rec_sahm_data_monthly_{t-1} + b_{18,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{18,6}^{1} rec_nber_data_monthly_{t-1} + b_{18,7}^{1} unemployment_data_monthly_{t-1} + b_{18,8}^{1} unemployment_level_data_monthly_{t-1} + b_{18,9}^{1} all_employed_data_monthly_{t-1} + b_{18,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{18,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{18,12}^{1} m1_data_monthly_{t-1} + b_{18,13}^{1} m2_data_monthly_{t-1} + b_{18,14}^{1} m2_real_data_monthly_{t-1} + b_{18,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{18,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{18,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{18,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{18,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{18,20}^{1} job_openings_data_monthly_{t-1} + b_{18,21}^{1} job_hires_data_monthly_{t-1} + b_{18,22}^{1} job_separations_data_monthly_{t-1} + b_{18,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{18,24}^{1} gdp_data_quarterly_{t-1} + b_{18,25}^{1} gdp_real_data_quarterly_{t-1} + b_{18,26}^{1} cpi_inflation_data_annual_{t-1} + b_{18,27}^{1} median_income_real_data_annual_{t-1} + b_{18,28}^{1} median_income_data_annual_{t-1} + b_{18,29}^{1} Close_{t-1} + b_{18,30}^{1} Volume_{t-1} + b_{18,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{18,2}^{2} cpi_data_monthly_{t-2} + b_{18,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{18,4}^{2} rec_sahm_data_monthly_{t-2} + b_{18,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{18,6}^{2} rec_nber_data_monthly_{t-2} + b_{18,7}^{2} unemployment_data_monthly_{t-2} + b_{18,8}^{2} unemployment_level_data_monthly_{t-2} + b_{18,9}^{2} all_employed_data_monthly_{t-2} + b_{18,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{18,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{18,12}^{2} m1_data_monthly_{t-2} + b_{18,13}^{2} m2_data_monthly_{t-2} + b_{18,14}^{2} m2_real_data_monthly_{t-2} + b_{18,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{18,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{18,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{18,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{18,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{18,20}^{2} job_openings_data_monthly_{t-2} + b_{18,21}^{2} job_hires_data_monthly_{t-2} + b_{18,22}^{2} job_separations_data_monthly_{t-2} + b_{18,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{18,24}^{2} gdp_data_quarterly_{t-2} + b_{18,25}^{2} gdp_real_data_quarterly_{t-2} + b_{18,26}^{2} cpi_inflation_data_annual_{t-2} + b_{18,27}^{2} median_income_real_data_annual_{t-2} + b_{18,28}^{2} median_income_data_annual_{t-2} + b_{18,29}^{2} Close_{t-2} + b_{18,30}^{2} Volume_{t-2} + b_{18,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{18,2}^{3} cpi_data_monthly_{t-3} + b_{18,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{18,4}^{3} rec_sahm_data_monthly_{t-3} + b_{18,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{18,6}^{3} rec_nber_data_monthly_{t-3} + b_{18,7}^{3} unemployment_data_monthly_{t-3} + b_{18,8}^{3} unemployment_level_data_monthly_{t-3} + b_{18,9}^{3} all_employed_data_monthly_{t-3} + b_{18,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{18,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{18,12}^{3} m1_data_monthly_{t-3} + b_{18,13}^{3} m2_data_monthly_{t-3} + b_{18,14}^{3} m2_real_data_monthly_{t-3} + b_{18,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{18,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{18,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{18,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{18,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{18,20}^{3} job_openings_data_monthly_{t-3} + b_{18,21}^{3} job_hires_data_monthly_{t-3} + b_{18,22}^{3} job_separations_data_monthly_{t-3} + b_{18,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{18,24}^{3} gdp_data_quarterly_{t-3} + b_{18,25}^{3} gdp_real_data_quarterly_{t-3} + b_{18,26}^{3} cpi_inflation_data_annual_{t-3} + b_{18,27}^{3} median_income_real_data_annual_{t-3} + b_{18,28}^{3} median_income_data_annual_{t-3} + b_{18,29}^{3} Close_{t-3} + b_{18,30}^{3} Volume_{t-3} + b_{18,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{18,2}^{4} cpi_data_monthly_{t-4} + b_{18,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{18,4}^{4} rec_sahm_data_monthly_{t-4} + b_{18,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{18,6}^{4} rec_nber_data_monthly_{t-4} + b_{18,7}^{4} unemployment_data_monthly_{t-4} + b_{18,8}^{4} unemployment_level_data_monthly_{t-4} + b_{18,9}^{4} all_employed_data_monthly_{t-4} + b_{18,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{18,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{18,12}^{4} m1_data_monthly_{t-4} + b_{18,13}^{4} m2_data_monthly_{t-4} + b_{18,14}^{4} m2_real_data_monthly_{t-4} + b_{18,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{18,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{18,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{18,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{18,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{18,20}^{4} job_openings_data_monthly_{t-4} + b_{18,21}^{4} job_hires_data_monthly_{t-4} + b_{18,22}^{4} job_separations_data_monthly_{t-4} + b_{18,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{18,24}^{4} gdp_data_quarterly_{t-4} + b_{18,25}^{4} gdp_real_data_quarterly_{t-4} + b_{18,26}^{4} cpi_inflation_data_annual_{t-4} + b_{18,27}^{4} median_income_real_data_annual_{t-4} + b_{18,28}^{4} median_income_data_annual_{t-4} + b_{18,29}^{4} Close_{t-4} + b_{18,30}^{4} Volume_{t-4} + b_{18,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{18,2}^{5} cpi_data_monthly_{t-5} + b_{18,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{18,4}^{5} rec_sahm_data_monthly_{t-5} + b_{18,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{18,6}^{5} rec_nber_data_monthly_{t-5} + b_{18,7}^{5} unemployment_data_monthly_{t-5} + b_{18,8}^{5} unemployment_level_data_monthly_{t-5} + b_{18,9}^{5} all_employed_data_monthly_{t-5} + b_{18,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{18,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{18,12}^{5} m1_data_monthly_{t-5} + b_{18,13}^{5} m2_data_monthly_{t-5} + b_{18,14}^{5} m2_real_data_monthly_{t-5} + b_{18,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{18,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{18,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{18,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{18,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{18,20}^{5} job_openings_data_monthly_{t-5} + b_{18,21}^{5} job_hires_data_monthly_{t-5} + b_{18,22}^{5} job_separations_data_monthly_{t-5} + b_{18,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{18,24}^{5} gdp_data_quarterly_{t-5} + b_{18,25}^{5} gdp_real_data_quarterly_{t-5} + b_{18,26}^{5} cpi_inflation_data_annual_{t-5} + b_{18,27}^{5} median_income_real_data_annual_{t-5} + b_{18,28}^{5} median_income_data_annual_{t-5} + b_{18,29}^{5} Close_{t-5} + b_{18,30}^{5} Volume_{t-5} + b_{18,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{18,2}^{6} cpi_data_monthly_{t-6} + b_{18,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{18,4}^{6} rec_sahm_data_monthly_{t-6} + b_{18,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{18,6}^{6} rec_nber_data_monthly_{t-6} + b_{18,7}^{6} unemployment_data_monthly_{t-6} + b_{18,8}^{6} unemployment_level_data_monthly_{t-6} + b_{18,9}^{6} all_employed_data_monthly_{t-6} + b_{18,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{18,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{18,12}^{6} m1_data_monthly_{t-6} + b_{18,13}^{6} m2_data_monthly_{t-6} + b_{18,14}^{6} m2_real_data_monthly_{t-6} + b_{18,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{18,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{18,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{18,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{18,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{18,20}^{6} job_openings_data_monthly_{t-6} + b_{18,21}^{6} job_hires_data_monthly_{t-6} + b_{18,22}^{6} job_separations_data_monthly_{t-6} + b_{18,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{18,24}^{6} gdp_data_quarterly_{t-6} + b_{18,25}^{6} gdp_real_data_quarterly_{t-6} + b_{18,26}^{6} cpi_inflation_data_annual_{t-6} + b_{18,27}^{6} median_income_real_data_annual_{t-6} + b_{18,28}^{6} median_income_data_annual_{t-6} + b_{18,29}^{6} Close_{t-6} + b_{18,30}^{6} Volume_{t-6} + c_{18} + e_{t}^{consumer_sentiment_composite_data_monthly} \\
consumer_sentiment_composite_amplitude_data_monthly_{t} &= b_{19,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{19,2}^{1} cpi_data_monthly_{t-1} + b_{19,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{19,4}^{1} rec_sahm_data_monthly_{t-1} + b_{19,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{19,6}^{1} rec_nber_data_monthly_{t-1} + b_{19,7}^{1} unemployment_data_monthly_{t-1} + b_{19,8}^{1} unemployment_level_data_monthly_{t-1} + b_{19,9}^{1} all_employed_data_monthly_{t-1} + b_{19,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{19,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{19,12}^{1} m1_data_monthly_{t-1} + b_{19,13}^{1} m2_data_monthly_{t-1} + b_{19,14}^{1} m2_real_data_monthly_{t-1} + b_{19,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{19,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{19,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{19,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{19,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{19,20}^{1} job_openings_data_monthly_{t-1} + b_{19,21}^{1} job_hires_data_monthly_{t-1} + b_{19,22}^{1} job_separations_data_monthly_{t-1} + b_{19,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{19,24}^{1} gdp_data_quarterly_{t-1} + b_{19,25}^{1} gdp_real_data_quarterly_{t-1} + b_{19,26}^{1} cpi_inflation_data_annual_{t-1} + b_{19,27}^{1} median_income_real_data_annual_{t-1} + b_{19,28}^{1} median_income_data_annual_{t-1} + b_{19,29}^{1} Close_{t-1} + b_{19,30}^{1} Volume_{t-1} + b_{19,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{19,2}^{2} cpi_data_monthly_{t-2} + b_{19,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{19,4}^{2} rec_sahm_data_monthly_{t-2} + b_{19,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{19,6}^{2} rec_nber_data_monthly_{t-2} + b_{19,7}^{2} unemployment_data_monthly_{t-2} + b_{19,8}^{2} unemployment_level_data_monthly_{t-2} + b_{19,9}^{2} all_employed_data_monthly_{t-2} + b_{19,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{19,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{19,12}^{2} m1_data_monthly_{t-2} + b_{19,13}^{2} m2_data_monthly_{t-2} + b_{19,14}^{2} m2_real_data_monthly_{t-2} + b_{19,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{19,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{19,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{19,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{19,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{19,20}^{2} job_openings_data_monthly_{t-2} + b_{19,21}^{2} job_hires_data_monthly_{t-2} + b_{19,22}^{2} job_separations_data_monthly_{t-2} + b_{19,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{19,24}^{2} gdp_data_quarterly_{t-2} + b_{19,25}^{2} gdp_real_data_quarterly_{t-2} + b_{19,26}^{2} cpi_inflation_data_annual_{t-2} + b_{19,27}^{2} median_income_real_data_annual_{t-2} + b_{19,28}^{2} median_income_data_annual_{t-2} + b_{19,29}^{2} Close_{t-2} + b_{19,30}^{2} Volume_{t-2} + b_{19,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{19,2}^{3} cpi_data_monthly_{t-3} + b_{19,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{19,4}^{3} rec_sahm_data_monthly_{t-3} + b_{19,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{19,6}^{3} rec_nber_data_monthly_{t-3} + b_{19,7}^{3} unemployment_data_monthly_{t-3} + b_{19,8}^{3} unemployment_level_data_monthly_{t-3} + b_{19,9}^{3} all_employed_data_monthly_{t-3} + b_{19,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{19,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{19,12}^{3} m1_data_monthly_{t-3} + b_{19,13}^{3} m2_data_monthly_{t-3} + b_{19,14}^{3} m2_real_data_monthly_{t-3} + b_{19,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{19,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{19,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{19,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{19,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{19,20}^{3} job_openings_data_monthly_{t-3} + b_{19,21}^{3} job_hires_data_monthly_{t-3} + b_{19,22}^{3} job_separations_data_monthly_{t-3} + b_{19,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{19,24}^{3} gdp_data_quarterly_{t-3} + b_{19,25}^{3} gdp_real_data_quarterly_{t-3} + b_{19,26}^{3} cpi_inflation_data_annual_{t-3} + b_{19,27}^{3} median_income_real_data_annual_{t-3} + b_{19,28}^{3} median_income_data_annual_{t-3} + b_{19,29}^{3} Close_{t-3} + b_{19,30}^{3} Volume_{t-3} + b_{19,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{19,2}^{4} cpi_data_monthly_{t-4} + b_{19,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{19,4}^{4} rec_sahm_data_monthly_{t-4} + b_{19,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{19,6}^{4} rec_nber_data_monthly_{t-4} + b_{19,7}^{4} unemployment_data_monthly_{t-4} + b_{19,8}^{4} unemployment_level_data_monthly_{t-4} + b_{19,9}^{4} all_employed_data_monthly_{t-4} + b_{19,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{19,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{19,12}^{4} m1_data_monthly_{t-4} + b_{19,13}^{4} m2_data_monthly_{t-4} + b_{19,14}^{4} m2_real_data_monthly_{t-4} + b_{19,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{19,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{19,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{19,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{19,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{19,20}^{4} job_openings_data_monthly_{t-4} + b_{19,21}^{4} job_hires_data_monthly_{t-4} + b_{19,22}^{4} job_separations_data_monthly_{t-4} + b_{19,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{19,24}^{4} gdp_data_quarterly_{t-4} + b_{19,25}^{4} gdp_real_data_quarterly_{t-4} + b_{19,26}^{4} cpi_inflation_data_annual_{t-4} + b_{19,27}^{4} median_income_real_data_annual_{t-4} + b_{19,28}^{4} median_income_data_annual_{t-4} + b_{19,29}^{4} Close_{t-4} + b_{19,30}^{4} Volume_{t-4} + b_{19,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{19,2}^{5} cpi_data_monthly_{t-5} + b_{19,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{19,4}^{5} rec_sahm_data_monthly_{t-5} + b_{19,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{19,6}^{5} rec_nber_data_monthly_{t-5} + b_{19,7}^{5} unemployment_data_monthly_{t-5} + b_{19,8}^{5} unemployment_level_data_monthly_{t-5} + b_{19,9}^{5} all_employed_data_monthly_{t-5} + b_{19,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{19,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{19,12}^{5} m1_data_monthly_{t-5} + b_{19,13}^{5} m2_data_monthly_{t-5} + b_{19,14}^{5} m2_real_data_monthly_{t-5} + b_{19,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{19,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{19,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{19,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{19,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{19,20}^{5} job_openings_data_monthly_{t-5} + b_{19,21}^{5} job_hires_data_monthly_{t-5} + b_{19,22}^{5} job_separations_data_monthly_{t-5} + b_{19,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{19,24}^{5} gdp_data_quarterly_{t-5} + b_{19,25}^{5} gdp_real_data_quarterly_{t-5} + b_{19,26}^{5} cpi_inflation_data_annual_{t-5} + b_{19,27}^{5} median_income_real_data_annual_{t-5} + b_{19,28}^{5} median_income_data_annual_{t-5} + b_{19,29}^{5} Close_{t-5} + b_{19,30}^{5} Volume_{t-5} + b_{19,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{19,2}^{6} cpi_data_monthly_{t-6} + b_{19,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{19,4}^{6} rec_sahm_data_monthly_{t-6} + b_{19,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{19,6}^{6} rec_nber_data_monthly_{t-6} + b_{19,7}^{6} unemployment_data_monthly_{t-6} + b_{19,8}^{6} unemployment_level_data_monthly_{t-6} + b_{19,9}^{6} all_employed_data_monthly_{t-6} + b_{19,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{19,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{19,12}^{6} m1_data_monthly_{t-6} + b_{19,13}^{6} m2_data_monthly_{t-6} + b_{19,14}^{6} m2_real_data_monthly_{t-6} + b_{19,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{19,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{19,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{19,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{19,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{19,20}^{6} job_openings_data_monthly_{t-6} + b_{19,21}^{6} job_hires_data_monthly_{t-6} + b_{19,22}^{6} job_separations_data_monthly_{t-6} + b_{19,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{19,24}^{6} gdp_data_quarterly_{t-6} + b_{19,25}^{6} gdp_real_data_quarterly_{t-6} + b_{19,26}^{6} cpi_inflation_data_annual_{t-6} + b_{19,27}^{6} median_income_real_data_annual_{t-6} + b_{19,28}^{6} median_income_data_annual_{t-6} + b_{19,29}^{6} Close_{t-6} + b_{19,30}^{6} Volume_{t-6} + c_{19} + e_{t}^{consumer_sentiment_composite_amplitude_data_monthly} \\
job_openings_data_monthly_{t} &= b_{20,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{20,2}^{1} cpi_data_monthly_{t-1} + b_{20,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{20,4}^{1} rec_sahm_data_monthly_{t-1} + b_{20,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{20,6}^{1} rec_nber_data_monthly_{t-1} + b_{20,7}^{1} unemployment_data_monthly_{t-1} + b_{20,8}^{1} unemployment_level_data_monthly_{t-1} + b_{20,9}^{1} all_employed_data_monthly_{t-1} + b_{20,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{20,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{20,12}^{1} m1_data_monthly_{t-1} + b_{20,13}^{1} m2_data_monthly_{t-1} + b_{20,14}^{1} m2_real_data_monthly_{t-1} + b_{20,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{20,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{20,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{20,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{20,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{20,20}^{1} job_openings_data_monthly_{t-1} + b_{20,21}^{1} job_hires_data_monthly_{t-1} + b_{20,22}^{1} job_separations_data_monthly_{t-1} + b_{20,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{20,24}^{1} gdp_data_quarterly_{t-1} + b_{20,25}^{1} gdp_real_data_quarterly_{t-1} + b_{20,26}^{1} cpi_inflation_data_annual_{t-1} + b_{20,27}^{1} median_income_real_data_annual_{t-1} + b_{20,28}^{1} median_income_data_annual_{t-1} + b_{20,29}^{1} Close_{t-1} + b_{20,30}^{1} Volume_{t-1} + b_{20,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{20,2}^{2} cpi_data_monthly_{t-2} + b_{20,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{20,4}^{2} rec_sahm_data_monthly_{t-2} + b_{20,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{20,6}^{2} rec_nber_data_monthly_{t-2} + b_{20,7}^{2} unemployment_data_monthly_{t-2} + b_{20,8}^{2} unemployment_level_data_monthly_{t-2} + b_{20,9}^{2} all_employed_data_monthly_{t-2} + b_{20,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{20,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{20,12}^{2} m1_data_monthly_{t-2} + b_{20,13}^{2} m2_data_monthly_{t-2} + b_{20,14}^{2} m2_real_data_monthly_{t-2} + b_{20,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{20,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{20,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{20,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{20,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{20,20}^{2} job_openings_data_monthly_{t-2} + b_{20,21}^{2} job_hires_data_monthly_{t-2} + b_{20,22}^{2} job_separations_data_monthly_{t-2} + b_{20,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{20,24}^{2} gdp_data_quarterly_{t-2} + b_{20,25}^{2} gdp_real_data_quarterly_{t-2} + b_{20,26}^{2} cpi_inflation_data_annual_{t-2} + b_{20,27}^{2} median_income_real_data_annual_{t-2} + b_{20,28}^{2} median_income_data_annual_{t-2} + b_{20,29}^{2} Close_{t-2} + b_{20,30}^{2} Volume_{t-2} + b_{20,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{20,2}^{3} cpi_data_monthly_{t-3} + b_{20,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{20,4}^{3} rec_sahm_data_monthly_{t-3} + b_{20,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{20,6}^{3} rec_nber_data_monthly_{t-3} + b_{20,7}^{3} unemployment_data_monthly_{t-3} + b_{20,8}^{3} unemployment_level_data_monthly_{t-3} + b_{20,9}^{3} all_employed_data_monthly_{t-3} + b_{20,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{20,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{20,12}^{3} m1_data_monthly_{t-3} + b_{20,13}^{3} m2_data_monthly_{t-3} + b_{20,14}^{3} m2_real_data_monthly_{t-3} + b_{20,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{20,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{20,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{20,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{20,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{20,20}^{3} job_openings_data_monthly_{t-3} + b_{20,21}^{3} job_hires_data_monthly_{t-3} + b_{20,22}^{3} job_separations_data_monthly_{t-3} + b_{20,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{20,24}^{3} gdp_data_quarterly_{t-3} + b_{20,25}^{3} gdp_real_data_quarterly_{t-3} + b_{20,26}^{3} cpi_inflation_data_annual_{t-3} + b_{20,27}^{3} median_income_real_data_annual_{t-3} + b_{20,28}^{3} median_income_data_annual_{t-3} + b_{20,29}^{3} Close_{t-3} + b_{20,30}^{3} Volume_{t-3} + b_{20,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{20,2}^{4} cpi_data_monthly_{t-4} + b_{20,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{20,4}^{4} rec_sahm_data_monthly_{t-4} + b_{20,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{20,6}^{4} rec_nber_data_monthly_{t-4} + b_{20,7}^{4} unemployment_data_monthly_{t-4} + b_{20,8}^{4} unemployment_level_data_monthly_{t-4} + b_{20,9}^{4} all_employed_data_monthly_{t-4} + b_{20,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{20,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{20,12}^{4} m1_data_monthly_{t-4} + b_{20,13}^{4} m2_data_monthly_{t-4} + b_{20,14}^{4} m2_real_data_monthly_{t-4} + b_{20,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{20,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{20,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{20,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{20,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{20,20}^{4} job_openings_data_monthly_{t-4} + b_{20,21}^{4} job_hires_data_monthly_{t-4} + b_{20,22}^{4} job_separations_data_monthly_{t-4} + b_{20,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{20,24}^{4} gdp_data_quarterly_{t-4} + b_{20,25}^{4} gdp_real_data_quarterly_{t-4} + b_{20,26}^{4} cpi_inflation_data_annual_{t-4} + b_{20,27}^{4} median_income_real_data_annual_{t-4} + b_{20,28}^{4} median_income_data_annual_{t-4} + b_{20,29}^{4} Close_{t-4} + b_{20,30}^{4} Volume_{t-4} + b_{20,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{20,2}^{5} cpi_data_monthly_{t-5} + b_{20,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{20,4}^{5} rec_sahm_data_monthly_{t-5} + b_{20,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{20,6}^{5} rec_nber_data_monthly_{t-5} + b_{20,7}^{5} unemployment_data_monthly_{t-5} + b_{20,8}^{5} unemployment_level_data_monthly_{t-5} + b_{20,9}^{5} all_employed_data_monthly_{t-5} + b_{20,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{20,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{20,12}^{5} m1_data_monthly_{t-5} + b_{20,13}^{5} m2_data_monthly_{t-5} + b_{20,14}^{5} m2_real_data_monthly_{t-5} + b_{20,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{20,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{20,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{20,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{20,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{20,20}^{5} job_openings_data_monthly_{t-5} + b_{20,21}^{5} job_hires_data_monthly_{t-5} + b_{20,22}^{5} job_separations_data_monthly_{t-5} + b_{20,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{20,24}^{5} gdp_data_quarterly_{t-5} + b_{20,25}^{5} gdp_real_data_quarterly_{t-5} + b_{20,26}^{5} cpi_inflation_data_annual_{t-5} + b_{20,27}^{5} median_income_real_data_annual_{t-5} + b_{20,28}^{5} median_income_data_annual_{t-5} + b_{20,29}^{5} Close_{t-5} + b_{20,30}^{5} Volume_{t-5} + b_{20,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{20,2}^{6} cpi_data_monthly_{t-6} + b_{20,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{20,4}^{6} rec_sahm_data_monthly_{t-6} + b_{20,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{20,6}^{6} rec_nber_data_monthly_{t-6} + b_{20,7}^{6} unemployment_data_monthly_{t-6} + b_{20,8}^{6} unemployment_level_data_monthly_{t-6} + b_{20,9}^{6} all_employed_data_monthly_{t-6} + b_{20,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{20,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{20,12}^{6} m1_data_monthly_{t-6} + b_{20,13}^{6} m2_data_monthly_{t-6} + b_{20,14}^{6} m2_real_data_monthly_{t-6} + b_{20,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{20,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{20,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{20,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{20,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{20,20}^{6} job_openings_data_monthly_{t-6} + b_{20,21}^{6} job_hires_data_monthly_{t-6} + b_{20,22}^{6} job_separations_data_monthly_{t-6} + b_{20,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{20,24}^{6} gdp_data_quarterly_{t-6} + b_{20,25}^{6} gdp_real_data_quarterly_{t-6} + b_{20,26}^{6} cpi_inflation_data_annual_{t-6} + b_{20,27}^{6} median_income_real_data_annual_{t-6} + b_{20,28}^{6} median_income_data_annual_{t-6} + b_{20,29}^{6} Close_{t-6} + b_{20,30}^{6} Volume_{t-6} + c_{20} + e_{t}^{job_openings_data_monthly} \\
job_hires_data_monthly_{t} &= b_{21,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{21,2}^{1} cpi_data_monthly_{t-1} + b_{21,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{21,4}^{1} rec_sahm_data_monthly_{t-1} + b_{21,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{21,6}^{1} rec_nber_data_monthly_{t-1} + b_{21,7}^{1} unemployment_data_monthly_{t-1} + b_{21,8}^{1} unemployment_level_data_monthly_{t-1} + b_{21,9}^{1} all_employed_data_monthly_{t-1} + b_{21,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{21,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{21,12}^{1} m1_data_monthly_{t-1} + b_{21,13}^{1} m2_data_monthly_{t-1} + b_{21,14}^{1} m2_real_data_monthly_{t-1} + b_{21,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{21,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{21,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{21,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{21,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{21,20}^{1} job_openings_data_monthly_{t-1} + b_{21,21}^{1} job_hires_data_monthly_{t-1} + b_{21,22}^{1} job_separations_data_monthly_{t-1} + b_{21,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{21,24}^{1} gdp_data_quarterly_{t-1} + b_{21,25}^{1} gdp_real_data_quarterly_{t-1} + b_{21,26}^{1} cpi_inflation_data_annual_{t-1} + b_{21,27}^{1} median_income_real_data_annual_{t-1} + b_{21,28}^{1} median_income_data_annual_{t-1} + b_{21,29}^{1} Close_{t-1} + b_{21,30}^{1} Volume_{t-1} + b_{21,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{21,2}^{2} cpi_data_monthly_{t-2} + b_{21,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{21,4}^{2} rec_sahm_data_monthly_{t-2} + b_{21,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{21,6}^{2} rec_nber_data_monthly_{t-2} + b_{21,7}^{2} unemployment_data_monthly_{t-2} + b_{21,8}^{2} unemployment_level_data_monthly_{t-2} + b_{21,9}^{2} all_employed_data_monthly_{t-2} + b_{21,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{21,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{21,12}^{2} m1_data_monthly_{t-2} + b_{21,13}^{2} m2_data_monthly_{t-2} + b_{21,14}^{2} m2_real_data_monthly_{t-2} + b_{21,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{21,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{21,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{21,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{21,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{21,20}^{2} job_openings_data_monthly_{t-2} + b_{21,21}^{2} job_hires_data_monthly_{t-2} + b_{21,22}^{2} job_separations_data_monthly_{t-2} + b_{21,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{21,24}^{2} gdp_data_quarterly_{t-2} + b_{21,25}^{2} gdp_real_data_quarterly_{t-2} + b_{21,26}^{2} cpi_inflation_data_annual_{t-2} + b_{21,27}^{2} median_income_real_data_annual_{t-2} + b_{21,28}^{2} median_income_data_annual_{t-2} + b_{21,29}^{2} Close_{t-2} + b_{21,30}^{2} Volume_{t-2} + b_{21,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{21,2}^{3} cpi_data_monthly_{t-3} + b_{21,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{21,4}^{3} rec_sahm_data_monthly_{t-3} + b_{21,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{21,6}^{3} rec_nber_data_monthly_{t-3} + b_{21,7}^{3} unemployment_data_monthly_{t-3} + b_{21,8}^{3} unemployment_level_data_monthly_{t-3} + b_{21,9}^{3} all_employed_data_monthly_{t-3} + b_{21,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{21,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{21,12}^{3} m1_data_monthly_{t-3} + b_{21,13}^{3} m2_data_monthly_{t-3} + b_{21,14}^{3} m2_real_data_monthly_{t-3} + b_{21,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{21,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{21,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{21,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{21,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{21,20}^{3} job_openings_data_monthly_{t-3} + b_{21,21}^{3} job_hires_data_monthly_{t-3} + b_{21,22}^{3} job_separations_data_monthly_{t-3} + b_{21,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{21,24}^{3} gdp_data_quarterly_{t-3} + b_{21,25}^{3} gdp_real_data_quarterly_{t-3} + b_{21,26}^{3} cpi_inflation_data_annual_{t-3} + b_{21,27}^{3} median_income_real_data_annual_{t-3} + b_{21,28}^{3} median_income_data_annual_{t-3} + b_{21,29}^{3} Close_{t-3} + b_{21,30}^{3} Volume_{t-3} + b_{21,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{21,2}^{4} cpi_data_monthly_{t-4} + b_{21,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{21,4}^{4} rec_sahm_data_monthly_{t-4} + b_{21,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{21,6}^{4} rec_nber_data_monthly_{t-4} + b_{21,7}^{4} unemployment_data_monthly_{t-4} + b_{21,8}^{4} unemployment_level_data_monthly_{t-4} + b_{21,9}^{4} all_employed_data_monthly_{t-4} + b_{21,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{21,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{21,12}^{4} m1_data_monthly_{t-4} + b_{21,13}^{4} m2_data_monthly_{t-4} + b_{21,14}^{4} m2_real_data_monthly_{t-4} + b_{21,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{21,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{21,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{21,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{21,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{21,20}^{4} job_openings_data_monthly_{t-4} + b_{21,21}^{4} job_hires_data_monthly_{t-4} + b_{21,22}^{4} job_separations_data_monthly_{t-4} + b_{21,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{21,24}^{4} gdp_data_quarterly_{t-4} + b_{21,25}^{4} gdp_real_data_quarterly_{t-4} + b_{21,26}^{4} cpi_inflation_data_annual_{t-4} + b_{21,27}^{4} median_income_real_data_annual_{t-4} + b_{21,28}^{4} median_income_data_annual_{t-4} + b_{21,29}^{4} Close_{t-4} + b_{21,30}^{4} Volume_{t-4} + b_{21,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{21,2}^{5} cpi_data_monthly_{t-5} + b_{21,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{21,4}^{5} rec_sahm_data_monthly_{t-5} + b_{21,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{21,6}^{5} rec_nber_data_monthly_{t-5} + b_{21,7}^{5} unemployment_data_monthly_{t-5} + b_{21,8}^{5} unemployment_level_data_monthly_{t-5} + b_{21,9}^{5} all_employed_data_monthly_{t-5} + b_{21,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{21,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{21,12}^{5} m1_data_monthly_{t-5} + b_{21,13}^{5} m2_data_monthly_{t-5} + b_{21,14}^{5} m2_real_data_monthly_{t-5} + b_{21,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{21,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{21,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{21,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{21,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{21,20}^{5} job_openings_data_monthly_{t-5} + b_{21,21}^{5} job_hires_data_monthly_{t-5} + b_{21,22}^{5} job_separations_data_monthly_{t-5} + b_{21,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{21,24}^{5} gdp_data_quarterly_{t-5} + b_{21,25}^{5} gdp_real_data_quarterly_{t-5} + b_{21,26}^{5} cpi_inflation_data_annual_{t-5} + b_{21,27}^{5} median_income_real_data_annual_{t-5} + b_{21,28}^{5} median_income_data_annual_{t-5} + b_{21,29}^{5} Close_{t-5} + b_{21,30}^{5} Volume_{t-5} + b_{21,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{21,2}^{6} cpi_data_monthly_{t-6} + b_{21,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{21,4}^{6} rec_sahm_data_monthly_{t-6} + b_{21,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{21,6}^{6} rec_nber_data_monthly_{t-6} + b_{21,7}^{6} unemployment_data_monthly_{t-6} + b_{21,8}^{6} unemployment_level_data_monthly_{t-6} + b_{21,9}^{6} all_employed_data_monthly_{t-6} + b_{21,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{21,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{21,12}^{6} m1_data_monthly_{t-6} + b_{21,13}^{6} m2_data_monthly_{t-6} + b_{21,14}^{6} m2_real_data_monthly_{t-6} + b_{21,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{21,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{21,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{21,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{21,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{21,20}^{6} job_openings_data_monthly_{t-6} + b_{21,21}^{6} job_hires_data_monthly_{t-6} + b_{21,22}^{6} job_separations_data_monthly_{t-6} + b_{21,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{21,24}^{6} gdp_data_quarterly_{t-6} + b_{21,25}^{6} gdp_real_data_quarterly_{t-6} + b_{21,26}^{6} cpi_inflation_data_annual_{t-6} + b_{21,27}^{6} median_income_real_data_annual_{t-6} + b_{21,28}^{6} median_income_data_annual_{t-6} + b_{21,29}^{6} Close_{t-6} + b_{21,30}^{6} Volume_{t-6} + c_{21} + e_{t}^{job_hires_data_monthly} \\
job_separations_data_monthly_{t} &= b_{22,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{22,2}^{1} cpi_data_monthly_{t-1} + b_{22,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{22,4}^{1} rec_sahm_data_monthly_{t-1} + b_{22,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{22,6}^{1} rec_nber_data_monthly_{t-1} + b_{22,7}^{1} unemployment_data_monthly_{t-1} + b_{22,8}^{1} unemployment_level_data_monthly_{t-1} + b_{22,9}^{1} all_employed_data_monthly_{t-1} + b_{22,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{22,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{22,12}^{1} m1_data_monthly_{t-1} + b_{22,13}^{1} m2_data_monthly_{t-1} + b_{22,14}^{1} m2_real_data_monthly_{t-1} + b_{22,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{22,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{22,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{22,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{22,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{22,20}^{1} job_openings_data_monthly_{t-1} + b_{22,21}^{1} job_hires_data_monthly_{t-1} + b_{22,22}^{1} job_separations_data_monthly_{t-1} + b_{22,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{22,24}^{1} gdp_data_quarterly_{t-1} + b_{22,25}^{1} gdp_real_data_quarterly_{t-1} + b_{22,26}^{1} cpi_inflation_data_annual_{t-1} + b_{22,27}^{1} median_income_real_data_annual_{t-1} + b_{22,28}^{1} median_income_data_annual_{t-1} + b_{22,29}^{1} Close_{t-1} + b_{22,30}^{1} Volume_{t-1} + b_{22,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{22,2}^{2} cpi_data_monthly_{t-2} + b_{22,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{22,4}^{2} rec_sahm_data_monthly_{t-2} + b_{22,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{22,6}^{2} rec_nber_data_monthly_{t-2} + b_{22,7}^{2} unemployment_data_monthly_{t-2} + b_{22,8}^{2} unemployment_level_data_monthly_{t-2} + b_{22,9}^{2} all_employed_data_monthly_{t-2} + b_{22,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{22,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{22,12}^{2} m1_data_monthly_{t-2} + b_{22,13}^{2} m2_data_monthly_{t-2} + b_{22,14}^{2} m2_real_data_monthly_{t-2} + b_{22,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{22,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{22,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{22,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{22,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{22,20}^{2} job_openings_data_monthly_{t-2} + b_{22,21}^{2} job_hires_data_monthly_{t-2} + b_{22,22}^{2} job_separations_data_monthly_{t-2} + b_{22,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{22,24}^{2} gdp_data_quarterly_{t-2} + b_{22,25}^{2} gdp_real_data_quarterly_{t-2} + b_{22,26}^{2} cpi_inflation_data_annual_{t-2} + b_{22,27}^{2} median_income_real_data_annual_{t-2} + b_{22,28}^{2} median_income_data_annual_{t-2} + b_{22,29}^{2} Close_{t-2} + b_{22,30}^{2} Volume_{t-2} + b_{22,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{22,2}^{3} cpi_data_monthly_{t-3} + b_{22,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{22,4}^{3} rec_sahm_data_monthly_{t-3} + b_{22,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{22,6}^{3} rec_nber_data_monthly_{t-3} + b_{22,7}^{3} unemployment_data_monthly_{t-3} + b_{22,8}^{3} unemployment_level_data_monthly_{t-3} + b_{22,9}^{3} all_employed_data_monthly_{t-3} + b_{22,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{22,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{22,12}^{3} m1_data_monthly_{t-3} + b_{22,13}^{3} m2_data_monthly_{t-3} + b_{22,14}^{3} m2_real_data_monthly_{t-3} + b_{22,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{22,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{22,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{22,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{22,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{22,20}^{3} job_openings_data_monthly_{t-3} + b_{22,21}^{3} job_hires_data_monthly_{t-3} + b_{22,22}^{3} job_separations_data_monthly_{t-3} + b_{22,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{22,24}^{3} gdp_data_quarterly_{t-3} + b_{22,25}^{3} gdp_real_data_quarterly_{t-3} + b_{22,26}^{3} cpi_inflation_data_annual_{t-3} + b_{22,27}^{3} median_income_real_data_annual_{t-3} + b_{22,28}^{3} median_income_data_annual_{t-3} + b_{22,29}^{3} Close_{t-3} + b_{22,30}^{3} Volume_{t-3} + b_{22,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{22,2}^{4} cpi_data_monthly_{t-4} + b_{22,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{22,4}^{4} rec_sahm_data_monthly_{t-4} + b_{22,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{22,6}^{4} rec_nber_data_monthly_{t-4} + b_{22,7}^{4} unemployment_data_monthly_{t-4} + b_{22,8}^{4} unemployment_level_data_monthly_{t-4} + b_{22,9}^{4} all_employed_data_monthly_{t-4} + b_{22,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{22,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{22,12}^{4} m1_data_monthly_{t-4} + b_{22,13}^{4} m2_data_monthly_{t-4} + b_{22,14}^{4} m2_real_data_monthly_{t-4} + b_{22,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{22,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{22,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{22,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{22,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{22,20}^{4} job_openings_data_monthly_{t-4} + b_{22,21}^{4} job_hires_data_monthly_{t-4} + b_{22,22}^{4} job_separations_data_monthly_{t-4} + b_{22,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{22,24}^{4} gdp_data_quarterly_{t-4} + b_{22,25}^{4} gdp_real_data_quarterly_{t-4} + b_{22,26}^{4} cpi_inflation_data_annual_{t-4} + b_{22,27}^{4} median_income_real_data_annual_{t-4} + b_{22,28}^{4} median_income_data_annual_{t-4} + b_{22,29}^{4} Close_{t-4} + b_{22,30}^{4} Volume_{t-4} + b_{22,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{22,2}^{5} cpi_data_monthly_{t-5} + b_{22,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{22,4}^{5} rec_sahm_data_monthly_{t-5} + b_{22,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{22,6}^{5} rec_nber_data_monthly_{t-5} + b_{22,7}^{5} unemployment_data_monthly_{t-5} + b_{22,8}^{5} unemployment_level_data_monthly_{t-5} + b_{22,9}^{5} all_employed_data_monthly_{t-5} + b_{22,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{22,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{22,12}^{5} m1_data_monthly_{t-5} + b_{22,13}^{5} m2_data_monthly_{t-5} + b_{22,14}^{5} m2_real_data_monthly_{t-5} + b_{22,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{22,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{22,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{22,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{22,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{22,20}^{5} job_openings_data_monthly_{t-5} + b_{22,21}^{5} job_hires_data_monthly_{t-5} + b_{22,22}^{5} job_separations_data_monthly_{t-5} + b_{22,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{22,24}^{5} gdp_data_quarterly_{t-5} + b_{22,25}^{5} gdp_real_data_quarterly_{t-5} + b_{22,26}^{5} cpi_inflation_data_annual_{t-5} + b_{22,27}^{5} median_income_real_data_annual_{t-5} + b_{22,28}^{5} median_income_data_annual_{t-5} + b_{22,29}^{5} Close_{t-5} + b_{22,30}^{5} Volume_{t-5} + b_{22,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{22,2}^{6} cpi_data_monthly_{t-6} + b_{22,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{22,4}^{6} rec_sahm_data_monthly_{t-6} + b_{22,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{22,6}^{6} rec_nber_data_monthly_{t-6} + b_{22,7}^{6} unemployment_data_monthly_{t-6} + b_{22,8}^{6} unemployment_level_data_monthly_{t-6} + b_{22,9}^{6} all_employed_data_monthly_{t-6} + b_{22,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{22,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{22,12}^{6} m1_data_monthly_{t-6} + b_{22,13}^{6} m2_data_monthly_{t-6} + b_{22,14}^{6} m2_real_data_monthly_{t-6} + b_{22,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{22,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{22,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{22,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{22,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{22,20}^{6} job_openings_data_monthly_{t-6} + b_{22,21}^{6} job_hires_data_monthly_{t-6} + b_{22,22}^{6} job_separations_data_monthly_{t-6} + b_{22,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{22,24}^{6} gdp_data_quarterly_{t-6} + b_{22,25}^{6} gdp_real_data_quarterly_{t-6} + b_{22,26}^{6} cpi_inflation_data_annual_{t-6} + b_{22,27}^{6} median_income_real_data_annual_{t-6} + b_{22,28}^{6} median_income_data_annual_{t-6} + b_{22,29}^{6} Close_{t-6} + b_{22,30}^{6} Volume_{t-6} + c_{22} + e_{t}^{job_separations_data_monthly} \\
average_weekly_earnings_data_monthly_{t} &= b_{23,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{23,2}^{1} cpi_data_monthly_{t-1} + b_{23,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{23,4}^{1} rec_sahm_data_monthly_{t-1} + b_{23,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{23,6}^{1} rec_nber_data_monthly_{t-1} + b_{23,7}^{1} unemployment_data_monthly_{t-1} + b_{23,8}^{1} unemployment_level_data_monthly_{t-1} + b_{23,9}^{1} all_employed_data_monthly_{t-1} + b_{23,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{23,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{23,12}^{1} m1_data_monthly_{t-1} + b_{23,13}^{1} m2_data_monthly_{t-1} + b_{23,14}^{1} m2_real_data_monthly_{t-1} + b_{23,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{23,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{23,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{23,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{23,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{23,20}^{1} job_openings_data_monthly_{t-1} + b_{23,21}^{1} job_hires_data_monthly_{t-1} + b_{23,22}^{1} job_separations_data_monthly_{t-1} + b_{23,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{23,24}^{1} gdp_data_quarterly_{t-1} + b_{23,25}^{1} gdp_real_data_quarterly_{t-1} + b_{23,26}^{1} cpi_inflation_data_annual_{t-1} + b_{23,27}^{1} median_income_real_data_annual_{t-1} + b_{23,28}^{1} median_income_data_annual_{t-1} + b_{23,29}^{1} Close_{t-1} + b_{23,30}^{1} Volume_{t-1} + b_{23,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{23,2}^{2} cpi_data_monthly_{t-2} + b_{23,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{23,4}^{2} rec_sahm_data_monthly_{t-2} + b_{23,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{23,6}^{2} rec_nber_data_monthly_{t-2} + b_{23,7}^{2} unemployment_data_monthly_{t-2} + b_{23,8}^{2} unemployment_level_data_monthly_{t-2} + b_{23,9}^{2} all_employed_data_monthly_{t-2} + b_{23,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{23,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{23,12}^{2} m1_data_monthly_{t-2} + b_{23,13}^{2} m2_data_monthly_{t-2} + b_{23,14}^{2} m2_real_data_monthly_{t-2} + b_{23,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{23,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{23,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{23,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{23,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{23,20}^{2} job_openings_data_monthly_{t-2} + b_{23,21}^{2} job_hires_data_monthly_{t-2} + b_{23,22}^{2} job_separations_data_monthly_{t-2} + b_{23,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{23,24}^{2} gdp_data_quarterly_{t-2} + b_{23,25}^{2} gdp_real_data_quarterly_{t-2} + b_{23,26}^{2} cpi_inflation_data_annual_{t-2} + b_{23,27}^{2} median_income_real_data_annual_{t-2} + b_{23,28}^{2} median_income_data_annual_{t-2} + b_{23,29}^{2} Close_{t-2} + b_{23,30}^{2} Volume_{t-2} + b_{23,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{23,2}^{3} cpi_data_monthly_{t-3} + b_{23,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{23,4}^{3} rec_sahm_data_monthly_{t-3} + b_{23,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{23,6}^{3} rec_nber_data_monthly_{t-3} + b_{23,7}^{3} unemployment_data_monthly_{t-3} + b_{23,8}^{3} unemployment_level_data_monthly_{t-3} + b_{23,9}^{3} all_employed_data_monthly_{t-3} + b_{23,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{23,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{23,12}^{3} m1_data_monthly_{t-3} + b_{23,13}^{3} m2_data_monthly_{t-3} + b_{23,14}^{3} m2_real_data_monthly_{t-3} + b_{23,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{23,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{23,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{23,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{23,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{23,20}^{3} job_openings_data_monthly_{t-3} + b_{23,21}^{3} job_hires_data_monthly_{t-3} + b_{23,22}^{3} job_separations_data_monthly_{t-3} + b_{23,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{23,24}^{3} gdp_data_quarterly_{t-3} + b_{23,25}^{3} gdp_real_data_quarterly_{t-3} + b_{23,26}^{3} cpi_inflation_data_annual_{t-3} + b_{23,27}^{3} median_income_real_data_annual_{t-3} + b_{23,28}^{3} median_income_data_annual_{t-3} + b_{23,29}^{3} Close_{t-3} + b_{23,30}^{3} Volume_{t-3} + b_{23,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{23,2}^{4} cpi_data_monthly_{t-4} + b_{23,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{23,4}^{4} rec_sahm_data_monthly_{t-4} + b_{23,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{23,6}^{4} rec_nber_data_monthly_{t-4} + b_{23,7}^{4} unemployment_data_monthly_{t-4} + b_{23,8}^{4} unemployment_level_data_monthly_{t-4} + b_{23,9}^{4} all_employed_data_monthly_{t-4} + b_{23,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{23,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{23,12}^{4} m1_data_monthly_{t-4} + b_{23,13}^{4} m2_data_monthly_{t-4} + b_{23,14}^{4} m2_real_data_monthly_{t-4} + b_{23,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{23,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{23,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{23,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{23,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{23,20}^{4} job_openings_data_monthly_{t-4} + b_{23,21}^{4} job_hires_data_monthly_{t-4} + b_{23,22}^{4} job_separations_data_monthly_{t-4} + b_{23,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{23,24}^{4} gdp_data_quarterly_{t-4} + b_{23,25}^{4} gdp_real_data_quarterly_{t-4} + b_{23,26}^{4} cpi_inflation_data_annual_{t-4} + b_{23,27}^{4} median_income_real_data_annual_{t-4} + b_{23,28}^{4} median_income_data_annual_{t-4} + b_{23,29}^{4} Close_{t-4} + b_{23,30}^{4} Volume_{t-4} + b_{23,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{23,2}^{5} cpi_data_monthly_{t-5} + b_{23,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{23,4}^{5} rec_sahm_data_monthly_{t-5} + b_{23,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{23,6}^{5} rec_nber_data_monthly_{t-5} + b_{23,7}^{5} unemployment_data_monthly_{t-5} + b_{23,8}^{5} unemployment_level_data_monthly_{t-5} + b_{23,9}^{5} all_employed_data_monthly_{t-5} + b_{23,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{23,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{23,12}^{5} m1_data_monthly_{t-5} + b_{23,13}^{5} m2_data_monthly_{t-5} + b_{23,14}^{5} m2_real_data_monthly_{t-5} + b_{23,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{23,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{23,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{23,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{23,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{23,20}^{5} job_openings_data_monthly_{t-5} + b_{23,21}^{5} job_hires_data_monthly_{t-5} + b_{23,22}^{5} job_separations_data_monthly_{t-5} + b_{23,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{23,24}^{5} gdp_data_quarterly_{t-5} + b_{23,25}^{5} gdp_real_data_quarterly_{t-5} + b_{23,26}^{5} cpi_inflation_data_annual_{t-5} + b_{23,27}^{5} median_income_real_data_annual_{t-5} + b_{23,28}^{5} median_income_data_annual_{t-5} + b_{23,29}^{5} Close_{t-5} + b_{23,30}^{5} Volume_{t-5} + b_{23,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{23,2}^{6} cpi_data_monthly_{t-6} + b_{23,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{23,4}^{6} rec_sahm_data_monthly_{t-6} + b_{23,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{23,6}^{6} rec_nber_data_monthly_{t-6} + b_{23,7}^{6} unemployment_data_monthly_{t-6} + b_{23,8}^{6} unemployment_level_data_monthly_{t-6} + b_{23,9}^{6} all_employed_data_monthly_{t-6} + b_{23,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{23,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{23,12}^{6} m1_data_monthly_{t-6} + b_{23,13}^{6} m2_data_monthly_{t-6} + b_{23,14}^{6} m2_real_data_monthly_{t-6} + b_{23,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{23,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{23,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{23,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{23,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{23,20}^{6} job_openings_data_monthly_{t-6} + b_{23,21}^{6} job_hires_data_monthly_{t-6} + b_{23,22}^{6} job_separations_data_monthly_{t-6} + b_{23,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{23,24}^{6} gdp_data_quarterly_{t-6} + b_{23,25}^{6} gdp_real_data_quarterly_{t-6} + b_{23,26}^{6} cpi_inflation_data_annual_{t-6} + b_{23,27}^{6} median_income_real_data_annual_{t-6} + b_{23,28}^{6} median_income_data_annual_{t-6} + b_{23,29}^{6} Close_{t-6} + b_{23,30}^{6} Volume_{t-6} + c_{23} + e_{t}^{average_weekly_earnings_data_monthly} \\
gdp_data_quarterly_{t} &= b_{24,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{24,2}^{1} cpi_data_monthly_{t-1} + b_{24,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{24,4}^{1} rec_sahm_data_monthly_{t-1} + b_{24,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{24,6}^{1} rec_nber_data_monthly_{t-1} + b_{24,7}^{1} unemployment_data_monthly_{t-1} + b_{24,8}^{1} unemployment_level_data_monthly_{t-1} + b_{24,9}^{1} all_employed_data_monthly_{t-1} + b_{24,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{24,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{24,12}^{1} m1_data_monthly_{t-1} + b_{24,13}^{1} m2_data_monthly_{t-1} + b_{24,14}^{1} m2_real_data_monthly_{t-1} + b_{24,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{24,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{24,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{24,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{24,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{24,20}^{1} job_openings_data_monthly_{t-1} + b_{24,21}^{1} job_hires_data_monthly_{t-1} + b_{24,22}^{1} job_separations_data_monthly_{t-1} + b_{24,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{24,24}^{1} gdp_data_quarterly_{t-1} + b_{24,25}^{1} gdp_real_data_quarterly_{t-1} + b_{24,26}^{1} cpi_inflation_data_annual_{t-1} + b_{24,27}^{1} median_income_real_data_annual_{t-1} + b_{24,28}^{1} median_income_data_annual_{t-1} + b_{24,29}^{1} Close_{t-1} + b_{24,30}^{1} Volume_{t-1} + b_{24,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{24,2}^{2} cpi_data_monthly_{t-2} + b_{24,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{24,4}^{2} rec_sahm_data_monthly_{t-2} + b_{24,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{24,6}^{2} rec_nber_data_monthly_{t-2} + b_{24,7}^{2} unemployment_data_monthly_{t-2} + b_{24,8}^{2} unemployment_level_data_monthly_{t-2} + b_{24,9}^{2} all_employed_data_monthly_{t-2} + b_{24,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{24,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{24,12}^{2} m1_data_monthly_{t-2} + b_{24,13}^{2} m2_data_monthly_{t-2} + b_{24,14}^{2} m2_real_data_monthly_{t-2} + b_{24,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{24,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{24,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{24,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{24,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{24,20}^{2} job_openings_data_monthly_{t-2} + b_{24,21}^{2} job_hires_data_monthly_{t-2} + b_{24,22}^{2} job_separations_data_monthly_{t-2} + b_{24,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{24,24}^{2} gdp_data_quarterly_{t-2} + b_{24,25}^{2} gdp_real_data_quarterly_{t-2} + b_{24,26}^{2} cpi_inflation_data_annual_{t-2} + b_{24,27}^{2} median_income_real_data_annual_{t-2} + b_{24,28}^{2} median_income_data_annual_{t-2} + b_{24,29}^{2} Close_{t-2} + b_{24,30}^{2} Volume_{t-2} + b_{24,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{24,2}^{3} cpi_data_monthly_{t-3} + b_{24,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{24,4}^{3} rec_sahm_data_monthly_{t-3} + b_{24,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{24,6}^{3} rec_nber_data_monthly_{t-3} + b_{24,7}^{3} unemployment_data_monthly_{t-3} + b_{24,8}^{3} unemployment_level_data_monthly_{t-3} + b_{24,9}^{3} all_employed_data_monthly_{t-3} + b_{24,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{24,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{24,12}^{3} m1_data_monthly_{t-3} + b_{24,13}^{3} m2_data_monthly_{t-3} + b_{24,14}^{3} m2_real_data_monthly_{t-3} + b_{24,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{24,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{24,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{24,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{24,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{24,20}^{3} job_openings_data_monthly_{t-3} + b_{24,21}^{3} job_hires_data_monthly_{t-3} + b_{24,22}^{3} job_separations_data_monthly_{t-3} + b_{24,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{24,24}^{3} gdp_data_quarterly_{t-3} + b_{24,25}^{3} gdp_real_data_quarterly_{t-3} + b_{24,26}^{3} cpi_inflation_data_annual_{t-3} + b_{24,27}^{3} median_income_real_data_annual_{t-3} + b_{24,28}^{3} median_income_data_annual_{t-3} + b_{24,29}^{3} Close_{t-3} + b_{24,30}^{3} Volume_{t-3} + b_{24,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{24,2}^{4} cpi_data_monthly_{t-4} + b_{24,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{24,4}^{4} rec_sahm_data_monthly_{t-4} + b_{24,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{24,6}^{4} rec_nber_data_monthly_{t-4} + b_{24,7}^{4} unemployment_data_monthly_{t-4} + b_{24,8}^{4} unemployment_level_data_monthly_{t-4} + b_{24,9}^{4} all_employed_data_monthly_{t-4} + b_{24,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{24,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{24,12}^{4} m1_data_monthly_{t-4} + b_{24,13}^{4} m2_data_monthly_{t-4} + b_{24,14}^{4} m2_real_data_monthly_{t-4} + b_{24,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{24,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{24,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{24,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{24,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{24,20}^{4} job_openings_data_monthly_{t-4} + b_{24,21}^{4} job_hires_data_monthly_{t-4} + b_{24,22}^{4} job_separations_data_monthly_{t-4} + b_{24,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{24,24}^{4} gdp_data_quarterly_{t-4} + b_{24,25}^{4} gdp_real_data_quarterly_{t-4} + b_{24,26}^{4} cpi_inflation_data_annual_{t-4} + b_{24,27}^{4} median_income_real_data_annual_{t-4} + b_{24,28}^{4} median_income_data_annual_{t-4} + b_{24,29}^{4} Close_{t-4} + b_{24,30}^{4} Volume_{t-4} + b_{24,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{24,2}^{5} cpi_data_monthly_{t-5} + b_{24,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{24,4}^{5} rec_sahm_data_monthly_{t-5} + b_{24,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{24,6}^{5} rec_nber_data_monthly_{t-5} + b_{24,7}^{5} unemployment_data_monthly_{t-5} + b_{24,8}^{5} unemployment_level_data_monthly_{t-5} + b_{24,9}^{5} all_employed_data_monthly_{t-5} + b_{24,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{24,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{24,12}^{5} m1_data_monthly_{t-5} + b_{24,13}^{5} m2_data_monthly_{t-5} + b_{24,14}^{5} m2_real_data_monthly_{t-5} + b_{24,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{24,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{24,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{24,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{24,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{24,20}^{5} job_openings_data_monthly_{t-5} + b_{24,21}^{5} job_hires_data_monthly_{t-5} + b_{24,22}^{5} job_separations_data_monthly_{t-5} + b_{24,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{24,24}^{5} gdp_data_quarterly_{t-5} + b_{24,25}^{5} gdp_real_data_quarterly_{t-5} + b_{24,26}^{5} cpi_inflation_data_annual_{t-5} + b_{24,27}^{5} median_income_real_data_annual_{t-5} + b_{24,28}^{5} median_income_data_annual_{t-5} + b_{24,29}^{5} Close_{t-5} + b_{24,30}^{5} Volume_{t-5} + b_{24,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{24,2}^{6} cpi_data_monthly_{t-6} + b_{24,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{24,4}^{6} rec_sahm_data_monthly_{t-6} + b_{24,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{24,6}^{6} rec_nber_data_monthly_{t-6} + b_{24,7}^{6} unemployment_data_monthly_{t-6} + b_{24,8}^{6} unemployment_level_data_monthly_{t-6} + b_{24,9}^{6} all_employed_data_monthly_{t-6} + b_{24,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{24,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{24,12}^{6} m1_data_monthly_{t-6} + b_{24,13}^{6} m2_data_monthly_{t-6} + b_{24,14}^{6} m2_real_data_monthly_{t-6} + b_{24,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{24,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{24,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{24,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{24,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{24,20}^{6} job_openings_data_monthly_{t-6} + b_{24,21}^{6} job_hires_data_monthly_{t-6} + b_{24,22}^{6} job_separations_data_monthly_{t-6} + b_{24,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{24,24}^{6} gdp_data_quarterly_{t-6} + b_{24,25}^{6} gdp_real_data_quarterly_{t-6} + b_{24,26}^{6} cpi_inflation_data_annual_{t-6} + b_{24,27}^{6} median_income_real_data_annual_{t-6} + b_{24,28}^{6} median_income_data_annual_{t-6} + b_{24,29}^{6} Close_{t-6} + b_{24,30}^{6} Volume_{t-6} + c_{24} + e_{t}^{gdp_data_quarterly} \\
gdp_real_data_quarterly_{t} &= b_{25,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{25,2}^{1} cpi_data_monthly_{t-1} + b_{25,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{25,4}^{1} rec_sahm_data_monthly_{t-1} + b_{25,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{25,6}^{1} rec_nber_data_monthly_{t-1} + b_{25,7}^{1} unemployment_data_monthly_{t-1} + b_{25,8}^{1} unemployment_level_data_monthly_{t-1} + b_{25,9}^{1} all_employed_data_monthly_{t-1} + b_{25,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{25,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{25,12}^{1} m1_data_monthly_{t-1} + b_{25,13}^{1} m2_data_monthly_{t-1} + b_{25,14}^{1} m2_real_data_monthly_{t-1} + b_{25,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{25,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{25,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{25,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{25,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{25,20}^{1} job_openings_data_monthly_{t-1} + b_{25,21}^{1} job_hires_data_monthly_{t-1} + b_{25,22}^{1} job_separations_data_monthly_{t-1} + b_{25,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{25,24}^{1} gdp_data_quarterly_{t-1} + b_{25,25}^{1} gdp_real_data_quarterly_{t-1} + b_{25,26}^{1} cpi_inflation_data_annual_{t-1} + b_{25,27}^{1} median_income_real_data_annual_{t-1} + b_{25,28}^{1} median_income_data_annual_{t-1} + b_{25,29}^{1} Close_{t-1} + b_{25,30}^{1} Volume_{t-1} + b_{25,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{25,2}^{2} cpi_data_monthly_{t-2} + b_{25,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{25,4}^{2} rec_sahm_data_monthly_{t-2} + b_{25,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{25,6}^{2} rec_nber_data_monthly_{t-2} + b_{25,7}^{2} unemployment_data_monthly_{t-2} + b_{25,8}^{2} unemployment_level_data_monthly_{t-2} + b_{25,9}^{2} all_employed_data_monthly_{t-2} + b_{25,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{25,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{25,12}^{2} m1_data_monthly_{t-2} + b_{25,13}^{2} m2_data_monthly_{t-2} + b_{25,14}^{2} m2_real_data_monthly_{t-2} + b_{25,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{25,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{25,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{25,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{25,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{25,20}^{2} job_openings_data_monthly_{t-2} + b_{25,21}^{2} job_hires_data_monthly_{t-2} + b_{25,22}^{2} job_separations_data_monthly_{t-2} + b_{25,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{25,24}^{2} gdp_data_quarterly_{t-2} + b_{25,25}^{2} gdp_real_data_quarterly_{t-2} + b_{25,26}^{2} cpi_inflation_data_annual_{t-2} + b_{25,27}^{2} median_income_real_data_annual_{t-2} + b_{25,28}^{2} median_income_data_annual_{t-2} + b_{25,29}^{2} Close_{t-2} + b_{25,30}^{2} Volume_{t-2} + b_{25,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{25,2}^{3} cpi_data_monthly_{t-3} + b_{25,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{25,4}^{3} rec_sahm_data_monthly_{t-3} + b_{25,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{25,6}^{3} rec_nber_data_monthly_{t-3} + b_{25,7}^{3} unemployment_data_monthly_{t-3} + b_{25,8}^{3} unemployment_level_data_monthly_{t-3} + b_{25,9}^{3} all_employed_data_monthly_{t-3} + b_{25,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{25,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{25,12}^{3} m1_data_monthly_{t-3} + b_{25,13}^{3} m2_data_monthly_{t-3} + b_{25,14}^{3} m2_real_data_monthly_{t-3} + b_{25,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{25,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{25,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{25,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{25,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{25,20}^{3} job_openings_data_monthly_{t-3} + b_{25,21}^{3} job_hires_data_monthly_{t-3} + b_{25,22}^{3} job_separations_data_monthly_{t-3} + b_{25,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{25,24}^{3} gdp_data_quarterly_{t-3} + b_{25,25}^{3} gdp_real_data_quarterly_{t-3} + b_{25,26}^{3} cpi_inflation_data_annual_{t-3} + b_{25,27}^{3} median_income_real_data_annual_{t-3} + b_{25,28}^{3} median_income_data_annual_{t-3} + b_{25,29}^{3} Close_{t-3} + b_{25,30}^{3} Volume_{t-3} + b_{25,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{25,2}^{4} cpi_data_monthly_{t-4} + b_{25,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{25,4}^{4} rec_sahm_data_monthly_{t-4} + b_{25,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{25,6}^{4} rec_nber_data_monthly_{t-4} + b_{25,7}^{4} unemployment_data_monthly_{t-4} + b_{25,8}^{4} unemployment_level_data_monthly_{t-4} + b_{25,9}^{4} all_employed_data_monthly_{t-4} + b_{25,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{25,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{25,12}^{4} m1_data_monthly_{t-4} + b_{25,13}^{4} m2_data_monthly_{t-4} + b_{25,14}^{4} m2_real_data_monthly_{t-4} + b_{25,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{25,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{25,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{25,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{25,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{25,20}^{4} job_openings_data_monthly_{t-4} + b_{25,21}^{4} job_hires_data_monthly_{t-4} + b_{25,22}^{4} job_separations_data_monthly_{t-4} + b_{25,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{25,24}^{4} gdp_data_quarterly_{t-4} + b_{25,25}^{4} gdp_real_data_quarterly_{t-4} + b_{25,26}^{4} cpi_inflation_data_annual_{t-4} + b_{25,27}^{4} median_income_real_data_annual_{t-4} + b_{25,28}^{4} median_income_data_annual_{t-4} + b_{25,29}^{4} Close_{t-4} + b_{25,30}^{4} Volume_{t-4} + b_{25,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{25,2}^{5} cpi_data_monthly_{t-5} + b_{25,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{25,4}^{5} rec_sahm_data_monthly_{t-5} + b_{25,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{25,6}^{5} rec_nber_data_monthly_{t-5} + b_{25,7}^{5} unemployment_data_monthly_{t-5} + b_{25,8}^{5} unemployment_level_data_monthly_{t-5} + b_{25,9}^{5} all_employed_data_monthly_{t-5} + b_{25,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{25,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{25,12}^{5} m1_data_monthly_{t-5} + b_{25,13}^{5} m2_data_monthly_{t-5} + b_{25,14}^{5} m2_real_data_monthly_{t-5} + b_{25,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{25,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{25,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{25,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{25,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{25,20}^{5} job_openings_data_monthly_{t-5} + b_{25,21}^{5} job_hires_data_monthly_{t-5} + b_{25,22}^{5} job_separations_data_monthly_{t-5} + b_{25,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{25,24}^{5} gdp_data_quarterly_{t-5} + b_{25,25}^{5} gdp_real_data_quarterly_{t-5} + b_{25,26}^{5} cpi_inflation_data_annual_{t-5} + b_{25,27}^{5} median_income_real_data_annual_{t-5} + b_{25,28}^{5} median_income_data_annual_{t-5} + b_{25,29}^{5} Close_{t-5} + b_{25,30}^{5} Volume_{t-5} + b_{25,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{25,2}^{6} cpi_data_monthly_{t-6} + b_{25,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{25,4}^{6} rec_sahm_data_monthly_{t-6} + b_{25,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{25,6}^{6} rec_nber_data_monthly_{t-6} + b_{25,7}^{6} unemployment_data_monthly_{t-6} + b_{25,8}^{6} unemployment_level_data_monthly_{t-6} + b_{25,9}^{6} all_employed_data_monthly_{t-6} + b_{25,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{25,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{25,12}^{6} m1_data_monthly_{t-6} + b_{25,13}^{6} m2_data_monthly_{t-6} + b_{25,14}^{6} m2_real_data_monthly_{t-6} + b_{25,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{25,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{25,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{25,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{25,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{25,20}^{6} job_openings_data_monthly_{t-6} + b_{25,21}^{6} job_hires_data_monthly_{t-6} + b_{25,22}^{6} job_separations_data_monthly_{t-6} + b_{25,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{25,24}^{6} gdp_data_quarterly_{t-6} + b_{25,25}^{6} gdp_real_data_quarterly_{t-6} + b_{25,26}^{6} cpi_inflation_data_annual_{t-6} + b_{25,27}^{6} median_income_real_data_annual_{t-6} + b_{25,28}^{6} median_income_data_annual_{t-6} + b_{25,29}^{6} Close_{t-6} + b_{25,30}^{6} Volume_{t-6} + c_{25} + e_{t}^{gdp_real_data_quarterly} \\
cpi_inflation_data_annual_{t} &= b_{26,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{26,2}^{1} cpi_data_monthly_{t-1} + b_{26,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{26,4}^{1} rec_sahm_data_monthly_{t-1} + b_{26,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{26,6}^{1} rec_nber_data_monthly_{t-1} + b_{26,7}^{1} unemployment_data_monthly_{t-1} + b_{26,8}^{1} unemployment_level_data_monthly_{t-1} + b_{26,9}^{1} all_employed_data_monthly_{t-1} + b_{26,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{26,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{26,12}^{1} m1_data_monthly_{t-1} + b_{26,13}^{1} m2_data_monthly_{t-1} + b_{26,14}^{1} m2_real_data_monthly_{t-1} + b_{26,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{26,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{26,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{26,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{26,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{26,20}^{1} job_openings_data_monthly_{t-1} + b_{26,21}^{1} job_hires_data_monthly_{t-1} + b_{26,22}^{1} job_separations_data_monthly_{t-1} + b_{26,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{26,24}^{1} gdp_data_quarterly_{t-1} + b_{26,25}^{1} gdp_real_data_quarterly_{t-1} + b_{26,26}^{1} cpi_inflation_data_annual_{t-1} + b_{26,27}^{1} median_income_real_data_annual_{t-1} + b_{26,28}^{1} median_income_data_annual_{t-1} + b_{26,29}^{1} Close_{t-1} + b_{26,30}^{1} Volume_{t-1} + b_{26,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{26,2}^{2} cpi_data_monthly_{t-2} + b_{26,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{26,4}^{2} rec_sahm_data_monthly_{t-2} + b_{26,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{26,6}^{2} rec_nber_data_monthly_{t-2} + b_{26,7}^{2} unemployment_data_monthly_{t-2} + b_{26,8}^{2} unemployment_level_data_monthly_{t-2} + b_{26,9}^{2} all_employed_data_monthly_{t-2} + b_{26,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{26,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{26,12}^{2} m1_data_monthly_{t-2} + b_{26,13}^{2} m2_data_monthly_{t-2} + b_{26,14}^{2} m2_real_data_monthly_{t-2} + b_{26,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{26,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{26,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{26,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{26,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{26,20}^{2} job_openings_data_monthly_{t-2} + b_{26,21}^{2} job_hires_data_monthly_{t-2} + b_{26,22}^{2} job_separations_data_monthly_{t-2} + b_{26,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{26,24}^{2} gdp_data_quarterly_{t-2} + b_{26,25}^{2} gdp_real_data_quarterly_{t-2} + b_{26,26}^{2} cpi_inflation_data_annual_{t-2} + b_{26,27}^{2} median_income_real_data_annual_{t-2} + b_{26,28}^{2} median_income_data_annual_{t-2} + b_{26,29}^{2} Close_{t-2} + b_{26,30}^{2} Volume_{t-2} + b_{26,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{26,2}^{3} cpi_data_monthly_{t-3} + b_{26,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{26,4}^{3} rec_sahm_data_monthly_{t-3} + b_{26,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{26,6}^{3} rec_nber_data_monthly_{t-3} + b_{26,7}^{3} unemployment_data_monthly_{t-3} + b_{26,8}^{3} unemployment_level_data_monthly_{t-3} + b_{26,9}^{3} all_employed_data_monthly_{t-3} + b_{26,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{26,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{26,12}^{3} m1_data_monthly_{t-3} + b_{26,13}^{3} m2_data_monthly_{t-3} + b_{26,14}^{3} m2_real_data_monthly_{t-3} + b_{26,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{26,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{26,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{26,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{26,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{26,20}^{3} job_openings_data_monthly_{t-3} + b_{26,21}^{3} job_hires_data_monthly_{t-3} + b_{26,22}^{3} job_separations_data_monthly_{t-3} + b_{26,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{26,24}^{3} gdp_data_quarterly_{t-3} + b_{26,25}^{3} gdp_real_data_quarterly_{t-3} + b_{26,26}^{3} cpi_inflation_data_annual_{t-3} + b_{26,27}^{3} median_income_real_data_annual_{t-3} + b_{26,28}^{3} median_income_data_annual_{t-3} + b_{26,29}^{3} Close_{t-3} + b_{26,30}^{3} Volume_{t-3} + b_{26,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{26,2}^{4} cpi_data_monthly_{t-4} + b_{26,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{26,4}^{4} rec_sahm_data_monthly_{t-4} + b_{26,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{26,6}^{4} rec_nber_data_monthly_{t-4} + b_{26,7}^{4} unemployment_data_monthly_{t-4} + b_{26,8}^{4} unemployment_level_data_monthly_{t-4} + b_{26,9}^{4} all_employed_data_monthly_{t-4} + b_{26,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{26,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{26,12}^{4} m1_data_monthly_{t-4} + b_{26,13}^{4} m2_data_monthly_{t-4} + b_{26,14}^{4} m2_real_data_monthly_{t-4} + b_{26,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{26,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{26,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{26,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{26,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{26,20}^{4} job_openings_data_monthly_{t-4} + b_{26,21}^{4} job_hires_data_monthly_{t-4} + b_{26,22}^{4} job_separations_data_monthly_{t-4} + b_{26,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{26,24}^{4} gdp_data_quarterly_{t-4} + b_{26,25}^{4} gdp_real_data_quarterly_{t-4} + b_{26,26}^{4} cpi_inflation_data_annual_{t-4} + b_{26,27}^{4} median_income_real_data_annual_{t-4} + b_{26,28}^{4} median_income_data_annual_{t-4} + b_{26,29}^{4} Close_{t-4} + b_{26,30}^{4} Volume_{t-4} + b_{26,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{26,2}^{5} cpi_data_monthly_{t-5} + b_{26,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{26,4}^{5} rec_sahm_data_monthly_{t-5} + b_{26,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{26,6}^{5} rec_nber_data_monthly_{t-5} + b_{26,7}^{5} unemployment_data_monthly_{t-5} + b_{26,8}^{5} unemployment_level_data_monthly_{t-5} + b_{26,9}^{5} all_employed_data_monthly_{t-5} + b_{26,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{26,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{26,12}^{5} m1_data_monthly_{t-5} + b_{26,13}^{5} m2_data_monthly_{t-5} + b_{26,14}^{5} m2_real_data_monthly_{t-5} + b_{26,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{26,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{26,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{26,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{26,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{26,20}^{5} job_openings_data_monthly_{t-5} + b_{26,21}^{5} job_hires_data_monthly_{t-5} + b_{26,22}^{5} job_separations_data_monthly_{t-5} + b_{26,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{26,24}^{5} gdp_data_quarterly_{t-5} + b_{26,25}^{5} gdp_real_data_quarterly_{t-5} + b_{26,26}^{5} cpi_inflation_data_annual_{t-5} + b_{26,27}^{5} median_income_real_data_annual_{t-5} + b_{26,28}^{5} median_income_data_annual_{t-5} + b_{26,29}^{5} Close_{t-5} + b_{26,30}^{5} Volume_{t-5} + b_{26,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{26,2}^{6} cpi_data_monthly_{t-6} + b_{26,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{26,4}^{6} rec_sahm_data_monthly_{t-6} + b_{26,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{26,6}^{6} rec_nber_data_monthly_{t-6} + b_{26,7}^{6} unemployment_data_monthly_{t-6} + b_{26,8}^{6} unemployment_level_data_monthly_{t-6} + b_{26,9}^{6} all_employed_data_monthly_{t-6} + b_{26,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{26,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{26,12}^{6} m1_data_monthly_{t-6} + b_{26,13}^{6} m2_data_monthly_{t-6} + b_{26,14}^{6} m2_real_data_monthly_{t-6} + b_{26,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{26,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{26,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{26,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{26,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{26,20}^{6} job_openings_data_monthly_{t-6} + b_{26,21}^{6} job_hires_data_monthly_{t-6} + b_{26,22}^{6} job_separations_data_monthly_{t-6} + b_{26,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{26,24}^{6} gdp_data_quarterly_{t-6} + b_{26,25}^{6} gdp_real_data_quarterly_{t-6} + b_{26,26}^{6} cpi_inflation_data_annual_{t-6} + b_{26,27}^{6} median_income_real_data_annual_{t-6} + b_{26,28}^{6} median_income_data_annual_{t-6} + b_{26,29}^{6} Close_{t-6} + b_{26,30}^{6} Volume_{t-6} + c_{26} + e_{t}^{cpi_inflation_data_annual} \\
median_income_real_data_annual_{t} &= b_{27,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{27,2}^{1} cpi_data_monthly_{t-1} + b_{27,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{27,4}^{1} rec_sahm_data_monthly_{t-1} + b_{27,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{27,6}^{1} rec_nber_data_monthly_{t-1} + b_{27,7}^{1} unemployment_data_monthly_{t-1} + b_{27,8}^{1} unemployment_level_data_monthly_{t-1} + b_{27,9}^{1} all_employed_data_monthly_{t-1} + b_{27,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{27,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{27,12}^{1} m1_data_monthly_{t-1} + b_{27,13}^{1} m2_data_monthly_{t-1} + b_{27,14}^{1} m2_real_data_monthly_{t-1} + b_{27,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{27,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{27,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{27,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{27,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{27,20}^{1} job_openings_data_monthly_{t-1} + b_{27,21}^{1} job_hires_data_monthly_{t-1} + b_{27,22}^{1} job_separations_data_monthly_{t-1} + b_{27,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{27,24}^{1} gdp_data_quarterly_{t-1} + b_{27,25}^{1} gdp_real_data_quarterly_{t-1} + b_{27,26}^{1} cpi_inflation_data_annual_{t-1} + b_{27,27}^{1} median_income_real_data_annual_{t-1} + b_{27,28}^{1} median_income_data_annual_{t-1} + b_{27,29}^{1} Close_{t-1} + b_{27,30}^{1} Volume_{t-1} + b_{27,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{27,2}^{2} cpi_data_monthly_{t-2} + b_{27,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{27,4}^{2} rec_sahm_data_monthly_{t-2} + b_{27,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{27,6}^{2} rec_nber_data_monthly_{t-2} + b_{27,7}^{2} unemployment_data_monthly_{t-2} + b_{27,8}^{2} unemployment_level_data_monthly_{t-2} + b_{27,9}^{2} all_employed_data_monthly_{t-2} + b_{27,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{27,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{27,12}^{2} m1_data_monthly_{t-2} + b_{27,13}^{2} m2_data_monthly_{t-2} + b_{27,14}^{2} m2_real_data_monthly_{t-2} + b_{27,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{27,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{27,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{27,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{27,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{27,20}^{2} job_openings_data_monthly_{t-2} + b_{27,21}^{2} job_hires_data_monthly_{t-2} + b_{27,22}^{2} job_separations_data_monthly_{t-2} + b_{27,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{27,24}^{2} gdp_data_quarterly_{t-2} + b_{27,25}^{2} gdp_real_data_quarterly_{t-2} + b_{27,26}^{2} cpi_inflation_data_annual_{t-2} + b_{27,27}^{2} median_income_real_data_annual_{t-2} + b_{27,28}^{2} median_income_data_annual_{t-2} + b_{27,29}^{2} Close_{t-2} + b_{27,30}^{2} Volume_{t-2} + b_{27,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{27,2}^{3} cpi_data_monthly_{t-3} + b_{27,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{27,4}^{3} rec_sahm_data_monthly_{t-3} + b_{27,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{27,6}^{3} rec_nber_data_monthly_{t-3} + b_{27,7}^{3} unemployment_data_monthly_{t-3} + b_{27,8}^{3} unemployment_level_data_monthly_{t-3} + b_{27,9}^{3} all_employed_data_monthly_{t-3} + b_{27,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{27,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{27,12}^{3} m1_data_monthly_{t-3} + b_{27,13}^{3} m2_data_monthly_{t-3} + b_{27,14}^{3} m2_real_data_monthly_{t-3} + b_{27,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{27,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{27,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{27,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{27,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{27,20}^{3} job_openings_data_monthly_{t-3} + b_{27,21}^{3} job_hires_data_monthly_{t-3} + b_{27,22}^{3} job_separations_data_monthly_{t-3} + b_{27,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{27,24}^{3} gdp_data_quarterly_{t-3} + b_{27,25}^{3} gdp_real_data_quarterly_{t-3} + b_{27,26}^{3} cpi_inflation_data_annual_{t-3} + b_{27,27}^{3} median_income_real_data_annual_{t-3} + b_{27,28}^{3} median_income_data_annual_{t-3} + b_{27,29}^{3} Close_{t-3} + b_{27,30}^{3} Volume_{t-3} + b_{27,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{27,2}^{4} cpi_data_monthly_{t-4} + b_{27,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{27,4}^{4} rec_sahm_data_monthly_{t-4} + b_{27,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{27,6}^{4} rec_nber_data_monthly_{t-4} + b_{27,7}^{4} unemployment_data_monthly_{t-4} + b_{27,8}^{4} unemployment_level_data_monthly_{t-4} + b_{27,9}^{4} all_employed_data_monthly_{t-4} + b_{27,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{27,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{27,12}^{4} m1_data_monthly_{t-4} + b_{27,13}^{4} m2_data_monthly_{t-4} + b_{27,14}^{4} m2_real_data_monthly_{t-4} + b_{27,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{27,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{27,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{27,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{27,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{27,20}^{4} job_openings_data_monthly_{t-4} + b_{27,21}^{4} job_hires_data_monthly_{t-4} + b_{27,22}^{4} job_separations_data_monthly_{t-4} + b_{27,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{27,24}^{4} gdp_data_quarterly_{t-4} + b_{27,25}^{4} gdp_real_data_quarterly_{t-4} + b_{27,26}^{4} cpi_inflation_data_annual_{t-4} + b_{27,27}^{4} median_income_real_data_annual_{t-4} + b_{27,28}^{4} median_income_data_annual_{t-4} + b_{27,29}^{4} Close_{t-4} + b_{27,30}^{4} Volume_{t-4} + b_{27,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{27,2}^{5} cpi_data_monthly_{t-5} + b_{27,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{27,4}^{5} rec_sahm_data_monthly_{t-5} + b_{27,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{27,6}^{5} rec_nber_data_monthly_{t-5} + b_{27,7}^{5} unemployment_data_monthly_{t-5} + b_{27,8}^{5} unemployment_level_data_monthly_{t-5} + b_{27,9}^{5} all_employed_data_monthly_{t-5} + b_{27,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{27,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{27,12}^{5} m1_data_monthly_{t-5} + b_{27,13}^{5} m2_data_monthly_{t-5} + b_{27,14}^{5} m2_real_data_monthly_{t-5} + b_{27,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{27,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{27,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{27,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{27,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{27,20}^{5} job_openings_data_monthly_{t-5} + b_{27,21}^{5} job_hires_data_monthly_{t-5} + b_{27,22}^{5} job_separations_data_monthly_{t-5} + b_{27,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{27,24}^{5} gdp_data_quarterly_{t-5} + b_{27,25}^{5} gdp_real_data_quarterly_{t-5} + b_{27,26}^{5} cpi_inflation_data_annual_{t-5} + b_{27,27}^{5} median_income_real_data_annual_{t-5} + b_{27,28}^{5} median_income_data_annual_{t-5} + b_{27,29}^{5} Close_{t-5} + b_{27,30}^{5} Volume_{t-5} + b_{27,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{27,2}^{6} cpi_data_monthly_{t-6} + b_{27,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{27,4}^{6} rec_sahm_data_monthly_{t-6} + b_{27,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{27,6}^{6} rec_nber_data_monthly_{t-6} + b_{27,7}^{6} unemployment_data_monthly_{t-6} + b_{27,8}^{6} unemployment_level_data_monthly_{t-6} + b_{27,9}^{6} all_employed_data_monthly_{t-6} + b_{27,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{27,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{27,12}^{6} m1_data_monthly_{t-6} + b_{27,13}^{6} m2_data_monthly_{t-6} + b_{27,14}^{6} m2_real_data_monthly_{t-6} + b_{27,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{27,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{27,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{27,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{27,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{27,20}^{6} job_openings_data_monthly_{t-6} + b_{27,21}^{6} job_hires_data_monthly_{t-6} + b_{27,22}^{6} job_separations_data_monthly_{t-6} + b_{27,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{27,24}^{6} gdp_data_quarterly_{t-6} + b_{27,25}^{6} gdp_real_data_quarterly_{t-6} + b_{27,26}^{6} cpi_inflation_data_annual_{t-6} + b_{27,27}^{6} median_income_real_data_annual_{t-6} + b_{27,28}^{6} median_income_data_annual_{t-6} + b_{27,29}^{6} Close_{t-6} + b_{27,30}^{6} Volume_{t-6} + c_{27} + e_{t}^{median_income_real_data_annual} \\
median_income_data_annual_{t} &= b_{28,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{28,2}^{1} cpi_data_monthly_{t-1} + b_{28,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{28,4}^{1} rec_sahm_data_monthly_{t-1} + b_{28,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{28,6}^{1} rec_nber_data_monthly_{t-1} + b_{28,7}^{1} unemployment_data_monthly_{t-1} + b_{28,8}^{1} unemployment_level_data_monthly_{t-1} + b_{28,9}^{1} all_employed_data_monthly_{t-1} + b_{28,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{28,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{28,12}^{1} m1_data_monthly_{t-1} + b_{28,13}^{1} m2_data_monthly_{t-1} + b_{28,14}^{1} m2_real_data_monthly_{t-1} + b_{28,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{28,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{28,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{28,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{28,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{28,20}^{1} job_openings_data_monthly_{t-1} + b_{28,21}^{1} job_hires_data_monthly_{t-1} + b_{28,22}^{1} job_separations_data_monthly_{t-1} + b_{28,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{28,24}^{1} gdp_data_quarterly_{t-1} + b_{28,25}^{1} gdp_real_data_quarterly_{t-1} + b_{28,26}^{1} cpi_inflation_data_annual_{t-1} + b_{28,27}^{1} median_income_real_data_annual_{t-1} + b_{28,28}^{1} median_income_data_annual_{t-1} + b_{28,29}^{1} Close_{t-1} + b_{28,30}^{1} Volume_{t-1} + b_{28,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{28,2}^{2} cpi_data_monthly_{t-2} + b_{28,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{28,4}^{2} rec_sahm_data_monthly_{t-2} + b_{28,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{28,6}^{2} rec_nber_data_monthly_{t-2} + b_{28,7}^{2} unemployment_data_monthly_{t-2} + b_{28,8}^{2} unemployment_level_data_monthly_{t-2} + b_{28,9}^{2} all_employed_data_monthly_{t-2} + b_{28,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{28,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{28,12}^{2} m1_data_monthly_{t-2} + b_{28,13}^{2} m2_data_monthly_{t-2} + b_{28,14}^{2} m2_real_data_monthly_{t-2} + b_{28,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{28,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{28,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{28,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{28,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{28,20}^{2} job_openings_data_monthly_{t-2} + b_{28,21}^{2} job_hires_data_monthly_{t-2} + b_{28,22}^{2} job_separations_data_monthly_{t-2} + b_{28,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{28,24}^{2} gdp_data_quarterly_{t-2} + b_{28,25}^{2} gdp_real_data_quarterly_{t-2} + b_{28,26}^{2} cpi_inflation_data_annual_{t-2} + b_{28,27}^{2} median_income_real_data_annual_{t-2} + b_{28,28}^{2} median_income_data_annual_{t-2} + b_{28,29}^{2} Close_{t-2} + b_{28,30}^{2} Volume_{t-2} + b_{28,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{28,2}^{3} cpi_data_monthly_{t-3} + b_{28,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{28,4}^{3} rec_sahm_data_monthly_{t-3} + b_{28,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{28,6}^{3} rec_nber_data_monthly_{t-3} + b_{28,7}^{3} unemployment_data_monthly_{t-3} + b_{28,8}^{3} unemployment_level_data_monthly_{t-3} + b_{28,9}^{3} all_employed_data_monthly_{t-3} + b_{28,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{28,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{28,12}^{3} m1_data_monthly_{t-3} + b_{28,13}^{3} m2_data_monthly_{t-3} + b_{28,14}^{3} m2_real_data_monthly_{t-3} + b_{28,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{28,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{28,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{28,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{28,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{28,20}^{3} job_openings_data_monthly_{t-3} + b_{28,21}^{3} job_hires_data_monthly_{t-3} + b_{28,22}^{3} job_separations_data_monthly_{t-3} + b_{28,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{28,24}^{3} gdp_data_quarterly_{t-3} + b_{28,25}^{3} gdp_real_data_quarterly_{t-3} + b_{28,26}^{3} cpi_inflation_data_annual_{t-3} + b_{28,27}^{3} median_income_real_data_annual_{t-3} + b_{28,28}^{3} median_income_data_annual_{t-3} + b_{28,29}^{3} Close_{t-3} + b_{28,30}^{3} Volume_{t-3} + b_{28,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{28,2}^{4} cpi_data_monthly_{t-4} + b_{28,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{28,4}^{4} rec_sahm_data_monthly_{t-4} + b_{28,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{28,6}^{4} rec_nber_data_monthly_{t-4} + b_{28,7}^{4} unemployment_data_monthly_{t-4} + b_{28,8}^{4} unemployment_level_data_monthly_{t-4} + b_{28,9}^{4} all_employed_data_monthly_{t-4} + b_{28,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{28,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{28,12}^{4} m1_data_monthly_{t-4} + b_{28,13}^{4} m2_data_monthly_{t-4} + b_{28,14}^{4} m2_real_data_monthly_{t-4} + b_{28,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{28,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{28,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{28,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{28,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{28,20}^{4} job_openings_data_monthly_{t-4} + b_{28,21}^{4} job_hires_data_monthly_{t-4} + b_{28,22}^{4} job_separations_data_monthly_{t-4} + b_{28,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{28,24}^{4} gdp_data_quarterly_{t-4} + b_{28,25}^{4} gdp_real_data_quarterly_{t-4} + b_{28,26}^{4} cpi_inflation_data_annual_{t-4} + b_{28,27}^{4} median_income_real_data_annual_{t-4} + b_{28,28}^{4} median_income_data_annual_{t-4} + b_{28,29}^{4} Close_{t-4} + b_{28,30}^{4} Volume_{t-4} + b_{28,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{28,2}^{5} cpi_data_monthly_{t-5} + b_{28,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{28,4}^{5} rec_sahm_data_monthly_{t-5} + b_{28,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{28,6}^{5} rec_nber_data_monthly_{t-5} + b_{28,7}^{5} unemployment_data_monthly_{t-5} + b_{28,8}^{5} unemployment_level_data_monthly_{t-5} + b_{28,9}^{5} all_employed_data_monthly_{t-5} + b_{28,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{28,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{28,12}^{5} m1_data_monthly_{t-5} + b_{28,13}^{5} m2_data_monthly_{t-5} + b_{28,14}^{5} m2_real_data_monthly_{t-5} + b_{28,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{28,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{28,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{28,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{28,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{28,20}^{5} job_openings_data_monthly_{t-5} + b_{28,21}^{5} job_hires_data_monthly_{t-5} + b_{28,22}^{5} job_separations_data_monthly_{t-5} + b_{28,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{28,24}^{5} gdp_data_quarterly_{t-5} + b_{28,25}^{5} gdp_real_data_quarterly_{t-5} + b_{28,26}^{5} cpi_inflation_data_annual_{t-5} + b_{28,27}^{5} median_income_real_data_annual_{t-5} + b_{28,28}^{5} median_income_data_annual_{t-5} + b_{28,29}^{5} Close_{t-5} + b_{28,30}^{5} Volume_{t-5} + b_{28,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{28,2}^{6} cpi_data_monthly_{t-6} + b_{28,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{28,4}^{6} rec_sahm_data_monthly_{t-6} + b_{28,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{28,6}^{6} rec_nber_data_monthly_{t-6} + b_{28,7}^{6} unemployment_data_monthly_{t-6} + b_{28,8}^{6} unemployment_level_data_monthly_{t-6} + b_{28,9}^{6} all_employed_data_monthly_{t-6} + b_{28,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{28,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{28,12}^{6} m1_data_monthly_{t-6} + b_{28,13}^{6} m2_data_monthly_{t-6} + b_{28,14}^{6} m2_real_data_monthly_{t-6} + b_{28,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{28,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{28,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{28,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{28,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{28,20}^{6} job_openings_data_monthly_{t-6} + b_{28,21}^{6} job_hires_data_monthly_{t-6} + b_{28,22}^{6} job_separations_data_monthly_{t-6} + b_{28,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{28,24}^{6} gdp_data_quarterly_{t-6} + b_{28,25}^{6} gdp_real_data_quarterly_{t-6} + b_{28,26}^{6} cpi_inflation_data_annual_{t-6} + b_{28,27}^{6} median_income_real_data_annual_{t-6} + b_{28,28}^{6} median_income_data_annual_{t-6} + b_{28,29}^{6} Close_{t-6} + b_{28,30}^{6} Volume_{t-6} + c_{28} + e_{t}^{median_income_data_annual} \\
Close_{t} &= b_{29,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{29,2}^{1} cpi_data_monthly_{t-1} + b_{29,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{29,4}^{1} rec_sahm_data_monthly_{t-1} + b_{29,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{29,6}^{1} rec_nber_data_monthly_{t-1} + b_{29,7}^{1} unemployment_data_monthly_{t-1} + b_{29,8}^{1} unemployment_level_data_monthly_{t-1} + b_{29,9}^{1} all_employed_data_monthly_{t-1} + b_{29,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{29,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{29,12}^{1} m1_data_monthly_{t-1} + b_{29,13}^{1} m2_data_monthly_{t-1} + b_{29,14}^{1} m2_real_data_monthly_{t-1} + b_{29,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{29,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{29,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{29,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{29,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{29,20}^{1} job_openings_data_monthly_{t-1} + b_{29,21}^{1} job_hires_data_monthly_{t-1} + b_{29,22}^{1} job_separations_data_monthly_{t-1} + b_{29,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{29,24}^{1} gdp_data_quarterly_{t-1} + b_{29,25}^{1} gdp_real_data_quarterly_{t-1} + b_{29,26}^{1} cpi_inflation_data_annual_{t-1} + b_{29,27}^{1} median_income_real_data_annual_{t-1} + b_{29,28}^{1} median_income_data_annual_{t-1} + b_{29,29}^{1} Close_{t-1} + b_{29,30}^{1} Volume_{t-1} + b_{29,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{29,2}^{2} cpi_data_monthly_{t-2} + b_{29,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{29,4}^{2} rec_sahm_data_monthly_{t-2} + b_{29,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{29,6}^{2} rec_nber_data_monthly_{t-2} + b_{29,7}^{2} unemployment_data_monthly_{t-2} + b_{29,8}^{2} unemployment_level_data_monthly_{t-2} + b_{29,9}^{2} all_employed_data_monthly_{t-2} + b_{29,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{29,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{29,12}^{2} m1_data_monthly_{t-2} + b_{29,13}^{2} m2_data_monthly_{t-2} + b_{29,14}^{2} m2_real_data_monthly_{t-2} + b_{29,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{29,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{29,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{29,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{29,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{29,20}^{2} job_openings_data_monthly_{t-2} + b_{29,21}^{2} job_hires_data_monthly_{t-2} + b_{29,22}^{2} job_separations_data_monthly_{t-2} + b_{29,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{29,24}^{2} gdp_data_quarterly_{t-2} + b_{29,25}^{2} gdp_real_data_quarterly_{t-2} + b_{29,26}^{2} cpi_inflation_data_annual_{t-2} + b_{29,27}^{2} median_income_real_data_annual_{t-2} + b_{29,28}^{2} median_income_data_annual_{t-2} + b_{29,29}^{2} Close_{t-2} + b_{29,30}^{2} Volume_{t-2} + b_{29,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{29,2}^{3} cpi_data_monthly_{t-3} + b_{29,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{29,4}^{3} rec_sahm_data_monthly_{t-3} + b_{29,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{29,6}^{3} rec_nber_data_monthly_{t-3} + b_{29,7}^{3} unemployment_data_monthly_{t-3} + b_{29,8}^{3} unemployment_level_data_monthly_{t-3} + b_{29,9}^{3} all_employed_data_monthly_{t-3} + b_{29,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{29,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{29,12}^{3} m1_data_monthly_{t-3} + b_{29,13}^{3} m2_data_monthly_{t-3} + b_{29,14}^{3} m2_real_data_monthly_{t-3} + b_{29,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{29,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{29,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{29,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{29,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{29,20}^{3} job_openings_data_monthly_{t-3} + b_{29,21}^{3} job_hires_data_monthly_{t-3} + b_{29,22}^{3} job_separations_data_monthly_{t-3} + b_{29,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{29,24}^{3} gdp_data_quarterly_{t-3} + b_{29,25}^{3} gdp_real_data_quarterly_{t-3} + b_{29,26}^{3} cpi_inflation_data_annual_{t-3} + b_{29,27}^{3} median_income_real_data_annual_{t-3} + b_{29,28}^{3} median_income_data_annual_{t-3} + b_{29,29}^{3} Close_{t-3} + b_{29,30}^{3} Volume_{t-3} + b_{29,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{29,2}^{4} cpi_data_monthly_{t-4} + b_{29,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{29,4}^{4} rec_sahm_data_monthly_{t-4} + b_{29,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{29,6}^{4} rec_nber_data_monthly_{t-4} + b_{29,7}^{4} unemployment_data_monthly_{t-4} + b_{29,8}^{4} unemployment_level_data_monthly_{t-4} + b_{29,9}^{4} all_employed_data_monthly_{t-4} + b_{29,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{29,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{29,12}^{4} m1_data_monthly_{t-4} + b_{29,13}^{4} m2_data_monthly_{t-4} + b_{29,14}^{4} m2_real_data_monthly_{t-4} + b_{29,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{29,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{29,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{29,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{29,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{29,20}^{4} job_openings_data_monthly_{t-4} + b_{29,21}^{4} job_hires_data_monthly_{t-4} + b_{29,22}^{4} job_separations_data_monthly_{t-4} + b_{29,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{29,24}^{4} gdp_data_quarterly_{t-4} + b_{29,25}^{4} gdp_real_data_quarterly_{t-4} + b_{29,26}^{4} cpi_inflation_data_annual_{t-4} + b_{29,27}^{4} median_income_real_data_annual_{t-4} + b_{29,28}^{4} median_income_data_annual_{t-4} + b_{29,29}^{4} Close_{t-4} + b_{29,30}^{4} Volume_{t-4} + b_{29,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{29,2}^{5} cpi_data_monthly_{t-5} + b_{29,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{29,4}^{5} rec_sahm_data_monthly_{t-5} + b_{29,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{29,6}^{5} rec_nber_data_monthly_{t-5} + b_{29,7}^{5} unemployment_data_monthly_{t-5} + b_{29,8}^{5} unemployment_level_data_monthly_{t-5} + b_{29,9}^{5} all_employed_data_monthly_{t-5} + b_{29,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{29,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{29,12}^{5} m1_data_monthly_{t-5} + b_{29,13}^{5} m2_data_monthly_{t-5} + b_{29,14}^{5} m2_real_data_monthly_{t-5} + b_{29,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{29,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{29,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{29,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{29,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{29,20}^{5} job_openings_data_monthly_{t-5} + b_{29,21}^{5} job_hires_data_monthly_{t-5} + b_{29,22}^{5} job_separations_data_monthly_{t-5} + b_{29,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{29,24}^{5} gdp_data_quarterly_{t-5} + b_{29,25}^{5} gdp_real_data_quarterly_{t-5} + b_{29,26}^{5} cpi_inflation_data_annual_{t-5} + b_{29,27}^{5} median_income_real_data_annual_{t-5} + b_{29,28}^{5} median_income_data_annual_{t-5} + b_{29,29}^{5} Close_{t-5} + b_{29,30}^{5} Volume_{t-5} + b_{29,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{29,2}^{6} cpi_data_monthly_{t-6} + b_{29,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{29,4}^{6} rec_sahm_data_monthly_{t-6} + b_{29,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{29,6}^{6} rec_nber_data_monthly_{t-6} + b_{29,7}^{6} unemployment_data_monthly_{t-6} + b_{29,8}^{6} unemployment_level_data_monthly_{t-6} + b_{29,9}^{6} all_employed_data_monthly_{t-6} + b_{29,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{29,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{29,12}^{6} m1_data_monthly_{t-6} + b_{29,13}^{6} m2_data_monthly_{t-6} + b_{29,14}^{6} m2_real_data_monthly_{t-6} + b_{29,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{29,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{29,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{29,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{29,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{29,20}^{6} job_openings_data_monthly_{t-6} + b_{29,21}^{6} job_hires_data_monthly_{t-6} + b_{29,22}^{6} job_separations_data_monthly_{t-6} + b_{29,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{29,24}^{6} gdp_data_quarterly_{t-6} + b_{29,25}^{6} gdp_real_data_quarterly_{t-6} + b_{29,26}^{6} cpi_inflation_data_annual_{t-6} + b_{29,27}^{6} median_income_real_data_annual_{t-6} + b_{29,28}^{6} median_income_data_annual_{t-6} + b_{29,29}^{6} Close_{t-6} + b_{29,30}^{6} Volume_{t-6} + c_{29} + e_{t}^{Close} \\
Volume_{t} &= b_{30,1}^{1} interest_rate_mtg_data_weekly_{t-1} + b_{30,2}^{1} cpi_data_monthly_{t-1} + b_{30,3}^{1} cpi_fesl_data_monthly_{t-1} + b_{30,4}^{1} rec_sahm_data_monthly_{t-1} + b_{30,5}^{1} rec_smooth_prob_data_monthly_{t-1} + b_{30,6}^{1} rec_nber_data_monthly_{t-1} + b_{30,7}^{1} unemployment_data_monthly_{t-1} + b_{30,8}^{1} unemployment_level_data_monthly_{t-1} + b_{30,9}^{1} all_employed_data_monthly_{t-1} + b_{30,10}^{1} employment_population_ratio_data_monthly_{t-1} + b_{30,11}^{1} labor_force_participation_data_monthly_{t-1} + b_{30,12}^{1} m1_data_monthly_{t-1} + b_{30,13}^{1} m2_data_monthly_{t-1} + b_{30,14}^{1} m2_real_data_monthly_{t-1} + b_{30,15}^{1} interest_rate_fedfunds_data_monthly_{t-1} + b_{30,16}^{1} consumer_sentiment_data_monthly_{t-1} + b_{30,17}^{1} consumer_sentiment_inflation_data_monthly_{t-1} + b_{30,18}^{1} consumer_sentiment_composite_data_monthly_{t-1} + b_{30,19}^{1} consumer_sentiment_composite_amplitude_data_monthly_{t-1} + b_{30,20}^{1} job_openings_data_monthly_{t-1} + b_{30,21}^{1} job_hires_data_monthly_{t-1} + b_{30,22}^{1} job_separations_data_monthly_{t-1} + b_{30,23}^{1} average_weekly_earnings_data_monthly_{t-1} + b_{30,24}^{1} gdp_data_quarterly_{t-1} + b_{30,25}^{1} gdp_real_data_quarterly_{t-1} + b_{30,26}^{1} cpi_inflation_data_annual_{t-1} + b_{30,27}^{1} median_income_real_data_annual_{t-1} + b_{30,28}^{1} median_income_data_annual_{t-1} + b_{30,29}^{1} Close_{t-1} + b_{30,30}^{1} Volume_{t-1} + b_{30,1}^{2} interest_rate_mtg_data_weekly_{t-2} + b_{30,2}^{2} cpi_data_monthly_{t-2} + b_{30,3}^{2} cpi_fesl_data_monthly_{t-2} + b_{30,4}^{2} rec_sahm_data_monthly_{t-2} + b_{30,5}^{2} rec_smooth_prob_data_monthly_{t-2} + b_{30,6}^{2} rec_nber_data_monthly_{t-2} + b_{30,7}^{2} unemployment_data_monthly_{t-2} + b_{30,8}^{2} unemployment_level_data_monthly_{t-2} + b_{30,9}^{2} all_employed_data_monthly_{t-2} + b_{30,10}^{2} employment_population_ratio_data_monthly_{t-2} + b_{30,11}^{2} labor_force_participation_data_monthly_{t-2} + b_{30,12}^{2} m1_data_monthly_{t-2} + b_{30,13}^{2} m2_data_monthly_{t-2} + b_{30,14}^{2} m2_real_data_monthly_{t-2} + b_{30,15}^{2} interest_rate_fedfunds_data_monthly_{t-2} + b_{30,16}^{2} consumer_sentiment_data_monthly_{t-2} + b_{30,17}^{2} consumer_sentiment_inflation_data_monthly_{t-2} + b_{30,18}^{2} consumer_sentiment_composite_data_monthly_{t-2} + b_{30,19}^{2} consumer_sentiment_composite_amplitude_data_monthly_{t-2} + b_{30,20}^{2} job_openings_data_monthly_{t-2} + b_{30,21}^{2} job_hires_data_monthly_{t-2} + b_{30,22}^{2} job_separations_data_monthly_{t-2} + b_{30,23}^{2} average_weekly_earnings_data_monthly_{t-2} + b_{30,24}^{2} gdp_data_quarterly_{t-2} + b_{30,25}^{2} gdp_real_data_quarterly_{t-2} + b_{30,26}^{2} cpi_inflation_data_annual_{t-2} + b_{30,27}^{2} median_income_real_data_annual_{t-2} + b_{30,28}^{2} median_income_data_annual_{t-2} + b_{30,29}^{2} Close_{t-2} + b_{30,30}^{2} Volume_{t-2} + b_{30,1}^{3} interest_rate_mtg_data_weekly_{t-3} + b_{30,2}^{3} cpi_data_monthly_{t-3} + b_{30,3}^{3} cpi_fesl_data_monthly_{t-3} + b_{30,4}^{3} rec_sahm_data_monthly_{t-3} + b_{30,5}^{3} rec_smooth_prob_data_monthly_{t-3} + b_{30,6}^{3} rec_nber_data_monthly_{t-3} + b_{30,7}^{3} unemployment_data_monthly_{t-3} + b_{30,8}^{3} unemployment_level_data_monthly_{t-3} + b_{30,9}^{3} all_employed_data_monthly_{t-3} + b_{30,10}^{3} employment_population_ratio_data_monthly_{t-3} + b_{30,11}^{3} labor_force_participation_data_monthly_{t-3} + b_{30,12}^{3} m1_data_monthly_{t-3} + b_{30,13}^{3} m2_data_monthly_{t-3} + b_{30,14}^{3} m2_real_data_monthly_{t-3} + b_{30,15}^{3} interest_rate_fedfunds_data_monthly_{t-3} + b_{30,16}^{3} consumer_sentiment_data_monthly_{t-3} + b_{30,17}^{3} consumer_sentiment_inflation_data_monthly_{t-3} + b_{30,18}^{3} consumer_sentiment_composite_data_monthly_{t-3} + b_{30,19}^{3} consumer_sentiment_composite_amplitude_data_monthly_{t-3} + b_{30,20}^{3} job_openings_data_monthly_{t-3} + b_{30,21}^{3} job_hires_data_monthly_{t-3} + b_{30,22}^{3} job_separations_data_monthly_{t-3} + b_{30,23}^{3} average_weekly_earnings_data_monthly_{t-3} + b_{30,24}^{3} gdp_data_quarterly_{t-3} + b_{30,25}^{3} gdp_real_data_quarterly_{t-3} + b_{30,26}^{3} cpi_inflation_data_annual_{t-3} + b_{30,27}^{3} median_income_real_data_annual_{t-3} + b_{30,28}^{3} median_income_data_annual_{t-3} + b_{30,29}^{3} Close_{t-3} + b_{30,30}^{3} Volume_{t-3} + b_{30,1}^{4} interest_rate_mtg_data_weekly_{t-4} + b_{30,2}^{4} cpi_data_monthly_{t-4} + b_{30,3}^{4} cpi_fesl_data_monthly_{t-4} + b_{30,4}^{4} rec_sahm_data_monthly_{t-4} + b_{30,5}^{4} rec_smooth_prob_data_monthly_{t-4} + b_{30,6}^{4} rec_nber_data_monthly_{t-4} + b_{30,7}^{4} unemployment_data_monthly_{t-4} + b_{30,8}^{4} unemployment_level_data_monthly_{t-4} + b_{30,9}^{4} all_employed_data_monthly_{t-4} + b_{30,10}^{4} employment_population_ratio_data_monthly_{t-4} + b_{30,11}^{4} labor_force_participation_data_monthly_{t-4} + b_{30,12}^{4} m1_data_monthly_{t-4} + b_{30,13}^{4} m2_data_monthly_{t-4} + b_{30,14}^{4} m2_real_data_monthly_{t-4} + b_{30,15}^{4} interest_rate_fedfunds_data_monthly_{t-4} + b_{30,16}^{4} consumer_sentiment_data_monthly_{t-4} + b_{30,17}^{4} consumer_sentiment_inflation_data_monthly_{t-4} + b_{30,18}^{4} consumer_sentiment_composite_data_monthly_{t-4} + b_{30,19}^{4} consumer_sentiment_composite_amplitude_data_monthly_{t-4} + b_{30,20}^{4} job_openings_data_monthly_{t-4} + b_{30,21}^{4} job_hires_data_monthly_{t-4} + b_{30,22}^{4} job_separations_data_monthly_{t-4} + b_{30,23}^{4} average_weekly_earnings_data_monthly_{t-4} + b_{30,24}^{4} gdp_data_quarterly_{t-4} + b_{30,25}^{4} gdp_real_data_quarterly_{t-4} + b_{30,26}^{4} cpi_inflation_data_annual_{t-4} + b_{30,27}^{4} median_income_real_data_annual_{t-4} + b_{30,28}^{4} median_income_data_annual_{t-4} + b_{30,29}^{4} Close_{t-4} + b_{30,30}^{4} Volume_{t-4} + b_{30,1}^{5} interest_rate_mtg_data_weekly_{t-5} + b_{30,2}^{5} cpi_data_monthly_{t-5} + b_{30,3}^{5} cpi_fesl_data_monthly_{t-5} + b_{30,4}^{5} rec_sahm_data_monthly_{t-5} + b_{30,5}^{5} rec_smooth_prob_data_monthly_{t-5} + b_{30,6}^{5} rec_nber_data_monthly_{t-5} + b_{30,7}^{5} unemployment_data_monthly_{t-5} + b_{30,8}^{5} unemployment_level_data_monthly_{t-5} + b_{30,9}^{5} all_employed_data_monthly_{t-5} + b_{30,10}^{5} employment_population_ratio_data_monthly_{t-5} + b_{30,11}^{5} labor_force_participation_data_monthly_{t-5} + b_{30,12}^{5} m1_data_monthly_{t-5} + b_{30,13}^{5} m2_data_monthly_{t-5} + b_{30,14}^{5} m2_real_data_monthly_{t-5} + b_{30,15}^{5} interest_rate_fedfunds_data_monthly_{t-5} + b_{30,16}^{5} consumer_sentiment_data_monthly_{t-5} + b_{30,17}^{5} consumer_sentiment_inflation_data_monthly_{t-5} + b_{30,18}^{5} consumer_sentiment_composite_data_monthly_{t-5} + b_{30,19}^{5} consumer_sentiment_composite_amplitude_data_monthly_{t-5} + b_{30,20}^{5} job_openings_data_monthly_{t-5} + b_{30,21}^{5} job_hires_data_monthly_{t-5} + b_{30,22}^{5} job_separations_data_monthly_{t-5} + b_{30,23}^{5} average_weekly_earnings_data_monthly_{t-5} + b_{30,24}^{5} gdp_data_quarterly_{t-5} + b_{30,25}^{5} gdp_real_data_quarterly_{t-5} + b_{30,26}^{5} cpi_inflation_data_annual_{t-5} + b_{30,27}^{5} median_income_real_data_annual_{t-5} + b_{30,28}^{5} median_income_data_annual_{t-5} + b_{30,29}^{5} Close_{t-5} + b_{30,30}^{5} Volume_{t-5} + b_{30,1}^{6} interest_rate_mtg_data_weekly_{t-6} + b_{30,2}^{6} cpi_data_monthly_{t-6} + b_{30,3}^{6} cpi_fesl_data_monthly_{t-6} + b_{30,4}^{6} rec_sahm_data_monthly_{t-6} + b_{30,5}^{6} rec_smooth_prob_data_monthly_{t-6} + b_{30,6}^{6} rec_nber_data_monthly_{t-6} + b_{30,7}^{6} unemployment_data_monthly_{t-6} + b_{30,8}^{6} unemployment_level_data_monthly_{t-6} + b_{30,9}^{6} all_employed_data_monthly_{t-6} + b_{30,10}^{6} employment_population_ratio_data_monthly_{t-6} + b_{30,11}^{6} labor_force_participation_data_monthly_{t-6} + b_{30,12}^{6} m1_data_monthly_{t-6} + b_{30,13}^{6} m2_data_monthly_{t-6} + b_{30,14}^{6} m2_real_data_monthly_{t-6} + b_{30,15}^{6} interest_rate_fedfunds_data_monthly_{t-6} + b_{30,16}^{6} consumer_sentiment_data_monthly_{t-6} + b_{30,17}^{6} consumer_sentiment_inflation_data_monthly_{t-6} + b_{30,18}^{6} consumer_sentiment_composite_data_monthly_{t-6} + b_{30,19}^{6} consumer_sentiment_composite_amplitude_data_monthly_{t-6} + b_{30,20}^{6} job_openings_data_monthly_{t-6} + b_{30,21}^{6} job_hires_data_monthly_{t-6} + b_{30,22}^{6} job_separations_data_monthly_{t-6} + b_{30,23}^{6} average_weekly_earnings_data_monthly_{t-6} + b_{30,24}^{6} gdp_data_quarterly_{t-6} + b_{30,25}^{6} gdp_real_data_quarterly_{t-6} + b_{30,26}^{6} cpi_inflation_data_annual_{t-6} + b_{30,27}^{6} median_income_real_data_annual_{t-6} + b_{30,28}^{6} median_income_data_annual_{t-6} + b_{30,29}^{6} Close_{t-6} + b_{30,30}^{6} Volume_{t-6} + c_{30} + e_{t}^{Volume} \\
\end{align*}
$$


In [ ]:
post_draws = bvar_monthly_gig.sample_posterior(plot_coefficients=True)

Sampling Posterior:   0%|          | 0/5000 [02:48<?, ?it/s]


KeyboardInterrupt: 